In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ============================================================
# STAGE PFA — YOLO11-OCR MODEL OPTIMIZATION WORKSPACE
# STEP 1C-OCR — GPU VERSION
# ============================================================
#
# PURPOSE:
#   1. Mount Google Drive
#   2. Verify the Colab GPU/CUDA environment
#   3. Define the YOLO11-OCR checkpoint
#   4. Verify the checkpoint exists
#   5. Load the model onto GPU
#   6. Record model + GPU information
#   7. Create a clean OCR optimization directory
#   8. Verify the single OCR test set:
#        - every image has a matching label
#        - every label has a matching image
#        - no duplicate stems
#   9. Save model registry and experiment metadata
#
# IMPORTANT:
#   Original .pt checkpoint is NEVER modified.
#
#   OCR model:
#       YOLO11 detection model
#       Individual character bounding-box detection
#       Arabic + digit character classes
#
#   This is NOT an image-classification OCR model.
# ============================================================


# ============================================================
# 1. IMPORT REQUIRED PYTHON MODULES
# ============================================================

import sys
import subprocess
import os
import json
import time
from pathlib import Path
from collections import defaultdict

print("=" * 90)
print("STEP 1C-OCR — GPU OPTIMIZATION WORKSPACE INITIALIZATION")
print("=" * 90)


# ============================================================
# 2. VERIFY / INSTALL ULTRALYTICS
# ============================================================

try:

    import ultralytics

    print(
        f"✓ Ultralytics already installed: "
        f"{ultralytics.__version__}"
    )

except ImportError:

    print("Installing Ultralytics...")

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "ultralytics"
    ])

    import ultralytics

    print(
        f"✓ Ultralytics installed: "
        f"{ultralytics.__version__}"
    )


# ============================================================
# 3. IMPORT PYTORCH
# ============================================================

import torch


# ============================================================
# 4. VERIFY GPU / CUDA
# ============================================================

print("\n" + "=" * 90)
print("GPU / CUDA ENVIRONMENT CHECK")
print("=" * 90)

print(
    f"\nPyTorch version : "
    f"{torch.__version__}"
)

print(
    f"CUDA available  : "
    f"{torch.cuda.is_available()}"
)

print(
    f"CUDA version    : "
    f"{torch.version.cuda}"
)

if not torch.cuda.is_available():

    raise RuntimeError(
        "\n"
        "============================================================\n"
        "CUDA GPU NOT AVAILABLE\n"
        "============================================================\n\n"
        "Please change the Colab runtime to GPU:\n\n"
        "Runtime → Change runtime type → GPU\n\n"
        "Then reconnect the runtime and run this cell again.\n"
    )


# ------------------------------------------------------------
# GPU information
# ------------------------------------------------------------

GPU_INDEX = 0

GPU_NAME = torch.cuda.get_device_name(
    GPU_INDEX
)

GPU_MEMORY_GB = (
    torch.cuda.get_device_properties(
        GPU_INDEX
    ).total_memory
    / (1024 ** 3)
)

CUDA_CAPABILITY = (
    torch.cuda.get_device_capability(
        GPU_INDEX
    )
)

print(
    f"GPU device        : "
    f"{GPU_NAME}"
)

print(
    f"GPU memory        : "
    f"{GPU_MEMORY_GB:.2f} GB"
)

print(
    f"Compute capability: "
    f"{CUDA_CAPABILITY[0]}.{CUDA_CAPABILITY[1]}"
)

print("\n✓ CUDA GPU is available")


# ============================================================
# 5. DEFINE PROJECT ROOT
# ============================================================

STAGE_PFA = (
    "/content/drive/MyDrive/Stage_PFA"
)

if not os.path.isdir(STAGE_PFA):

    raise FileNotFoundError(
        f"\nStage_PFA directory not found:\n"
        f"{STAGE_PFA}"
    )

print(
    f"\n✓ Project directory found:\n"
    f"  {STAGE_PFA}"
)


# ============================================================
# 6. DEFINE THE YOLO11-OCR MODEL
# ============================================================
#
# >>> UPDATE THIS PATH TO YOUR YOLO11-OCR best.pt <<<
#
# The model name MUST remain:
#
#     yolo11_ocr
#
# This identifier will be used by Step 2-OCR onward.
# ============================================================

OCR_MODEL_PATHS = {

    "yolo11_ocr":
        "/content/drive/MyDrive/Stage_PFA/experiments/YOLOV11_OCR/"
        "weights/best.pt",

}


# ============================================================
# 7. DEFINE OCR DATASET PATHS
# ============================================================
#
# SINGLE TEST SET ONLY
#
# Expected structure:
#
# OCR_Dataset/
# ├── data.yaml
# ├── train/
# │   ├── images/
# │   └── labels/
# ├── val/
# │   ├── images/
# │   └── labels/
# └── test/
#     ├── images/
#     └── labels/
#
# The optimization pipeline uses the TEST split for the
# apples-to-apples evaluation of:
#
#   - Character Detection Accuracy
#   - Character Accuracy
#   - CER
#   - Exact-Match Accuracy
#   - mAP50
#   - mAP50-95
#
# The test set is NOT modified.
# ============================================================

OCR_DATA_YAML = (
    "/content/drive/MyDrive/Stage_PFA/datasets/data.yaml"
)

OCR_DATASET_ROOT = (
    "/content/drive/MyDrive/Stage_PFA/datasets"
)

OCR_TEST_IMAGES = (
    "/content/drive/MyDrive/Stage_PFA/datasets/test/images"
)

OCR_TEST_LABELS = (
    "/content/drive/MyDrive/Stage_PFA/datasets/test/labels"
)


# ============================================================
# 8. CHECK MODEL CHECKPOINT
# ============================================================

print("\n" + "=" * 90)
print("CHECKING OCR MODEL CHECKPOINT")
print("=" * 90)

missing_models = []

for model_name, model_path in OCR_MODEL_PATHS.items():

    if not os.path.isfile(model_path):

        missing_models.append(
            (
                model_name,
                model_path
            )
        )

        print(
            f"✗ {model_name}"
        )

        print(
            f"    NOT FOUND: "
            f"{model_path}"
        )

    else:

        size_mb = (
            os.path.getsize(model_path)
            / (1024 ** 2)
        )

        print(
            f"✓ {model_name}"
        )

        print(
            f"    {model_path}"
        )

        print(
            f"    Size: "
            f"{size_mb:.2f} MB"
        )


if missing_models:

    print(
        "\n" + "!" * 90
    )

    print(
        "ERROR: OCR MODEL PATH IS INVALID"
    )

    print(
        "!" * 90
    )

    for name, path in missing_models:

        print(
            f"\n{name}"
        )

        print(path)

    raise FileNotFoundError(
        "\nCorrect the OCR_MODEL_PATHS dictionary "
        "and run this cell again."
    )


print(
    "\n✓ OCR checkpoint found."
)


# ============================================================
# 9. CHECK OCR DATASET PATHS
# ============================================================

print("\n" + "=" * 90)
print("CHECKING OCR DATASET PATHS")
print("=" * 90)

dataset_paths_to_check = {

    "data_yaml":
        OCR_DATA_YAML,

    "dataset_root":
        OCR_DATASET_ROOT,

    "test_images":
        OCR_TEST_IMAGES,

    "test_labels":
        OCR_TEST_LABELS,

}


dataset_path_errors = []

for name, path in dataset_paths_to_check.items():

    if os.path.exists(path):

        print(
            f"✓ {name:15s}: {path}"
        )

    else:

        print(
            f"✗ {name:15s}: NOT FOUND"
        )

        print(
            f"  {path}"
        )

        dataset_path_errors.append(
            (name, path)
        )


if dataset_path_errors:

    raise FileNotFoundError(
        "\nOne or more OCR dataset paths are invalid. "
        "Correct the paths in Section 7 and run again."
    )


print(
    "\n✓ OCR dataset paths verified."
)


# ============================================================
# 10. CREATE OCR OPTIMIZATION DIRECTORY STRUCTURE
# ============================================================
#
# A SEPARATE parallel tree:
#
#   Stage_PFA/
#   ├── MODEL_OPTIMIZATION/
#   └── OCR_OPTIMIZATION/
#
# OCR artifacts never mix with the detection pipeline.
# ============================================================

OCR_OPT_ROOT = os.path.join(
    STAGE_PFA,
    "OCR_OPTIMIZATION"
)


OCR_DIRECTORIES = {

    "root":
        OCR_OPT_ROOT,

    "original":
        os.path.join(
            OCR_OPT_ROOT,
            "01_original"
        ),

    "onnx_fp32":
        os.path.join(
            OCR_OPT_ROOT,
            "02_onnx_fp32"
        ),

    "onnx_fp16":
        os.path.join(
            OCR_OPT_ROOT,
            "03_onnx_fp16"
        ),

    "onnx_int8":
        os.path.join(
            OCR_OPT_ROOT,
            "04_onnx_int8"
        ),

    "calibration":
        os.path.join(
            OCR_OPT_ROOT,
            "05_calibration"
        ),

    "validation":
        os.path.join(
            OCR_OPT_ROOT,
            "06_validation"
        ),

    "benchmarks":
        os.path.join(
            OCR_OPT_ROOT,
            "07_benchmarks"
        ),

    "final":
        os.path.join(
            OCR_OPT_ROOT,
            "08_final"
        ),

    "logs":
        os.path.join(
            OCR_OPT_ROOT,
            "09_logs"
        ),

}


for path in OCR_DIRECTORIES.values():

    os.makedirs(
        path,
        exist_ok=True
    )


print(
    "\n" + "=" * 90
)

print(
    "OCR OPTIMIZATION DIRECTORY STRUCTURE"
)

print(
    "=" * 90
)


for key, path in OCR_DIRECTORIES.items():

    print(
        f"{key:15s} → {path}"
    )


# ============================================================
# 11. IMAGE ↔ LABEL FILENAME EQUIVALENCE CHECK
# ============================================================
#
# SINGLE TEST SET
#
# PURPOSE:
#   Confirm:
#
#     1. Every image has a matching label.
#     2. Every label has a matching image.
#     3. No duplicate image stems.
#     4. No duplicate label stems.
#
# We intentionally do NOT enforce a hard-coded image count.
# The actual dataset count is recorded automatically.
# ============================================================

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
}

LABEL_EXTENSIONS = {
    ".txt"
}


def check_image_label_equivalence(
    split_name,
    images_dir,
    labels_dir
):

    print(
        "\n" + "-" * 90
    )

    print(
        f"CHECKING SPLIT: {split_name}"
    )

    print(
        "-" * 90
    )

    result = {

        "split":
            split_name,

        "images_dir":
            images_dir,

        "labels_dir":
            labels_dir,

        "status":
            "unknown",

    }


    # --------------------------------------------------------
    # Verify directories
    # --------------------------------------------------------

    if not os.path.isdir(images_dir):

        print(
            f"✗ Images directory not found:"
            f" {images_dir}"
        )

        result["status"] = "images_dir_missing"

        return result


    if not os.path.isdir(labels_dir):

        print(
            f"✗ Labels directory not found:"
            f" {labels_dir}"
        )

        result["status"] = "labels_dir_missing"

        return result


    # --------------------------------------------------------
    # Collect image stems
    # --------------------------------------------------------

    image_stems = defaultdict(list)

    for p in Path(images_dir).iterdir():

        if (
            p.is_file()
            and p.suffix.lower() in IMAGE_EXTENSIONS
        ):

            image_stems[p.stem].append(
                p.name
            )


    # --------------------------------------------------------
    # Collect label stems
    # --------------------------------------------------------

    label_stems = defaultdict(list)

    for p in Path(labels_dir).iterdir():

        if (
            p.is_file()
            and p.suffix.lower() in LABEL_EXTENSIONS
        ):

            label_stems[p.stem].append(
                p.name
            )


    # --------------------------------------------------------
    # Detect duplicates
    # --------------------------------------------------------

    duplicate_image_stems = {

        k: v

        for k, v in image_stems.items()

        if len(v) > 1

    }


    duplicate_label_stems = {

        k: v

        for k, v in label_stems.items()

        if len(v) > 1

    }


    # --------------------------------------------------------
    # Compare stems
    # --------------------------------------------------------

    image_stem_set = set(
        image_stems.keys()
    )

    label_stem_set = set(
        label_stems.keys()
    )


    images_without_labels = sorted(
        image_stem_set - label_stem_set
    )

    labels_without_images = sorted(
        label_stem_set - image_stem_set
    )

    matched = sorted(
        image_stem_set & label_stem_set
    )


    # --------------------------------------------------------
    # Counts
    # --------------------------------------------------------

    total_images = sum(
        len(v)
        for v in image_stems.values()
    )

    total_labels = sum(
        len(v)
        for v in label_stems.values()
    )


    # --------------------------------------------------------
    # Report
    # --------------------------------------------------------

    print(
        f"  Images found          : "
        f"{total_images}"
    )

    print(
        f"  Labels found          : "
        f"{total_labels}"
    )

    print(
        f"  Matched pairs         : "
        f"{len(matched)}"
    )

    print(
        f"  Images without labels : "
        f"{len(images_without_labels)}"
    )

    print(
        f"  Labels without images : "
        f"{len(labels_without_images)}"
    )

    print(
        f"  Duplicate image stems : "
        f"{len(duplicate_image_stems)}"
    )

    print(
        f"  Duplicate label stems : "
        f"{len(duplicate_label_stems)}"
    )


    # --------------------------------------------------------
    # Display mismatches
    # --------------------------------------------------------

    if images_without_labels:

        print(
            "\n  ✗ Sample images missing labels "
            "(up to 10 shown):"
        )

        for stem in images_without_labels[:10]:

            print(
                f"      {stem}"
            )


    if labels_without_images:

        print(
            "\n  ✗ Sample labels missing images "
            "(up to 10 shown):"
        )

        for stem in labels_without_images[:10]:

            print(
                f"      {stem}"
            )


    if duplicate_image_stems:

        print(
            "\n  ✗ Duplicate image stems "
            "(up to 10 shown):"
        )

        for stem, files in list(
            duplicate_image_stems.items()
        )[:10]:

            print(
                f"      {stem}: {files}"
            )


    if duplicate_label_stems:

        print(
            "\n  ✗ Duplicate label stems "
            "(up to 10 shown):"
        )

        for stem, files in list(
            duplicate_label_stems.items()
        )[:10]:

            print(
                f"      {stem}: {files}"
            )


    # --------------------------------------------------------
    # Final status
    # --------------------------------------------------------

    is_clean = (

        not images_without_labels

        and not labels_without_images

        and not duplicate_image_stems

        and not duplicate_label_stems

    )


    if is_clean:

        print(
            "\n  ✓ Images and labels are fully "
            "equivalent (1:1 match, no duplicates)."
        )

        result["status"] = "clean"

    else:

        print(
            "\n  ✗ MISMATCH DETECTED — resolve before "
            "validation/calibration."
        )

        result["status"] = "mismatch"


    result.update({

        "total_images":
            total_images,

        "total_labels":
            total_labels,

        "matched_pairs":
            len(matched),

        "images_without_labels":
            images_without_labels,

        "labels_without_images":
            labels_without_images,

        "duplicate_image_stems":
            duplicate_image_stems,

        "duplicate_label_stems":
            duplicate_label_stems,

    })


    return result


# ============================================================
# 12. RUN TEST SET EQUIVALENCE CHECK
# ============================================================

print(
    "\n" + "=" * 90
)

print(
    "IMAGE ↔ LABEL EQUIVALENCE CHECK — OCR TEST SET"
)

print(
    "=" * 90
)


equivalence_results = {}


equivalence_results["test"] = (
    check_image_label_equivalence(

        split_name="test",

        images_dir=
            OCR_TEST_IMAGES,

        labels_dir=
            OCR_TEST_LABELS,

    )
)


if equivalence_results["test"]["status"] != "clean":

    print(
        "\n" + "!" * 90
    )

    print(
        "WARNING: OCR TEST SET IS NOT CLEAN"
    )

    print(
        "Resolve the image/label mismatch before "
        "performing official validation."
    )

    print(
        "!" * 90
    )

else:

    print(
        "\n✓ OCR test set is clean."
    )


# ============================================================
# 13. LOAD MODEL ON GPU AND COLLECT INFORMATION
# ============================================================

from ultralytics import YOLO


print(
    "\n" + "=" * 90
)

print(
    "LOADING AND INSPECTING OCR MODEL ON GPU"
)

print(
    "=" * 90
)


OCR_MODEL_INFO = {}


for model_name, model_path in OCR_MODEL_PATHS.items():

    print(
        f"\n{'-' * 90}"
    )

    print(
        f"MODEL: {model_name}"
    )

    print(
        f"{'-' * 90}"
    )


    try:

        # ----------------------------------------------------
        # Load checkpoint
        # ----------------------------------------------------

        print(
            "→ Loading checkpoint..."
        )

        model = YOLO(
            model_path
        )


        # ----------------------------------------------------
        # Move model to GPU
        # ----------------------------------------------------

        print(
            "→ Moving model to CUDA..."
        )

        model.to(
            f"cuda:{GPU_INDEX}"
        )


        # ----------------------------------------------------
        # Basic file information
        # ----------------------------------------------------

        size_mb = (

            os.path.getsize(
                model_path
            )

            / (1024 ** 2)

        )


        # ----------------------------------------------------
        # Underlying PyTorch model
        # ----------------------------------------------------

        torch_model = model.model


        # ----------------------------------------------------
        # Parameter count
        # ----------------------------------------------------

        parameters = sum(

            p.numel()

            for p in torch_model.parameters()

        )


        # ----------------------------------------------------
        # Trainable parameters
        # ----------------------------------------------------

        trainable_parameters = sum(

            p.numel()

            for p in torch_model.parameters()

            if p.requires_grad

        )


        # ----------------------------------------------------
        # Model task
        # ----------------------------------------------------

        task = getattr(
            model,
            "task",
            "unknown"
        )


        if task != "detect":

            print(
                f"  ⚠ WARNING: expected task 'detect' "
                f"for the per-character OCR model, "
                f"found '{task}'."
            )

        else:

            print(
                "  ✓ Model task confirmed: detect"
            )


        # ----------------------------------------------------
        # Class names
        # ----------------------------------------------------

        names = getattr(
            model,
            "names",
            None
        )


        num_classes = (
            len(names)
            if names
            else 0
        )


        # ----------------------------------------------------
        # Detect actual model device
        # ----------------------------------------------------

        try:

            actual_device = str(

                next(
                    torch_model.parameters()
                ).device

            )

        except Exception:

            actual_device = "unknown"


        # ----------------------------------------------------
        # Save information
        # ----------------------------------------------------

        OCR_MODEL_INFO[model_name] = {

            "checkpoint":
                model_path,

            "checkpoint_size_mb":
                round(
                    size_mb,
                    3
                ),

            "parameters":
                parameters,

            "trainable_parameters":
                trainable_parameters,

            "task":
                task,

            "num_classes":
                num_classes,

            "names":
                names,

            "device":
                actual_device,

            "gpu":
                GPU_NAME,

            "gpu_memory_gb":
                round(
                    GPU_MEMORY_GB,
                    3
                ),

            "cuda_version":
                str(
                    torch.version.cuda
                ),

            "pytorch_version":
                torch.__version__,

            "ultralytics_version":
                ultralytics.__version__,

        }


        # ----------------------------------------------------
        # Print information
        # ----------------------------------------------------

        print(
            "✓ Model loaded successfully"
        )

        print(
            f"  Device:             "
            f"{actual_device}"
        )

        print(
            f"  Task:               "
            f"{task}"
        )

        print(
            f"  Parameters:         "
            f"{parameters:,}"
        )

        print(
            f"  Trainable params:   "
            f"{trainable_parameters:,}"
        )

        print(
            f"  Checkpoint size:    "
            f"{size_mb:.2f} MB"
        )

        print(
            f"  Num classes:        "
            f"{num_classes}"
        )

        print(
            f"  Classes:            "
            f"{names}"
        )


        # ----------------------------------------------------
        # Release GPU memory
        # ----------------------------------------------------

        del model
        del torch_model

        torch.cuda.empty_cache()


    except Exception as e:

        print(
            f"✗ FAILED TO LOAD "
            f"{model_name}"
        )

        print(
            f"  Error: {repr(e)}"
        )

        raise


# ============================================================
# 14. SAVE MODEL INVENTORY
# ============================================================

ocr_inventory_path = os.path.join(

    OCR_DIRECTORIES["logs"],

    "ocr_model_inventory.json"

)


with open(

    ocr_inventory_path,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        OCR_MODEL_INFO,

        f,

        indent=4,

        ensure_ascii=False,

        default=str

    )


print(
    "\n✓ OCR model inventory saved:"
)

print(
    f"  {ocr_inventory_path}"
)


# ============================================================
# 15. SAVE GPU ENVIRONMENT INFORMATION
# ============================================================

OCR_GPU_INFO = {

    "pytorch_version":
        torch.__version__,

    "ultralytics_version":
        ultralytics.__version__,

    "cuda_available":
        torch.cuda.is_available(),

    "cuda_version":
        str(
            torch.version.cuda
        ),

    "gpu_index":
        GPU_INDEX,

    "gpu_name":
        GPU_NAME,

    "gpu_memory_gb":
        round(
            GPU_MEMORY_GB,
            3
        ),

    "compute_capability":
        (
            f"{CUDA_CAPABILITY[0]}."
            f"{CUDA_CAPABILITY[1]}"
        ),

    "device":
        f"cuda:{GPU_INDEX}",

    "experiment_stage":
        "Step 1C-OCR",

    "execution_environment":
        "Google Colab GPU",

}


ocr_gpu_info_path = os.path.join(

    OCR_DIRECTORIES["logs"],

    "ocr_gpu_environment.json"

)


with open(

    ocr_gpu_info_path,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        OCR_GPU_INFO,

        f,

        indent=4

    )


print(
    "\n✓ GPU environment information saved:"
)

print(
    f"  {ocr_gpu_info_path}"
)


# ============================================================
# 16. SAVE OCR MODEL PATHS
# ============================================================

ocr_paths_file = os.path.join(

    OCR_DIRECTORIES["logs"],

    "ocr_model_paths.json"

)


with open(

    ocr_paths_file,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        OCR_MODEL_PATHS,

        f,

        indent=4

    )


print(
    "\n✓ OCR model paths saved:"
)

print(
    f"  {ocr_paths_file}"
)


# ============================================================
# 17. SAVE OCR DATASET PATHS
# ============================================================

ocr_dataset_paths_file = os.path.join(

    OCR_DIRECTORIES["logs"],

    "ocr_dataset_paths.json"

)


OCR_DATASET_PATHS = {

    "data_yaml":
        OCR_DATA_YAML,

    "dataset_root":
        OCR_DATASET_ROOT,

    "test_images":
        OCR_TEST_IMAGES,

    "test_labels":
        OCR_TEST_LABELS,

}


with open(

    ocr_dataset_paths_file,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        OCR_DATASET_PATHS,

        f,

        indent=4

    )


print(
    "\n✓ OCR dataset paths saved:"
)

print(
    f"  {ocr_dataset_paths_file}"
)


# ============================================================
# 18. SAVE IMAGE ↔ LABEL EQUIVALENCE RESULTS
# ============================================================

equivalence_log_path = os.path.join(

    OCR_DIRECTORIES["logs"],

    "step1c_ocr_image_label_equivalence.json"

)


with open(

    equivalence_log_path,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        equivalence_results,

        f,

        indent=4,

        ensure_ascii=False,

        default=str

    )


print(
    "\n✓ Image/label equivalence results saved:"
)

print(
    f"  {equivalence_log_path}"
)


# ============================================================
# 19. SAVE COMPLETE STEP 1C-OCR METADATA
# ============================================================

STEP1C_OCR_METADATA = {

    "step":
        "1C-OCR",

    "description":
        "GPU optimization workspace initialization "
        "(YOLO11-OCR)",

    "timestamp":
        time.strftime(
            "%Y-%m-%d %H:%M:%S"
        ),

    "execution_environment":
        "Google Colab GPU",

    "gpu":
        OCR_GPU_INFO,

    "models":
        OCR_MODEL_INFO,

    "model_paths":
        OCR_MODEL_PATHS,

    "dataset_paths":
        OCR_DATASET_PATHS,

    "image_label_equivalence":
        equivalence_results,

    "optimization_root":
        OCR_OPT_ROOT,

}


ocr_metadata_path = os.path.join(

    OCR_DIRECTORIES["logs"],

    "step1c_ocr_gpu_metadata.json"

)


with open(

    ocr_metadata_path,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        STEP1C_OCR_METADATA,

        f,

        indent=4,

        ensure_ascii=False,

        default=str

    )


print(
    "\n✓ Complete Step 1C-OCR metadata saved:"
)

print(
    f"  {ocr_metadata_path}"
)


# ============================================================
# 20. FINAL GPU MEMORY STATUS
# ============================================================

torch.cuda.empty_cache()


allocated_gb = (

    torch.cuda.memory_allocated(
        GPU_INDEX
    )

    / (1024 ** 3)

)


reserved_gb = (

    torch.cuda.memory_reserved(
        GPU_INDEX
    )

    / (1024 ** 3)

)


print(
    "\n" + "=" * 90
)

print(
    "FINAL GPU MEMORY STATUS"
)

print(
    "=" * 90
)

print(
    f"GPU allocated : "
    f"{allocated_gb:.3f} GB"
)

print(
    f"GPU reserved  : "
    f"{reserved_gb:.3f} GB"
)


# ============================================================
# FINAL STATUS
# ============================================================

print(
    "\n" + "=" * 90
)

print(
    "STEP 1C-OCR — GPU VERSION COMPLETED"
)

print(
    "=" * 90
)


test_status = (
    equivalence_results["test"]["status"]
)


test_images_count = (
    equivalence_results["test"].get(
        "total_images",
        0
    )
)


test_labels_count = (
    equivalence_results["test"].get(
        "total_labels",
        0
    )
)


print(
    f"""
OCR model registered:

  1. yolo11_ocr — per-character detection
     Task: {OCR_MODEL_INFO['yolo11_ocr']['task']}

GPU environment verified.

Checkpoint loaded successfully on CUDA.

Original .pt checkpoint remains untouched.
No model has been converted.
No model has been modified.

OCR test set:

  Images : {test_images_count}
  Labels : {test_labels_count}
  Status : {test_status}

GPU metadata has been recorded.

NEXT:

    STEP 2-OCR
        ↓
    FP32 GPU baseline
        ↓
    Primary metrics:
        - Character Accuracy
        - CER
        - Exact-Match Accuracy
        ↓
    Secondary metrics:
        - mAP50
        - mAP50-95
        ↓
    ONNX FP32 export
        ↓
    ONNX FP32 verification
        ↓
    PT vs ONNX equivalence
        ↓
    STEP 3-OCR
        ↓
    FP16 conversion
        ↓
    STEP 4-OCR
        ↓
    INT8 static quantization
        ↓
    Conv-only quantization
        ↓
    Tail-node exclusion
        ↓
    Calibration using cropped-plate images
        ↓
    STEP 5-OCR
        ↓
    Raspberry Pi deployment
        ↓
    On-device validation + benchmarking
"""
)

print(
    "\n✓ READY FOR STEP 2-OCR"
)

STEP 1C-OCR — GPU OPTIMIZATION WORKSPACE INITIALIZATION
Installing Ultralytics...
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
✓ Ultralytics installed: 8.4.137

GPU / CUDA ENVIRONMENT CHECK

PyTorch version : 2.11.0+cu128
CUDA available  : True
CUDA version    : 12.8
GPU device        : Tesla T4
GPU memory        : 14.56 GB
Compute capability: 7.5

✓ CUDA GPU is available

✓ Project directory found:
  /content/drive/MyDrive/Stage_PFA

CHECKING OCR MODEL CHECKPOINT
✓ yolo11_ocr
    /content/drive/MyDrive/Stage_PFA/experiments/YOLOV11_OCR/weights/best.pt
    Size: 15.22 MB

✓ OCR checkpoint found.

CHECKING OCR DATASET PATHS
✓ data_yaml      : /content/drive/MyDrive/Stage_PFA/datasets/data.yaml
✓ dataset_root   : /content/drive

In [3]:
# ============================================================
# STAGE PFA — YOLO11-OCR
# STEP 2-OCR — CELL 1
# ENVIRONMENT INSTALLATION ONLY
# ============================================================
#
# IMPORTANT:
#   1. Run this cell once.
#   2. Restart the Colab runtime.
#   3. DO NOT run the next cell before restarting.
#
# This cell intentionally does not import ONNX Runtime after
# installation because the runtime must be restarted first.
# ============================================================

import sys
import subprocess

print("=" * 90)
print("STEP 2-OCR — CELL 1: ENVIRONMENT INSTALLATION")
print("=" * 90)

print(f"Python: {sys.version}")

# ------------------------------------------------------------
# Remove conflicting CPU/GPU ONNX Runtime installations
# ------------------------------------------------------------

print("\nRemoving conflicting ONNX Runtime packages...")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "onnxruntime",
        "onnxruntime-gpu",
    ],
    check=False,
)

# ------------------------------------------------------------
# Install ONNX Runtime GPU
#
# CUDA 12.x is the target family for this environment.
# ------------------------------------------------------------

print("\nInstalling ONNX Runtime GPU...")

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "onnxruntime-gpu",
    ]
)

# ------------------------------------------------------------
# ONNX
# ------------------------------------------------------------

print("\nInstalling ONNX...")

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "onnx",
    ]
)

# ------------------------------------------------------------
# ONNX simplifier used by Ultralytics export
# ------------------------------------------------------------

print("\nInstalling ONNX Slim...")

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "onnxslim>=0.1.82",
    ]
)

# ------------------------------------------------------------
# OpenCV
# ------------------------------------------------------------

print("\nInstalling OpenCV...")

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "opencv-python-headless",
    ]
)

print("\n" + "=" * 90)
print("INSTALLATION COMPLETED")
print("=" * 90)

print(
    """
IMPORTANT:

The Python environment has changed.

NOW DO THIS:

    Runtime
       ↓
    Restart session

After restarting, run CELL 2.

DO NOT run the complete benchmark yet.
"""
)


STEP 2-OCR — CELL 1: ENVIRONMENT INSTALLATION
Python: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]

Removing conflicting ONNX Runtime packages...

Installing ONNX Runtime GPU...

Installing ONNX...

Installing ONNX Slim...

Installing OpenCV...

INSTALLATION COMPLETED

IMPORTANT:

The Python environment has changed.

NOW DO THIS:

    Runtime
       ↓
    Restart session

After restarting, run CELL 2.

DO NOT run the complete benchmark yet.



In [4]:
# ============================================================
# FIX — INSTALL CUDA 12.8 COMPATIBLE ONNX RUNTIME GPU
# ============================================================

!pip uninstall -y onnxruntime onnxruntime-gpu

!pip install -q "onnxruntime-gpu==1.26.0"

Found existing installation: onnxruntime-gpu 1.29.0
Uninstalling onnxruntime-gpu-1.29.0:
  Successfully uninstalled onnxruntime-gpu-1.29.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 277.0/277.0 MB 1.6 MB/s eta 0:00:00


In [1]:
# ============================================================
# ONNX RUNTIME GPU VERIFICATION
# ============================================================

import torch
import onnxruntime as ort

print("=" * 100)
print("GPU / ONNX RUNTIME VERIFICATION")
print("=" * 100)

print(f"\nPyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
print(f"CUDA version    : {torch.version.cuda}")

if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(
        f"GPU memory      : "
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
    )

print("\n" + "=" * 100)
print("ONNX RUNTIME")
print("=" * 100)

print(f"\nONNX Runtime version : {ort.__version__}")

providers = ort.get_available_providers()

print("\nAvailable ONNX Runtime providers:")

for provider in providers:
    print(f"  - {provider}")

if "CUDAExecutionProvider" not in providers:

    raise RuntimeError(
        "\nCUDAExecutionProvider is STILL NOT available.\n\n"
        "Do NOT continue with Step 2D.\n\n"
        f"Available providers:\n{providers}"
    )

print("\n✓ CUDAExecutionProvider is available")
print("✓ ONNX Runtime GPU is ready")

GPU / ONNX RUNTIME VERIFICATION

PyTorch version : 2.11.0+cu128
CUDA available  : True
CUDA version    : 12.8
GPU             : Tesla T4
GPU memory      : 14.56 GB

ONNX RUNTIME

ONNX Runtime version : 1.26.0

Available ONNX Runtime providers:
  - TensorrtExecutionProvider
  - CUDAExecutionProvider
  - CPUExecutionProvider

✓ CUDAExecutionProvider is available
✓ ONNX Runtime GPU is ready


In [2]:
# ============================================================
# STAGE PFA — YOLO11-OCR MODEL OPTIMIZATION
# STEP 2-OCR
#
# FP32 PYTORCH CUDA BASELINE
# +
# ONNX FP32 CUDA
#
# CELL 3 — MAIN EXPERIMENT
# ============================================================

import os
import sys
import json
import time
import math
import shutil
from pathlib import Path

import numpy as np
import torch
import cv2
import onnx
import onnxruntime as ort

from ultralytics import YOLO


# ============================================================
# 1. HEADER
# ============================================================

print("=" * 90)
print("STEP 2-OCR — FP32 PYTORCH CUDA + ONNX FP32 CUDA")
print("=" * 90)

print(
    f"Ultralytics : "
    f"__pending__"
)

try:
    import ultralytics

    print(
        f"Ultralytics : "
        f"{ultralytics.__version__}"
    )

except Exception:
    pass

print(
    f"ONNX        : "
    f"{onnx.__version__}"
)

print(
    f"ONNX Runtime: "
    f"{ort.__version__}"
)

print(
    f"Python      : "
    f"{sys.version.split()[0]}"
)


# ============================================================
# 2. CUDA VERIFICATION
# ============================================================

print("\n" + "=" * 90)
print("VERIFYING CUDA")
print("=" * 90)

if not torch.cuda.is_available():

    raise RuntimeError(
        "PyTorch CUDA is unavailable."
    )


GPU_NAME = (
    torch.cuda.get_device_name(0)
)

GPU_MEMORY_GB = (
    torch.cuda.get_device_properties(0)
    .total_memory
    / 1024**3
)


print(
    "✓ CUDA available"
)

print(
    f"✓ GPU: {GPU_NAME}"
)

print(
    f"✓ GPU memory: {GPU_MEMORY_GB:.2f} GB"
)

print(
    f"✓ PyTorch CUDA: {torch.version.cuda}"
)


# ============================================================
# 3. ONNX RUNTIME CUDA VERIFICATION
# ============================================================

print("\n" + "=" * 90)
print("VERIFYING ONNX RUNTIME CUDA")
print("=" * 90)


available_providers = (
    ort.get_available_providers()
)


print(
    "Available providers:"
)

for provider in available_providers:

    print(
        f"  - {provider}"
    )


if (
    "CUDAExecutionProvider"
    not in
    available_providers
):

    raise RuntimeError(
        "CUDAExecutionProvider is not available."
    )


# ------------------------------------------------------------
# Preload CUDA/cuDNN libraries
# ------------------------------------------------------------

try:

    ort.preload_dlls(
        cuda=True,
        cudnn=True,
        msvc=True,
    )

    print(
        "✓ CUDA/cuDNN libraries preloaded."
    )

except Exception as e:

    print(
        f"⚠ preload_dlls(): {e}"
    )


# ============================================================
# 4. CONFIGURATION
# ============================================================

OCR_OPT_ROOT = (
    "/content/drive/MyDrive/Stage_PFA/"
    "OCR_OPTIMIZATION"
)

OCR_LOG_DIR = os.path.join(
    OCR_OPT_ROOT,
    "09_logs"
)

OCR_FP32_DIR = os.path.join(
    OCR_OPT_ROOT,
    "02_onnx_fp32"
)

OCR_VALIDATION_DIR = os.path.join(
    OCR_OPT_ROOT,
    "06_validation"
)

OCR_BENCHMARK_DIR = os.path.join(
    OCR_OPT_ROOT,
    "07_benchmarks"
)


OCR_IMAGE_SIZE = 608

OCR_SORT_DIRECTION = (
    "left_to_right"
)

CONF_THRESHOLD = 0.25

IOU_THRESHOLD = 0.45

DEVICE = "cuda:0"

WARMUP_RUNS = 10


# ============================================================
# 5. DIRECTORIES
# ============================================================

for directory in [

    OCR_FP32_DIR,

    OCR_VALIDATION_DIR,

    OCR_BENCHMARK_DIR,

    OCR_LOG_DIR,

]:

    os.makedirs(
        directory,
        exist_ok=True
    )


# ============================================================
# 6. LOAD REGISTRY
# ============================================================

print("\n" + "=" * 90)
print("VERIFYING STEP 1C-OCR REGISTRY")
print("=" * 90)


paths_file = os.path.join(
    OCR_LOG_DIR,
    "ocr_model_paths.json"
)

dataset_paths_file = os.path.join(
    OCR_LOG_DIR,
    "ocr_dataset_paths.json"
)


if not os.path.isfile(
    paths_file
):

    raise FileNotFoundError(
        paths_file
    )


if not os.path.isfile(
    dataset_paths_file
):

    raise FileNotFoundError(
        dataset_paths_file
    )


with open(
    paths_file,
    "r",
    encoding="utf-8"
) as f:

    OCR_MODEL_PATHS = json.load(f)


with open(
    dataset_paths_file,
    "r",
    encoding="utf-8"
) as f:

    OCR_DATASET_PATHS = json.load(f)


MODEL_PATH = (
    OCR_MODEL_PATHS[
        "yolo11_ocr"
    ]
)

TEST_IMAGES = (
    OCR_DATASET_PATHS[
        "test_images"
    ]
)

TEST_LABELS = (
    OCR_DATASET_PATHS[
        "test_labels"
    ]
)

DATA_YAML = (
    OCR_DATASET_PATHS[
        "data_yaml"
    ]
)


print(
    f"✓ Model:\n  {MODEL_PATH}"
)

print(
    f"✓ Test images:\n  {TEST_IMAGES}"
)

print(
    f"✓ Test labels:\n  {TEST_LABELS}"
)

print(
    f"✓ Data YAML:\n  {DATA_YAML}"
)


# ============================================================
# 7. LOAD MODEL
# ============================================================

print("\n" + "=" * 90)
print("LOADING YOLO11-OCR MODEL")
print("=" * 90)


model = YOLO(
    MODEL_PATH
)

model.to(
    DEVICE
)

model.model.float()


print(
    "✓ Model loaded."
)

print(
    "✓ Model converted to FP32."
)

print(
    f"Task: {model.task}"
)

print(
    f"Classes: {model.names}"
)


if model.task != "detect":

    raise RuntimeError(
        f"Expected detect model, "
        f"got {model.task}"
    )


TOTAL_PARAMETERS = sum(

    parameter.numel()

    for parameter
    in model.model.parameters()

)


print(
    f"Total parameters: "
    f"{TOTAL_PARAMETERS:,}"
)


# ============================================================
# 8. HELPER FUNCTIONS
# ============================================================

def levenshtein_distance(
    a,
    b
):

    if a == b:
        return 0

    if not a:
        return len(b)

    if not b:
        return len(a)

    previous = list(
        range(len(b) + 1)
    )

    for i, ca in enumerate(
        a,
        start=1
    ):

        current = [i]

        for j, cb in enumerate(
            b,
            start=1
        ):

            insertion = (
                current[j - 1] + 1
            )

            deletion = (
                previous[j] + 1
            )

            substitution = (
                previous[j - 1]
                + int(ca != cb)
            )

            current.append(
                min(
                    insertion,
                    deletion,
                    substitution
                )
            )

        previous = current

    return previous[-1]


def load_yolo_label(
    label_path
):

    annotations = []

    if not os.path.isfile(
        label_path
    ):

        return annotations


    with open(
        label_path,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            parts = line.split()

            if len(parts) < 5:
                continue

            annotations.append({

                "class_id":
                    int(float(parts[0])),

                "x_center":
                    float(parts[1]),

                "y_center":
                    float(parts[2]),

                "width":
                    float(parts[3]),

                "height":
                    float(parts[4]),

            })

    return annotations


def sort_characters(
    characters,
    direction="left_to_right"
):

    if direction == "left_to_right":

        return sorted(
            characters,
            key=lambda x: (
                x["x_center"],
                x["y_center"]
            )
        )

    if direction == "right_to_left":

        return sorted(
            characters,
            key=lambda x: (
                -x["x_center"],
                x["y_center"]
            )
        )

    raise ValueError(
        "Invalid OCR sorting direction."
    )


def annotations_to_string(
    annotations,
    names
):

    characters = []

    for ann in annotations:

        class_id = (
            ann["class_id"]
        )

        if class_id not in names:
            continue

        characters.append({

            "char":
                str(names[class_id]),

            "x_center":
                ann["x_center"],

            "y_center":
                ann["y_center"],

        })


    characters = sort_characters(
        characters,
        OCR_SORT_DIRECTION
    )


    return "".join(
        item["char"]
        for item in characters
    )


def prediction_to_string(
    result,
    names
):

    if result.boxes is None:
        return ""

    boxes = result.boxes

    if len(boxes) == 0:
        return ""

    xywhn = (
        boxes.xywhn
        .detach()
        .cpu()
        .numpy()
    )

    classes = (
        boxes.cls
        .detach()
        .cpu()
        .numpy()
    )

    confidences = (
        boxes.conf
        .detach()
        .cpu()
        .numpy()
    )

    characters = []

    for i in range(
        len(classes)
    ):

        confidence = float(
            confidences[i]
        )

        if confidence < CONF_THRESHOLD:
            continue

        class_id = int(
            classes[i]
        )

        if class_id not in names:
            continue

        characters.append({

            "char":
                str(names[class_id]),

            "x_center":
                float(xywhn[i][0]),

            "y_center":
                float(xywhn[i][1]),

        })


    characters = sort_characters(
        characters,
        OCR_SORT_DIRECTION
    )


    return "".join(
        item["char"]
        for item in characters
    )


def calculate_ocr_metrics(
    records
):

    total_gt = 0

    total_edit = 0

    position_matches = 0

    position_slots = 0

    exact_matches = 0

    total_samples = len(records)


    for record in records:

        gt = record[
            "ground_truth"
        ]

        pred = record[
            "prediction"
        ]

        distance = (
            levenshtein_distance(
                gt,
                pred
            )
        )

        total_edit += distance

        total_gt += len(gt)

        common = min(
            len(gt),
            len(pred)
        )

        position_matches += sum(

            gt[i] == pred[i]

            for i in range(common)

        )

        position_slots += max(
            len(gt),
            len(pred)
        )

        if gt == pred:

            exact_matches += 1


    cer = (

        total_edit / total_gt

        if total_gt > 0

        else 0.0

    )


    character_accuracy = (

        position_matches
        / position_slots

        if position_slots > 0

        else 0.0

    )


    exact_match = (

        exact_matches
        / total_samples

        if total_samples > 0

        else 0.0

    )


    return {

        "samples":
            total_samples,

        "total_ground_truth_characters":
            total_gt,

        "total_edit_distance":
            total_edit,

        "character_accuracy":
            character_accuracy,

        "cer":
            cer,

        "exact_match_accuracy":
            exact_match,

        "exact_matches":
            exact_matches,

    }


def calculate_latency_statistics(
    latencies
):

    latencies = np.asarray(
        latencies,
        dtype=np.float64
    )

    total_time = (
        np.sum(latencies)
    )

    mean_latency = (
        np.mean(latencies)
    )

    median_latency = (
        np.median(latencies)
    )

    p95_latency = (
        np.percentile(
            latencies,
            95
        )
    )

    min_latency = (
        np.min(latencies)
    )

    max_latency = (
        np.max(latencies)
    )

    fps = (

        1000.0
        / mean_latency

    )


    return {

        "total_time_seconds":
            float(total_time / 1000.0),

        "average_latency_ms":
            float(mean_latency),

        "median_latency_ms":
            float(median_latency),

        "p95_latency_ms":
            float(p95_latency),

        "min_latency_ms":
            float(min_latency),

        "max_latency_ms":
            float(max_latency),

        "fps":
            float(fps),

    }


# ============================================================
# 9. TEST IMAGES
# ============================================================

print("\n" + "=" * 90)
print("PREPARING OCR TEST SET")
print("=" * 90)


IMAGE_EXTENSIONS = {

    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"

}


test_images = sorted(

    [

        p

        for p in Path(
            TEST_IMAGES
        ).iterdir()

        if (

            p.is_file()

            and

            p.suffix.lower()
            in IMAGE_EXTENSIONS

        )

    ]

)


print(
    f"Test images: "
    f"{len(test_images)}"
)


if not test_images:

    raise RuntimeError(
        "No test images found."
    )


# ============================================================
# 10. PYTORCH WARM-UP
# ============================================================

print("\n" + "=" * 90)
print("PYTORCH CUDA WARM-UP")
print("=" * 90)


warmup_image = str(
    test_images[0]
)


for _ in range(
    WARMUP_RUNS
):

    model.predict(

        source=warmup_image,

        imgsz=OCR_IMAGE_SIZE,

        conf=CONF_THRESHOLD,

        iou=IOU_THRESHOLD,

        device=DEVICE,

        verbose=False,

    )


torch.cuda.synchronize()


print(
    f"✓ Completed "
    f"{WARMUP_RUNS} warm-up runs."
)


# ============================================================
# 11. PYTORCH OCR BENCHMARK
# ============================================================

print("\n" + "=" * 90)
print("FP32 PYTORCH CUDA OCR BASELINE")
print("=" * 90)


pt_records = []

pt_latencies = []


for index, image_path in enumerate(
    test_images,
    start=1
):

    label_path = (

        Path(TEST_LABELS)

        / f"{image_path.stem}.txt"

    )


    annotations = (
        load_yolo_label(
            str(label_path)
        )
    )


    ground_truth = (
        annotations_to_string(
            annotations,
            model.names
        )
    )


    # --------------------------------------------------------
    # CUDA timing
    # --------------------------------------------------------

    torch.cuda.synchronize()

    start = time.perf_counter()


    results = model.predict(

        source=str(image_path),

        imgsz=OCR_IMAGE_SIZE,

        conf=CONF_THRESHOLD,

        iou=IOU_THRESHOLD,

        device=DEVICE,

        verbose=False,

    )


    torch.cuda.synchronize()

    end = time.perf_counter()


    latency_ms = (
        end - start
    ) * 1000.0


    prediction = (
        prediction_to_string(
            results[0],
            model.names
        )
    )


    pt_latencies.append(
        latency_ms
    )


    pt_records.append({

        "image":
            image_path.name,

        "ground_truth":
            ground_truth,

        "prediction":
            prediction,

        "correct":
            ground_truth == prediction,

        "latency_ms":
            latency_ms,

    })


    if index % 50 == 0:

        print(
            f"Processed "
            f"{index}/{len(test_images)}"
        )


pt_ocr_metrics = (
    calculate_ocr_metrics(
        pt_records
    )
)


pt_performance = (
    calculate_latency_statistics(
        pt_latencies
    )
)


print(
    "\nFP32 PyTorch CUDA OCR results:"
)

print(
    f"  Character Accuracy : "
    f"{pt_ocr_metrics['character_accuracy'] * 100:.4f}%"
)

print(
    f"  CER                : "
    f"{pt_ocr_metrics['cer']:.6f}"
)

print(
    f"  Exact Match        : "
    f"{pt_ocr_metrics['exact_match_accuracy'] * 100:.4f}%"
)

print("\nPerformance:")

print(
    f"  Total time         : "
    f"{pt_performance['total_time_seconds']:.3f} s"
)

print(
    f"  Average latency    : "
    f"{pt_performance['average_latency_ms']:.3f} ms"
)

print(
    f"  Median latency     : "
    f"{pt_performance['median_latency_ms']:.3f} ms"
)

print(
    f"  P95 latency        : "
    f"{pt_performance['p95_latency_ms']:.3f} ms"
)

print(
    f"  Min latency        : "
    f"{pt_performance['min_latency_ms']:.3f} ms"
)

print(
    f"  Max latency        : "
    f"{pt_performance['max_latency_ms']:.3f} ms"
)

print(
    f"  FPS                : "
    f"{pt_performance['fps']:.3f}"
)


# ============================================================
# 12. PYTORCH DETECTION VALIDATION
# ============================================================

print("\n" + "=" * 90)
print("FP32 PYTORCH CUDA DETECTION METRICS")
print("=" * 90)


pt_val = model.val(

    data=DATA_YAML,

    imgsz=OCR_IMAGE_SIZE,

    batch=16,

    device=DEVICE,

    plots=False,

    verbose=False,

)


pt_detection_metrics = {

    "precision":
        float(
            pt_val.box.mp
        ),

    "recall":
        float(
            pt_val.box.mr
        ),

    "map50":
        float(
            pt_val.box.map50
        ),

    "map50_95":
        float(
            pt_val.box.map
        ),

}


print(
    f"Precision : "
    f"{pt_detection_metrics['precision']:.6f}"
)

print(
    f"Recall    : "
    f"{pt_detection_metrics['recall']:.6f}"
)

print(
    f"mAP50     : "
    f"{pt_detection_metrics['map50']:.6f}"
)

print(
    f"mAP50-95  : "
    f"{pt_detection_metrics['map50_95']:.6f}"
)


# ============================================================
# 13. EXPORT ONNX FP32
# ============================================================

print("\n" + "=" * 90)
print("EXPORTING YOLO11-OCR → ONNX FP32")
print("=" * 90)


print(
    f"Image size : "
    f"{OCR_IMAGE_SIZE}x{OCR_IMAGE_SIZE}"
)

print(
    "Precision  : FP32"
)

print(
    "Dynamic    : False"
)

print(
    "Simplify   : True"
)

print(
    "NMS        : True"
)

print(
    f"IoU        : {IOU_THRESHOLD}"
)

print(
    f"Confidence : {CONF_THRESHOLD}"
)


exported_path = model.export(

    format="onnx",

    imgsz=OCR_IMAGE_SIZE,

    half=False,

    dynamic=False,

    simplify=True,

    opset=17,

    nms=True,

)


print(
    f"\nUltralytics ONNX export:"
)

print(
    f"  {exported_path}"
)


final_onnx_path = os.path.join(

    OCR_FP32_DIR,

    "yolo11_ocr_fp32.onnx"

)


shutil.copy2(

    exported_path,

    final_onnx_path

)


onnx_size_mb = (

    os.path.getsize(
        final_onnx_path
    )
    / 1024**2

)


print(
    f"✓ ONNX copied to:"
)

print(
    f"  {final_onnx_path}"
)

print(
    f"ONNX model size: "
    f"{onnx_size_mb:.2f} MB"
)


# ============================================================
# 14. ONNX STRUCTURE
# ============================================================

print("\n" + "=" * 90)
print("VALIDATING ONNX STRUCTURE")
print("=" * 90)


onnx_model = onnx.load(
    final_onnx_path
)


onnx.checker.check_model(
    onnx_model
)


print(
    "✓ ONNX checker passed."
)


for graph_input in (
    onnx_model.graph.input
):

    shape = [

        dim.dim_value

        for dim
        in graph_input
        .type.tensor_type.shape.dim

    ]

    print(
        f"\nINPUT:"
    )

    print(
        f"  Name : "
        f"{graph_input.name}"
    )

    print(
        f"  Shape: "
        f"{shape}"
    )


for graph_output in (
    onnx_model.graph.output
):

    shape = [

        dim.dim_value

        for dim
        in graph_output
        .type.tensor_type.shape.dim

    ]

    print(
        f"\nOUTPUT:"
    )

    print(
        f"  Name : "
        f"{graph_output.name}"
    )

    print(
        f"  Shape: "
        f"{shape}"
    )


# ============================================================
# 15. CREATE ONNX CUDA SESSION
# ============================================================

print("\n" + "=" * 90)
print("INITIALIZING ONNX RUNTIME CUDA")
print("=" * 90)


# ------------------------------------------------------------
# Preload again after export/import activity
# ------------------------------------------------------------

try:

    ort.preload_dlls(
        cuda=True,
        cudnn=True,
        msvc=True,
    )

except Exception:
    pass


requested_providers = [

    (
        "CUDAExecutionProvider",
        {
            "device_id": 0,
        },
    ),

    "CPUExecutionProvider",

]


print(
    "Requested providers:"
)

for provider in requested_providers:

    print(
        f"  - {provider}"
    )


session = ort.InferenceSession(

    final_onnx_path,

    providers=requested_providers,

)


session_providers = (
    session.get_providers()
)


print(
    "\nSession providers:"
)

for provider in session_providers:

    print(
        f"  - {provider}"
    )


if (
    "CUDAExecutionProvider"
    not in
    session_providers
):

    raise RuntimeError(

        f"""
CRITICAL:

ONNX Runtime CUDAExecutionProvider failed
to initialize.

Session providers:
{session_providers}

The ONNX CUDA benchmark is being stopped.

DO NOT interpret any ONNX timing as GPU timing.
"""

    )


print(
    "\n✓ ONNX Runtime CUDAExecutionProvider initialized."
)


# ============================================================
# 16. ONNX INPUT INFORMATION
# ============================================================

input_info = (
    session.get_inputs()[0]
)

input_name = (
    input_info.name
)

print(
    "\nONNX Runtime input:"
)

print(
    f"  Name : {input_name}"
)

print(
    f"  Shape: {input_info.shape}"
)

print(
    f"  Type : {input_info.type}"
)


# ============================================================
# 17. PREPROCESSING
# ============================================================

def preprocess_image(
    image_path,
    image_size
):

    image = cv2.imread(
        str(image_path)
    )

    if image is None:

        raise RuntimeError(
            f"Unable to read: "
            f"{image_path}"
        )


    image = cv2.resize(

        image,

        (
            image_size,
            image_size
        ),

        interpolation=cv2.INTER_LINEAR

    )


    image = cv2.cvtColor(

        image,

        cv2.COLOR_BGR2RGB

    )


    image = image.transpose(
        2,
        0,
        1
    )


    image = (

        image.astype(
            np.float32
        )
        / 255.0

    )


    image = np.expand_dims(
        image,
        axis=0
    )


    return image


# ============================================================
# 18. CORRECT NMS OUTPUT DECODER
# ============================================================

def decode_onnx_nms_output(
    output,
    names
):

    """
    YOLO11 ONNX exported with NMS=True.

    Expected output:

        [1, 300, 6]

    Each detection:

        [x1, y1, x2, y2, confidence, class_id]
    """


    output = np.asarray(
        output
    )


    if output.ndim == 3:

        output = output[0]


    if output.ndim != 2:

        raise RuntimeError(

            f"Unexpected ONNX output shape: "
            f"{output.shape}"

        )


    if output.shape[1] != 6:

        raise RuntimeError(

            f"Expected NMS output "
            f"[N, 6], got "
            f"{output.shape}"

        )


    characters = []


    for row in output:

        x1 = float(
            row[0]
        )

        y1 = float(
            row[1]
        )

        x2 = float(
            row[2]
        )

        y2 = float(
            row[3]
        )

        confidence = float(
            row[4]
        )

        class_id = int(
            row[5]
        )


        # Empty NMS rows
        if confidence <= 0:
            continue


        if confidence < CONF_THRESHOLD:
            continue


        if class_id not in names:
            continue


        x_center = (
            (x1 + x2)
            / 2.0
        )

        y_center = (
            (y1 + y2)
            / 2.0
        )


        # Convert to normalized coordinates
        if x_center > 1.0:

            x_center /= OCR_IMAGE_SIZE

        if y_center > 1.0:

            y_center /= OCR_IMAGE_SIZE


        characters.append({

            "class_id":
                class_id,

            "char":
                str(names[class_id]),

            "x_center":
                x_center,

            "y_center":
                y_center,

            "confidence":
                confidence,

        })


    characters = sort_characters(

        characters,

        OCR_SORT_DIRECTION

    )


    return "".join(

        item["char"]

        for item in characters

    )


# ============================================================
# 19. ONNX CUDA WARM-UP
# ============================================================

print("\n" + "=" * 90)
print("ONNX RUNTIME CUDA WARM-UP")
print("=" * 90)


warmup_tensor = preprocess_image(

    test_images[0],

    OCR_IMAGE_SIZE

)


for _ in range(
    WARMUP_RUNS
):

    session.run(

        None,

        {
            input_name:
                warmup_tensor
        }

    )


print(
    f"✓ Completed "
    f"{WARMUP_RUNS} warm-up runs."
)


# ============================================================
# 20. ONNX CUDA OCR BENCHMARK
# ============================================================

print("\n" + "=" * 90)
print("ONNX FP32 CUDA OCR EVALUATION")
print("=" * 90)


onnx_records = []

onnx_latencies = []

onnx_preprocess_times = []


for index, image_path in enumerate(

    test_images,

    start=1

):

    label_path = (

        Path(TEST_LABELS)

        / f"{image_path.stem}.txt"

    )


    annotations = (
        load_yolo_label(
            str(label_path)
        )
    )


    ground_truth = (
        annotations_to_string(
            annotations,
            model.names
        )
    )


    # --------------------------------------------------------
    # Preprocessing timing
    # --------------------------------------------------------

    preprocess_start = (
        time.perf_counter()
    )


    input_tensor = (
        preprocess_image(
            image_path,
            OCR_IMAGE_SIZE
        )
    )


    preprocess_end = (
        time.perf_counter()
    )


    preprocess_ms = (

        preprocess_end
        - preprocess_start

    ) * 1000.0


    # --------------------------------------------------------
    # ONNX Runtime timing
    # --------------------------------------------------------

    start = (
        time.perf_counter()
    )


    outputs = session.run(

        None,

        {
            input_name:
                input_tensor
        }

    )


    end = (
        time.perf_counter()
    )


    inference_ms = (

        end - start

    ) * 1000.0


    prediction = (
        decode_onnx_nms_output(
            outputs[0],
            model.names
        )
    )


    onnx_latencies.append(
        inference_ms
    )

    onnx_preprocess_times.append(
        preprocess_ms
    )


    onnx_records.append({

        "image":
            image_path.name,

        "ground_truth":
            ground_truth,

        "prediction":
            prediction,

        "correct":
            ground_truth == prediction,

        "preprocess_latency_ms":
            preprocess_ms,

        "inference_latency_ms":
            inference_ms,

    })


    if index % 50 == 0:

        print(
            f"Processed "
            f"{index}/{len(test_images)}"
        )


onnx_ocr_metrics = (
    calculate_ocr_metrics(
        onnx_records
    )
)


onnx_performance = (
    calculate_latency_statistics(
        onnx_latencies
    )
)


onnx_preprocess_mean = (
    float(
        np.mean(
            onnx_preprocess_times
        )
    )
)


print(
    "\nONNX FP32 CUDA OCR results:"
)

print(
    f"  Character Accuracy : "
    f"{onnx_ocr_metrics['character_accuracy'] * 100:.4f}%"
)

print(
    f"  CER                : "
    f"{onnx_ocr_metrics['cer']:.6f}"
)

print(
    f"  Exact Match        : "
    f"{onnx_ocr_metrics['exact_match_accuracy'] * 100:.4f}%"
)

print("\nONNX Performance:")

print(
    f"  Total inference    : "
    f"{onnx_performance['total_time_seconds']:.3f} s"
)

print(
    f"  Average latency    : "
    f"{onnx_performance['average_latency_ms']:.3f} ms"
)

print(
    f"  Median latency     : "
    f"{onnx_performance['median_latency_ms']:.3f} ms"
)

print(
    f"  P95 latency        : "
    f"{onnx_performance['p95_latency_ms']:.3f} ms"
)

print(
    f"  Min latency        : "
    f"{onnx_performance['min_latency_ms']:.3f} ms"
)

print(
    f"  Max latency        : "
    f"{onnx_performance['max_latency_ms']:.3f} ms"
)

print(
    f"  FPS                : "
    f"{onnx_performance['fps']:.3f}"
)

print(
    f"  Mean preprocessing : "
    f"{onnx_preprocess_mean:.3f} ms"
)


# ============================================================
# 21. SAMPLE COMPARISON
# ============================================================

print("\n" + "=" * 90)
print("PYTORCH ↔ ONNX SAMPLE COMPARISON")
print("=" * 90)


sample_mismatches = []


for pt_record, onnx_record in zip(

    pt_records,

    onnx_records

):

    if (

        pt_record["prediction"]

        !=

        onnx_record["prediction"]

    ):

        sample_mismatches.append({

            "image":
                pt_record["image"],

            "ground_truth":
                pt_record["ground_truth"],

            "pytorch":
                pt_record["prediction"],

            "onnx":
                onnx_record["prediction"],

        })


print(
    f"Samples evaluated: "
    f"{len(test_images)}"
)

print(
    f"PT/ONNX prediction mismatches: "
    f"{len(sample_mismatches)}"
)


if sample_mismatches:

    print(
        "\nFirst 10 mismatches:"
    )

    for item in sample_mismatches[:10]:

        print(
            f"\nImage: {item['image']}"
        )

        print(
            f"GT      : "
            f"{item['ground_truth']}"
        )

        print(
            f"PyTorch : "
            f"{item['pytorch']}"
        )

        print(
            f"ONNX    : "
            f"{item['onnx']}"
        )

else:

    print(
        "\n✓ PyTorch and ONNX predictions "
        "are identical for all samples."
    )


# ============================================================
# 22. PERFORMANCE COMPARISON
# ============================================================

speedup = (

    pt_performance[
        "average_latency_ms"
    ]

    /

    onnx_performance[
        "average_latency_ms"
    ]

)


fps_improvement = (

    (
        onnx_performance["fps"]
        /
        pt_performance["fps"]
    )

    - 1.0

) * 100.0


print("\n" + "=" * 90)
print("PERFORMANCE COMPARISON")
print("=" * 90)


print(
    f"""
                       PyTorch FP32       ONNX FP32 CUDA
------------------------------------------------------------
Average latency       {pt_performance['average_latency_ms']:>10.3f} ms   {onnx_performance['average_latency_ms']:>10.3f} ms
Median latency        {pt_performance['median_latency_ms']:>10.3f} ms   {onnx_performance['median_latency_ms']:>10.3f} ms
P95 latency           {pt_performance['p95_latency_ms']:>10.3f} ms   {onnx_performance['p95_latency_ms']:>10.3f} ms
Minimum latency       {pt_performance['min_latency_ms']:>10.3f} ms   {onnx_performance['min_latency_ms']:>10.3f} ms
Maximum latency       {pt_performance['max_latency_ms']:>10.3f} ms   {onnx_performance['max_latency_ms']:>10.3f} ms
FPS                   {pt_performance['fps']:>10.3f}       {onnx_performance['fps']:>10.3f}

Speedup:
    {speedup:.3f}x

FPS improvement:
    {fps_improvement:+.2f}%
"""
)


# ============================================================
# 23. OCR COMPARISON
# ============================================================

print("=" * 90)
print("OCR ACCURACY COMPARISON")
print("=" * 90)


print(
    f"""
Character Accuracy:
    PyTorch : {pt_ocr_metrics['character_accuracy'] * 100:.4f}%
    ONNX    : {onnx_ocr_metrics['character_accuracy'] * 100:.4f}%

CER:
    PyTorch : {pt_ocr_metrics['cer']:.6f}
    ONNX    : {onnx_ocr_metrics['cer']:.6f}

Exact Match:
    PyTorch : {pt_ocr_metrics['exact_match_accuracy'] * 100:.4f}%
    ONNX    : {onnx_ocr_metrics['exact_match_accuracy'] * 100:.4f}%
"""
)


# ============================================================
# 24. SAVE RESULTS
# ============================================================

pt_results_file = os.path.join(

    OCR_VALIDATION_DIR,

    "step2_ocr_fp32_pytorch_predictions.json"

)


onnx_results_file = os.path.join(

    OCR_VALIDATION_DIR,

    "step2_ocr_fp32_onnx_cuda_predictions.json"

)


with open(
    pt_results_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        pt_records,
        f,
        indent=4,
        ensure_ascii=False
    )


with open(
    onnx_results_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        onnx_records,
        f,
        indent=4,
        ensure_ascii=False
    )


# ============================================================
# 25. FINAL SUMMARY
# ============================================================

summary = {

    "step":
        "2-OCR",

    "timestamp":
        time.strftime(
            "%Y-%m-%d %H:%M:%S"
        ),

    "model":
        MODEL_PATH,

    "onnx_model":
        final_onnx_path,

    "gpu":
        GPU_NAME,

    "gpu_memory_gb":
        GPU_MEMORY_GB,

    "input_size":
        [
            OCR_IMAGE_SIZE,
            OCR_IMAGE_SIZE
        ],

    "test_images":
        len(test_images),

    "parameters":
        TOTAL_PARAMETERS,

    "confidence_threshold":
        CONF_THRESHOLD,

    "iou_threshold":
        IOU_THRESHOLD,

    "sort_direction":
        OCR_SORT_DIRECTION,

    "pytorch_fp32_cuda":
        {

            "ocr_metrics":
                pt_ocr_metrics,

            "detection_metrics":
                pt_detection_metrics,

            "performance":
                pt_performance,

        },

    "onnx_fp32_cuda":
        {

            "ocr_metrics":
                onnx_ocr_metrics,

            "performance":
                onnx_performance,

            "mean_preprocess_latency_ms":
                onnx_preprocess_mean,

            "execution_provider":
                "CUDAExecutionProvider",

        },

    "comparison":
        {

            "latency_speedup":
                speedup,

            "fps_improvement_percent":
                fps_improvement,

            "prediction_mismatches":
                len(
                    sample_mismatches
                ),

        },

    "onnx_model_size_mb":
        onnx_size_mb,

}


summary_file = os.path.join(

    OCR_VALIDATION_DIR,

    "step2_ocr_fp32_cuda_summary.json"

)


with open(

    summary_file,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        summary,

        f,

        indent=4,

        ensure_ascii=False

    )


# ============================================================
# 26. FINAL REPORT
# ============================================================

print("\n" + "=" * 90)
print("STEP 2-OCR — COMPLETED")
print("=" * 90)


print(
    f"""
TEST SET
--------
Images                 : {len(test_images)}
Input                  : {OCR_IMAGE_SIZE}x{OCR_IMAGE_SIZE}
Parameters             : {TOTAL_PARAMETERS:,}
GPU                    : {GPU_NAME}

PYTORCH FP32 CUDA
-----------------
Character Accuracy     : {pt_ocr_metrics['character_accuracy'] * 100:.4f}%
CER                    : {pt_ocr_metrics['cer']:.6f}
Exact Match            : {pt_ocr_metrics['exact_match_accuracy'] * 100:.4f}%

Average latency        : {pt_performance['average_latency_ms']:.3f} ms
Median latency         : {pt_performance['median_latency_ms']:.3f} ms
P95 latency            : {pt_performance['p95_latency_ms']:.3f} ms
Min latency            : {pt_performance['min_latency_ms']:.3f} ms
Max latency            : {pt_performance['max_latency_ms']:.3f} ms
FPS                    : {pt_performance['fps']:.3f}

Precision              : {pt_detection_metrics['precision']:.6f}
Recall                 : {pt_detection_metrics['recall']:.6f}
mAP50                  : {pt_detection_metrics['map50']:.6f}
mAP50-95               : {pt_detection_metrics['map50_95']:.6f}

ONNX FP32 CUDA
--------------
Character Accuracy     : {onnx_ocr_metrics['character_accuracy'] * 100:.4f}%
CER                    : {onnx_ocr_metrics['cer']:.6f}
Exact Match            : {onnx_ocr_metrics['exact_match_accuracy'] * 100:.4f}%

Average latency        : {onnx_performance['average_latency_ms']:.3f} ms
Median latency         : {onnx_performance['median_latency_ms']:.3f} ms
P95 latency            : {onnx_performance['p95_latency_ms']:.3f} ms
Min latency            : {onnx_performance['min_latency_ms']:.3f} ms
Max latency            : {onnx_performance['max_latency_ms']:.3f} ms
FPS                    : {onnx_performance['fps']:.3f}

Mean preprocessing     : {onnx_preprocess_mean:.3f} ms

ONNX MODEL
----------
Size                   : {onnx_size_mb:.2f} MB
Execution Provider     : CUDAExecutionProvider

PT → ONNX
---------
Latency speedup        : {speedup:.3f}x
FPS improvement        : {fps_improvement:+.2f}%
Prediction mismatches  : {len(sample_mismatches)} / {len(test_images)}

SUMMARY
-------
{summary_file}

✓ Original best.pt remains untouched.
✓ Test dataset remains untouched.
✓ ONNX CUDA was explicitly verified.
✓ Warm-up runs excluded from benchmark.
✓ Latency/FPS statistics saved.
"""
)


STEP 2-OCR — FP32 PYTORCH CUDA + ONNX FP32 CUDA
Ultralytics : __pending__
Ultralytics : 8.4.137
ONNX        : 1.22.0
ONNX Runtime: 1.26.0
Python      : 3.13.15

VERIFYING CUDA
✓ CUDA available
✓ GPU: Tesla T4
✓ GPU memory: 14.56 GB
✓ PyTorch CUDA: 12.8

VERIFYING ONNX RUNTIME CUDA
Available providers:
  - TensorrtExecutionProvider
  - CUDAExecutionProvider
  - CPUExecutionProvider
✓ CUDA/cuDNN libraries preloaded.

VERIFYING STEP 1C-OCR REGISTRY
✓ Model:
  /content/drive/MyDrive/Stage_PFA/experiments/YOLOV11_OCR/weights/best.pt
✓ Test images:
  /content/drive/MyDrive/Stage_PFA/datasets/test/images
✓ Test labels:
  /content/drive/MyDrive/Stage_PFA/datasets/test/labels
✓ Data YAML:
  /content/drive/MyDrive/Stage_PFA/datasets/data.yaml

LOADING YOLO11-OCR MODEL
✓ Model loaded.
✓ Model converted to FP32.
Task: detect
Classes: {0: '0', 1: '1', 2: '2', 3: '3', 4: '4', 5: '5', 6: '6', 7: '7', 8: '8', 9: '9', 10: 'a', 11: 'b', 12: 'd', 13: 'gr', 14: 'h', 15: 't', 16: 'w', 17: 'waw', 18: 'y'}
T

In [2]:
# =============================================================================
# STEP 3-OCR — DEPENDENCY SETUP
# ONNX FP16 CUDA Optimization
# =============================================================================
#
# IMPORTANT:
# If Step 2 already completed successfully in this SAME runtime environment,
# you normally DO NOT need to install anything here.
#
# Run this cell first.
# If it reports that packages are already installed correctly, continue directly
# to CELL 2.
#
# If packages are installed/changed, restart the Colab runtime before CELL 2.
# =============================================================================

import sys
import subprocess
import importlib.util

print("=" * 90)
print("STEP 3-OCR — DEPENDENCY SETUP")
print("=" * 90)

print(f"Python: {sys.version}")

# -------------------------------------------------------------------------
# Required packages
# -------------------------------------------------------------------------

required = {
    "onnx": "onnx",
    "onnxruntime": "onnxruntime-gpu",
    "onnxconverter_common": "onnxconverter-common",
}

installed_anything = False

for module_name, package_name in required.items():

    if importlib.util.find_spec(module_name) is None:

        print(f"\nInstalling {package_name}...")

        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            package_name
        ])

        installed_anything = True

        print(f"✓ Installed {package_name}")

    else:
        print(f"✓ {module_name} already installed")


# -------------------------------------------------------------------------
# Verify imports
# -------------------------------------------------------------------------

print("\n" + "=" * 90)
print("VERIFYING PACKAGES")
print("=" * 90)

import onnx
import onnxruntime as ort

print(f"✓ ONNX Runtime : {ort.__version__}")
print(f"✓ ONNX         : {onnx.__version__}")

try:

    import onnxconverter_common

    print("✓ onnxconverter-common available")

except Exception:

    print("⚠ onnxconverter-common not available")


# -------------------------------------------------------------------------
# CUDA provider verification
# -------------------------------------------------------------------------

print("\n" + "=" * 90)
print("VERIFYING ONNX RUNTIME PROVIDERS")
print("=" * 90)

providers = ort.get_available_providers()

for provider in providers:
    print(f"  - {provider}")

if "CUDAExecutionProvider" in providers:

    print("\n✓ CUDAExecutionProvider is available.")

else:

    print("\n✗ CUDAExecutionProvider is NOT available.")
    print("\nDo not continue to CELL 2.")

# -------------------------------------------------------------------------
# Final message
# -------------------------------------------------------------------------

print("\n" + "=" * 90)

if installed_anything:

    print("⚠ PACKAGES WERE INSTALLED/CHANGED.")
    print("RESTART THE COLAB RUNTIME BEFORE RUNNING CELL 2.")

else:

    print("✓ NO PACKAGE CHANGES WERE REQUIRED.")
    print("You can proceed directly to CELL 2.")

print("=" * 90)

STEP 3-OCR — DEPENDENCY SETUP
Python: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
✓ onnx already installed
✓ onnxruntime already installed

Installing onnxconverter-common...
✓ Installed onnxconverter-common

VERIFYING PACKAGES
✓ ONNX Runtime : 1.26.0
✓ ONNX         : 1.22.0
✓ onnxconverter-common available

VERIFYING ONNX RUNTIME PROVIDERS
  - TensorrtExecutionProvider
  - CUDAExecutionProvider
  - CPUExecutionProvider

✓ CUDAExecutionProvider is available.

⚠ PACKAGES WERE INSTALLED/CHANGED.
RESTART THE COLAB RUNTIME BEFORE RUNNING CELL 2.


In [3]:
!pip uninstall -y onnxruntime onnxruntime-gpu

!pip install -q "onnxruntime-gpu==1.26.0"

Found existing installation: onnxruntime-gpu 1.26.0
Uninstalling onnxruntime-gpu-1.26.0:
  Successfully uninstalled onnxruntime-gpu-1.26.0


In [10]:
# =============================================================================
# STEP 3-OCR — ONNX FP16 CUDA OPTIMIZATION
# =============================================================================
#
# Purpose:
#   Generate a valid FP16 ONNX version of the validated YOLO11-OCR model,
#   validate it, benchmark it on NVIDIA Tesla T4 using CUDAExecutionProvider,
#   and compare it against the Step 2 ONNX FP32 CUDA baseline.
#
# IMPORTANT:
#   The previous FP16 conversion approach using onnxconverter_common caused
#   graph/type errors with this YOLO11 model.
#
#   Therefore, this version uses Ultralytics' native ONNX FP16 export from
#   the ORIGINAL best.pt model. This avoids manually modifying the already
#   exported FP32 graph and is considerably safer for this model.
#
# Comparison:
#
#   Step 2:
#       ONNX FP32 CUDA
#              ↓
#   Step 3:
#       ONNX FP16 CUDA
#
# Metrics:
#   - Character Accuracy
#   - CER
#   - Exact Match
#   - Normalized Similarity
#   - Mean latency
#   - Median latency
#   - P95 latency
#   - Min latency
#   - Max latency
#   - FPS
#   - Preprocessing time
#   - Model size
#   - Model size reduction
#   - Prediction mismatches
#   - Latency speedup
#   - Latency reduction
#   - FPS improvement
#
# Original files remain untouched.
# =============================================================================


# =============================================================================
# IMPORTS
# =============================================================================

import os
import sys
import json
import time
import statistics
from pathlib import Path

import cv2
import numpy as np
import torch
import onnx
import onnxruntime as ort

from ultralytics import YOLO


# =============================================================================
# HEADER
# =============================================================================

print("=" * 90)
print("STEP 3-OCR — ONNX FP16 CUDA OPTIMIZATION")
print("=" * 90)


# =============================================================================
# CONFIGURATION
# =============================================================================

PT_MODEL = (
    "/content/drive/MyDrive/Stage_PFA/"
    "experiments/YOLOV11_OCR/weights/best.pt"
)

FP32_ONNX = (
    "/content/drive/MyDrive/Stage_PFA/"
    "OCR_OPTIMIZATION/02_onnx_fp32/"
    "yolo11_ocr_fp32.onnx"
)

OUTPUT_DIR = (
    "/content/drive/MyDrive/Stage_PFA/"
    "OCR_OPTIMIZATION/03_onnx_fp16"
)

VALIDATION_DIR = (
    "/content/drive/MyDrive/Stage_PFA/"
    "OCR_OPTIMIZATION/06_validation"
)

FP16_ONNX = os.path.join(
    OUTPUT_DIR,
    "yolo11_ocr_fp16.onnx"
)

SUMMARY_JSON = os.path.join(
    VALIDATION_DIR,
    "step3_ocr_fp16_cuda_summary.json"
)

TEST_IMAGES_DIR = (
    "/content/drive/MyDrive/Stage_PFA/"
    "datasets/test/images"
)

TEST_LABELS_DIR = (
    "/content/drive/MyDrive/Stage_PFA/"
    "datasets/test/labels"
)

DATA_YAML = (
    "/content/drive/MyDrive/Stage_PFA/"
    "datasets/data.yaml"
)

IMG_SIZE = 608
DEVICE_ID = 0
WARMUP_RUNS = 10


# =============================================================================
# CREATE DIRECTORIES
# =============================================================================

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

os.makedirs(
    VALIDATION_DIR,
    exist_ok=True
)


# =============================================================================
# VERSION INFORMATION
# =============================================================================

print(
    f"Python       : {sys.version.split()[0]}"
)

print(
    f"PyTorch      : {torch.__version__}"
)

print(
    "Ultralytics  : 8.4.137"
)

print(
    f"ONNX         : {onnx.__version__}"
)

print(
    f"ONNX Runtime : {ort.__version__}"
)


# =============================================================================
# VERIFY CUDA
# =============================================================================

print("\n" + "=" * 90)
print("VERIFYING CUDA")
print("=" * 90)


if not torch.cuda.is_available():

    raise RuntimeError(
        "CUDA is not available in PyTorch."
    )


gpu_name = torch.cuda.get_device_name(
    DEVICE_ID
)

gpu_memory = (
    torch.cuda.get_device_properties(
        DEVICE_ID
    ).total_memory
    / (1024 ** 3)
)


print("✓ CUDA available")

print(
    f"✓ GPU: {gpu_name}"
)

print(
    f"✓ GPU memory: {gpu_memory:.2f} GB"
)

print(
    f"✓ PyTorch CUDA: {torch.version.cuda}"
)


# =============================================================================
# VERIFY ONNX RUNTIME CUDA
# =============================================================================

print("\n" + "=" * 90)
print("VERIFYING ONNX RUNTIME CUDA")
print("=" * 90)


available_providers = (
    ort.get_available_providers()
)


print("Available providers:")

for provider in available_providers:

    print(
        f"  - {provider}"
    )


if "CUDAExecutionProvider" not in available_providers:

    raise RuntimeError(
        """
CRITICAL ERROR:

CUDAExecutionProvider is not available.

Do NOT continue with FP16 CUDA benchmarking.
"""
    )


print(
    "\n✓ CUDAExecutionProvider is available."
)


# =============================================================================
# VERIFY REGISTRY
# =============================================================================

print("\n" + "=" * 90)
print("VERIFYING STEP 1C / STEP 2 REGISTRY")
print("=" * 90)


required_files = {

    "PyTorch model":
        PT_MODEL,

    "ONNX FP32 model":
        FP32_ONNX,

    "Data YAML":
        DATA_YAML,
}


required_dirs = {

    "Test images":
        TEST_IMAGES_DIR,

    "Test labels":
        TEST_LABELS_DIR,
}


for name, path in required_files.items():

    if not os.path.isfile(path):

        raise FileNotFoundError(
            f"{name} not found:\n{path}"
        )

    print(
        f"✓ {name}:"
    )

    print(
        f"  {path}"
    )


for name, path in required_dirs.items():

    if not os.path.isdir(path):

        raise FileNotFoundError(
            f"{name} directory not found:\n{path}"
        )

    print(
        f"✓ {name}:"
    )

    print(
        f"  {path}"
    )


# =============================================================================
# VERIFY STEP 2 FP32 ONNX
# =============================================================================

print("\n" + "=" * 90)
print("VALIDATING STEP 2 FP32 ONNX")
print("=" * 90)


try:

    onnx.checker.check_model(
        FP32_ONNX
    )

except Exception as e:

    raise RuntimeError(
        "The Step 2 FP32 ONNX model failed validation.\n\n"
        f"Error:\n{e}"
    )


print(
    "✓ Step 2 FP32 ONNX checker passed."
)


fp32_model_check = onnx.load(
    FP32_ONNX
)


print("\nFP32 ONNX input:")

for inp in fp32_model_check.graph.input:

    print(
        f"  Name : {inp.name}"
    )

    print(
        f"  Shape: "
        f"{[d.dim_value if d.dim_value > 0 else d.dim_param for d in inp.type.tensor_type.shape.dim]}"
    )

    print(
        f"  Type : {inp.type.tensor_type.elem_type}"
    )


print("\nFP32 ONNX output:")

for output in fp32_model_check.graph.output:

    print(
        f"  Name : {output.name}"
    )

    print(
        f"  Shape: "
        f"{[d.dim_value if d.dim_value > 0 else d.dim_param for d in output.type.tensor_type.shape.dim]}"
    )


# =============================================================================
# LOAD YOLO MODEL
# =============================================================================

print("\n" + "=" * 90)
print("LOADING YOLO11-OCR MODEL")
print("=" * 90)


model = YOLO(
    PT_MODEL
)


model.model.float()


print(
    "✓ Model loaded."
)

print(
    "✓ Model converted to FP32."
)

print(
    f"Task: {model.task}"
)

print(
    f"Classes: {model.names}"
)


total_params = sum(
    p.numel()
    for p in model.model.parameters()
)


print(
    f"Total parameters: {total_params:,}"
)


class_names = model.names


# =============================================================================
# PREPARE TEST DATASET
# =============================================================================

print("\n" + "=" * 90)
print("PREPARING OCR TEST SET")
print("=" * 90)


image_extensions = {

    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
}


test_images = sorted(

    [

        p

        for p in Path(
            TEST_IMAGES_DIR
        ).iterdir()

        if p.suffix.lower()
        in image_extensions

    ]
)


if len(test_images) == 0:

    raise RuntimeError(
        "No test images found."
    )


print(
    f"Test images: {len(test_images)}"
)


# =============================================================================
# OCR GROUND-TRUTH FUNCTIONS
# =============================================================================

def load_ground_truth(label_file):

    if not os.path.isfile(
        label_file
    ):

        return ""


    entries = []


    with open(
        label_file,
        "r",
        encoding="utf-8"
    ) as f:

        lines = [

            line.strip()

            for line in f

            if line.strip()

        ]


    for line in lines:

        parts = line.split()


        if len(parts) < 5:

            continue


        class_id = int(
            float(parts[0])
        )


        center_x = float(
            parts[1]
        )


        entries.append(
            (
                center_x,
                class_id
            )
        )


    entries.sort(
        key=lambda x: x[0]
    )


    text = ""


    for _, class_id in entries:

        if class_id in class_names:

            text += str(
                class_names[class_id]
            )


    return text


def get_gt_for_image(
    image_path
):

    label_path = (

        Path(
            TEST_LABELS_DIR
        )

        / f"{image_path.stem}.txt"

    )


    return load_ground_truth(
        str(label_path)
    )


# =============================================================================
# OCR METRICS
# =============================================================================

def edit_distance(
    a,
    b
):

    if a == b:

        return 0


    if len(a) == 0:

        return len(b)


    if len(b) == 0:

        return len(a)


    previous = list(
        range(
            len(b) + 1
        )
    )


    for i, ca in enumerate(
        a,
        start=1
    ):

        current = [i]


        for j, cb in enumerate(
            b,
            start=1
        ):

            insertion = (
                current[j - 1]
                + 1
            )


            deletion = (
                previous[j]
                + 1
            )


            substitution = (
                previous[j - 1]
                + (ca != cb)
            )


            current.append(
                min(
                    insertion,
                    deletion,
                    substitution
                )
            )


        previous = current


    return previous[-1]


def calculate_metrics(
    ground_truths,
    predictions
):

    total_chars = 0

    correct_chars = 0

    total_edit_distance = 0

    exact_matches = 0


    for gt, pred in zip(
        ground_truths,
        predictions
    ):

        distance = edit_distance(
            gt,
            pred
        )


        total_edit_distance += (
            distance
        )


        total_chars += len(gt)


        max_len = max(
            len(gt),
            len(pred)
        )


        if max_len > 0:

            correct_chars += (
                max_len - distance
            )


        if gt == pred:

            exact_matches += 1


    character_accuracy = (

        correct_chars
        / total_chars

        if total_chars > 0

        else 0.0

    )


    cer = (

        total_edit_distance
        / total_chars

        if total_chars > 0

        else 0.0

    )


    exact_match = (

        exact_matches
        / len(ground_truths)

        if ground_truths

        else 0.0

    )


    normalized_similarity = (
        1.0 - cer
    )


    return {

        "character_accuracy":
            character_accuracy,

        "cer":
            cer,

        "exact_match":
            exact_match,

        "normalized_similarity":
            normalized_similarity,

    }


# =============================================================================
# IMAGE PREPROCESSING
# =============================================================================

print("\n" + "=" * 90)
print("PREPARING INPUT IMAGES")
print("=" * 90)


def preprocess_image(
    image_path,
    target_dtype=np.float32
):

    start = time.perf_counter()


    image = cv2.imread(
        str(image_path)
    )


    if image is None:

        raise RuntimeError(
            f"Unable to read image:\n{image_path}"
        )


    image = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2RGB
    )


    image = cv2.resize(
        image,
        (
            IMG_SIZE,
            IMG_SIZE
        ),
        interpolation=cv2.INTER_LINEAR
    )


    image = image.astype(
        target_dtype
    ) / target_dtype(255.0)


    image = np.transpose(
        image,
        (2, 0, 1)
    )


    image = np.expand_dims(
        image,
        axis=0
    )


    preprocessing_time = (
        time.perf_counter()
        - start
    )


    return (
        image,
        preprocessing_time
    )


# =============================================================================
# GENERATE FP16 ONNX
# =============================================================================
#
# IMPORTANT:
#
# We deliberately do NOT use:
#
#   onnxconverter_common.float16
#
# because it previously produced invalid graph structures for this model:
#
#   - Resize output type mismatch
#   - graph_input_cast_0 topological ordering error
#
# Instead, we use Ultralytics' native ONNX exporter with half=True.
#


print("\n" + "=" * 90)
print("EXPORTING YOLO11-OCR → ONNX FP16")
print("=" * 90)


print(
    "Source PyTorch model:"
)

print(
    f"  {PT_MODEL}"
)


print(
    "\nTarget FP16 ONNX:"
)

print(
    f"  {FP16_ONNX}"
)


if os.path.isfile(
    FP16_ONNX
):

    print(
        "\nRemoving previous FP16 model..."
    )

    os.remove(
        FP16_ONNX
    )

    print(
        "✓ Previous FP16 model removed."
    )


print(
    "\nStarting native Ultralytics FP16 ONNX export..."
)


try:

    exported_path = model.export(

        format="onnx",

        imgsz=IMG_SIZE,

        half=True,

        dynamic=False,

        simplify=True,

        opset=17,

        nms=True,

        device=DEVICE_ID,

    )

except Exception as e:

    raise RuntimeError(
        "FP16 ONNX export failed.\n\n"
        f"Error:\n{e}"
    )


print(
    "\n✓ Ultralytics FP16 export completed."
)

print(
    f"Ultralytics output:\n{exported_path}"
)


# =============================================================================
# LOCATE EXPORTED MODEL
# =============================================================================

exported_path = Path(
    exported_path
)


if not exported_path.is_file():

    raise FileNotFoundError(
        "Ultralytics reported a successful export, "
        "but the ONNX file could not be found:\n"
        f"{exported_path}"
    )


# =============================================================================
# COPY TO STEP 3 DIRECTORY
# =============================================================================

if str(
    exported_path
) != str(
    Path(FP16_ONNX)
):

    import shutil

    shutil.copy2(
        exported_path,
        FP16_ONNX
    )


print(
    "\n✓ FP16 ONNX copied to:"
)

print(
    f"  {FP16_ONNX}"
)


# =============================================================================
# MODEL SIZE
# =============================================================================

fp32_size_mb = (

    os.path.getsize(
        FP32_ONNX
    )

    / (1024 ** 2)

)


fp16_size_mb = (

    os.path.getsize(
        FP16_ONNX
    )

    / (1024 ** 2)

)


size_reduction = (

    (
        fp32_size_mb
        - fp16_size_mb
    )

    / fp32_size_mb

) * 100.0


print("\nModel size comparison:")

print(
    f"  FP32 : {fp32_size_mb:.2f} MB"
)

print(
    f"  FP16 : {fp16_size_mb:.2f} MB"
)

print(
    f"  Reduction : {size_reduction:.2f}%"
)


# =============================================================================
# VALIDATE FP16 ONNX
# =============================================================================

print("\n" + "=" * 90)
print("VALIDATING FP16 ONNX STRUCTURE")
print("=" * 90)


try:

    onnx.checker.check_model(
        FP16_ONNX
    )

except Exception as e:

    raise RuntimeError(
        """
FP16 ONNX checker FAILED.

The exported FP16 model is not valid.

Error:
"""
        + str(e)
    )


print(
    "✓ ONNX checker passed."
)


fp16_model_check = onnx.load(
    FP16_ONNX
)


print("\nFP16 ONNX INPUTS:")


for inp in fp16_model_check.graph.input:

    shape = [

        d.dim_value
        if d.dim_value > 0
        else d.dim_param

        for d
        in inp.type.tensor_type.shape.dim

    ]


    print(
        f"  Name : {inp.name}"
    )

    print(
        f"  Shape: {shape}"
    )

    print(
        f"  Type : "
        f"{inp.type.tensor_type.elem_type}"
    )


print("\nFP16 ONNX OUTPUTS:")


for output in fp16_model_check.graph.output:

    shape = [

        d.dim_value
        if d.dim_value > 0
        else d.dim_param

        for d
        in output.type.tensor_type.shape.dim

    ]


    print(
        f"  Name : {output.name}"
    )

    print(
        f"  Shape: {shape}"
    )

    print(
        f"  Type : "
        f"{output.type.tensor_type.elem_type}"
    )


# =============================================================================
# INITIALIZE FP16 ONNX RUNTIME CUDA
# =============================================================================

print("\n" + "=" * 90)
print("INITIALIZING ONNX RUNTIME FP16 CUDA")
print("=" * 90)


requested_providers = [

    (
        "CUDAExecutionProvider",
        {
            "device_id":
                DEVICE_ID
        }
    ),

    "CPUExecutionProvider",

]


print("Requested providers:")


for provider in requested_providers:

    print(
        f"  - {provider}"
    )


session_options = (
    ort.SessionOptions()
)


session_options.graph_optimization_level = (
    ort.GraphOptimizationLevel.ORT_ENABLE_ALL
)


session_options.log_severity_level = 3


try:

    fp16_session = (
        ort.InferenceSession(

            FP16_ONNX,

            sess_options=
                session_options,

            providers=
                requested_providers,

        )
    )

except Exception as e:

    raise RuntimeError(
        """
CRITICAL ERROR:

The FP16 ONNX model could not be loaded by
ONNX Runtime.

The model passed the ONNX checker, but ONNX Runtime
could not initialize the inference session.

Error:
"""
        + str(e)
    )


fp16_session_providers = (
    fp16_session.get_providers()
)


print(
    "\nSession providers:"
)


for provider in fp16_session_providers:

    print(
        f"  - {provider}"
    )


if (
    "CUDAExecutionProvider"
    not in fp16_session_providers
):

    raise RuntimeError(
        f"""
CRITICAL ERROR:

FP16 ONNX Runtime did not initialize
CUDAExecutionProvider.

Providers:
{fp16_session_providers}
"""
    )


print(
    "\n✓ ONNX Runtime CUDAExecutionProvider initialized."
)


# =============================================================================
# FP16 INPUT INFORMATION
# =============================================================================

fp16_input_info = (
    fp16_session.get_inputs()[0]
)


fp16_input_name = (
    fp16_input_info.name
)


fp16_input_type = (
    fp16_input_info.type
)


print(
    "\nFP16 ONNX Runtime input:"
)

print(
    f"  Name : {fp16_input_name}"
)

print(
    f"  Shape: {fp16_input_info.shape}"
)

print(
    f"  Type : {fp16_input_type}"
)


# =============================================================================
# DETERMINE INPUT DTYPE
# =============================================================================

if "float16" in fp16_input_type:

    FP16_INPUT_DTYPE = np.float16

elif "float" in fp16_input_type:

    FP16_INPUT_DTYPE = np.float32

else:

    raise RuntimeError(
        "Unsupported FP16 ONNX input type:\n"
        f"{fp16_input_type}"
    )


print(
    f"\n✓ Runtime input NumPy dtype: "
    f"{FP16_INPUT_DTYPE}"
)


# =============================================================================
# FP16 WARM-UP
# =============================================================================

print("\n" + "=" * 90)
print("ONNX RUNTIME FP16 CUDA WARM-UP")
print("=" * 90)


warmup_image, _ = (
    preprocess_image(

        test_images[0],

        target_dtype=
            FP16_INPUT_DTYPE

    )
)


for _ in range(
    WARMUP_RUNS
):

    fp16_session.run(

        None,

        {
            fp16_input_name:
                warmup_image
        }

    )


print(
    f"✓ Completed {WARMUP_RUNS} warm-up runs."
)


# =============================================================================
# FP16 OCR EVALUATION
# =============================================================================

print("\n" + "=" * 90)
print("ONNX FP16 CUDA OCR EVALUATION")
print("=" * 90)


ground_truths = []

predictions_fp16 = []

latencies_ms = []

preprocessing_times_ms = []


total_start = (
    time.perf_counter()
)


for index, image_path in enumerate(

    test_images,

    start=1

):

    # -------------------------------------------------------------------------
    # PREPROCESSING
    # -------------------------------------------------------------------------

    input_tensor, preprocessing_time = (
        preprocess_image(

            image_path,

            target_dtype=
                FP16_INPUT_DTYPE

        )
    )


    preprocessing_times_ms.append(

        preprocessing_time
        * 1000.0

    )


    # -------------------------------------------------------------------------
    # GPU INFERENCE
    # -------------------------------------------------------------------------

    start = (
        time.perf_counter()
    )


    outputs = (
        fp16_session.run(

            None,

            {
                fp16_input_name:
                    input_tensor
            }

        )
    )


    elapsed_ms = (

        time.perf_counter()
        - start

    ) * 1000.0


    latencies_ms.append(
        elapsed_ms
    )


    # -------------------------------------------------------------------------
    # DECODE YOLO OUTPUT
    # -------------------------------------------------------------------------

    output = outputs[0]


    if output.ndim == 3:

        output = output[0]


    detections = []


    for detection in output:

        if len(detection) < 6:

            continue


        x1, y1, x2, y2, conf, cls_id = (
            detection
        )


        conf = float(conf)


        if conf <= 0:

            continue


        cls_id = int(
            cls_id
        )


        if cls_id not in class_names:

            continue


        center_x = (

            float(x1)
            + float(x2)

        ) / 2.0


        detections.append(

            (
                center_x,
                cls_id,
                conf
            )

        )


    detections.sort(
        key=lambda x: x[0]
    )


    prediction = ""


    for _, cls_id, _ in detections:

        prediction += str(

            class_names[
                cls_id
            ]

        )


    gt = get_gt_for_image(
        image_path
    )


    ground_truths.append(
        gt
    )


    predictions_fp16.append(
        prediction
    )


    # -------------------------------------------------------------------------
    # PROGRESS
    # -------------------------------------------------------------------------

    if index % 50 == 0:

        print(
            f"Processed "
            f"{index}/{len(test_images)}"
        )


total_time = (

    time.perf_counter()
    - total_start

)


# =============================================================================
# CALCULATE FP16 OCR METRICS
# =============================================================================

fp16_metrics = calculate_metrics(

    ground_truths,

    predictions_fp16

)


# =============================================================================
# PERFORMANCE METRICS
# =============================================================================

average_latency_ms = (
    statistics.mean(
        latencies_ms
    )
)


median_latency_ms = (
    statistics.median(
        latencies_ms
    )
)


p95_latency_ms = float(

    np.percentile(
        latencies_ms,
        95
    )

)


min_latency_ms = min(
    latencies_ms
)


max_latency_ms = max(
    latencies_ms
)


fps = (

    1000.0
    / average_latency_ms

)


mean_preprocessing_ms = (
    statistics.mean(
        preprocessing_times_ms
    )
)


total_inference_time_s = (

    sum(latencies_ms)
    / 1000.0

)


# =============================================================================
# PRINT FP16 RESULTS
# =============================================================================

print("\n" + "=" * 90)
print("ONNX FP16 CUDA OCR RESULTS")
print("=" * 90)


print(
    f"Character Accuracy : "
    f"{fp16_metrics['character_accuracy'] * 100:.4f}%"
)


print(
    f"CER                : "
    f"{fp16_metrics['cer']:.6f}"
)


print(
    f"Exact Match        : "
    f"{fp16_metrics['exact_match'] * 100:.4f}%"
)


print("\nPerformance:")


print(
    f"  Total inference    : "
    f"{total_inference_time_s:.3f} s"
)


print(
    f"  Average latency    : "
    f"{average_latency_ms:.3f} ms"
)


print(
    f"  Median latency     : "
    f"{median_latency_ms:.3f} ms"
)


print(
    f"  P95 latency        : "
    f"{p95_latency_ms:.3f} ms"
)


print(
    f"  Min latency        : "
    f"{min_latency_ms:.3f} ms"
)


print(
    f"  Max latency        : "
    f"{max_latency_ms:.3f} ms"
)


print(
    f"  FPS                : "
    f"{fps:.3f}"
)


print(
    f"  Mean preprocessing : "
    f"{mean_preprocessing_ms:.3f} ms"
)


# =============================================================================
# LOAD FP32 ONNX CUDA SESSION
# =============================================================================

print("\n" + "=" * 90)
print("INITIALIZING FP32 ONNX CUDA REFERENCE")
print("=" * 90)


fp32_session = (
    ort.InferenceSession(

        FP32_ONNX,

        sess_options=
            session_options,

        providers=
            requested_providers,

    )
)


fp32_session_providers = (
    fp32_session.get_providers()
)


print(
    "FP32 session providers:"
)


for provider in fp32_session_providers:

    print(
        f"  - {provider}"
    )


if (
    "CUDAExecutionProvider"
    not in fp32_session_providers
):

    raise RuntimeError(
        "FP32 ONNX reference failed to initialize CUDA."
    )


fp32_input_info = (
    fp32_session.get_inputs()[0]
)


fp32_input_name = (
    fp32_input_info.name
)


fp32_input_type = (
    fp32_input_info.type
)


if "float16" in fp32_input_type:

    FP32_INPUT_DTYPE = np.float16

else:

    FP32_INPUT_DTYPE = np.float32


print(
    "\nFP32 ONNX input:"
)

print(
    f"  Name : {fp32_input_name}"
)

print(
    f"  Shape: {fp32_input_info.shape}"
)

print(
    f"  Type : {fp32_input_type}"
)


# =============================================================================
# GENERATE FP32 PREDICTIONS FOR DIRECT COMPARISON
# =============================================================================

print("\n" + "=" * 90)
print("FP32 ↔ FP16 PREDICTION COMPARISON")
print("=" * 90)


print(
    "Generating FP32 ONNX CUDA predictions..."
)


predictions_fp32 = []


for image_path in test_images:

    input_tensor, _ = (
        preprocess_image(

            image_path,

            target_dtype=
                FP32_INPUT_DTYPE

        )
    )


    outputs = (
        fp32_session.run(

            None,

            {
                fp32_input_name:
                    input_tensor
            }

        )
    )


    output = outputs[0]


    if output.ndim == 3:

        output = output[0]


    detections = []


    for detection in output:

        if len(detection) < 6:

            continue


        x1, y1, x2, y2, conf, cls_id = (
            detection
        )


        conf = float(conf)


        if conf <= 0:

            continue


        cls_id = int(
            cls_id
        )


        if cls_id not in class_names:

            continue


        center_x = (

            float(x1)
            + float(x2)

        ) / 2.0


        detections.append(

            (
                center_x,
                cls_id,
                conf
            )

        )


    detections.sort(
        key=lambda x: x[0]
    )


    prediction = ""


    for _, cls_id, _ in detections:

        prediction += str(

            class_names[
                cls_id
            ]

        )


    predictions_fp32.append(
        prediction
    )


print(
    "✓ FP32 reference predictions generated."
)


# =============================================================================
# PREDICTION MISMATCH ANALYSIS
# =============================================================================

mismatches = []


for (
    image_path,
    gt,
    fp32_pred,
    fp16_pred
) in zip(

    test_images,

    ground_truths,

    predictions_fp32,

    predictions_fp16

):

    if fp32_pred != fp16_pred:

        mismatches.append(

            {
                "image":
                    image_path.name,

                "ground_truth":
                    gt,

                "fp32":
                    fp32_pred,

                "fp16":
                    fp16_pred,
            }

        )


mismatch_rate_percent = (

    len(mismatches)
    / len(test_images)

) * 100.0


print(
    f"\nSamples evaluated: "
    f"{len(test_images)}"
)


print(
    f"FP32/FP16 prediction mismatches: "
    f"{len(mismatches)}"
)


print(
    f"Prediction mismatch rate: "
    f"{mismatch_rate_percent:.4f}%"
)


if mismatches:

    print(
        "\nFirst 10 mismatches:"
    )


    for item in mismatches[:10]:

        print(
            f"\nImage: "
            f"{item['image']}"
        )

        print(
            f"GT      : "
            f"{item['ground_truth']}"
        )

        print(
            f"FP32    : "
            f"{item['fp32']}"
        )

        print(
            f"FP16    : "
            f"{item['fp16']}"
        )


# =============================================================================
# STEP 2 FP32 REFERENCE VALUES
# =============================================================================
#
# These are the actual values obtained from Step 2.
# They are NOT recalculated here.
#
# This preserves the original Step 2 benchmark as the official reference.
# =============================================================================

FP32_REFERENCE = {

    "character_accuracy":
        0.866790,

    "cer":
        0.071123,

    "exact_match":
        0.844488,

    "average_latency_ms":
        10.423,

    "median_latency_ms":
        10.286,

    "p95_latency_ms":
        11.696,

    "min_latency_ms":
        9.144,

    "max_latency_ms":
        16.518,

    "fps":
        95.944,

    "mean_preprocessing_ms":
        8.385,

    "model_size_mb":
        10.14,

}


# =============================================================================
# PERFORMANCE COMPARISON
# =============================================================================

latency_speedup = (

    FP32_REFERENCE[
        "average_latency_ms"
    ]

    / average_latency_ms

)


latency_reduction = (

    (

        FP32_REFERENCE[
            "average_latency_ms"
        ]

        - average_latency_ms

    )

    / FP32_REFERENCE[
        "average_latency_ms"
    ]

) * 100.0


fps_improvement = (

    (

        fps

        - FP32_REFERENCE[
            "fps"
        ]

    )

    / FP32_REFERENCE[
        "fps"
    ]

) * 100.0


# =============================================================================
# ACCURACY COMPARISON
# =============================================================================

character_accuracy_change = (

    fp16_metrics[
        "character_accuracy"
    ]

    - FP32_REFERENCE[
        "character_accuracy"
    ]

) * 100.0


cer_change = (

    fp16_metrics[
        "cer"
    ]

    - FP32_REFERENCE[
        "cer"
    ]

)


exact_match_change = (

    fp16_metrics[
        "exact_match"
    ]

    - FP32_REFERENCE[
        "exact_match"
    ]

) * 100.0


# =============================================================================
# PERFORMANCE COMPARISON TABLE
# =============================================================================

print("\n" + "=" * 90)
print("PERFORMANCE COMPARISON")
print("=" * 90)


print(
    f"{'Metric':<25}"
    f"{'ONNX FP32 CUDA':>20}"
    f"{'ONNX FP16 CUDA':>20}"
)


print(
    "-" * 70
)


print(
    f"{'Average latency':<25}"
    f"{FP32_REFERENCE['average_latency_ms']:>17.3f} ms"
    f"{average_latency_ms:>17.3f} ms"
)


print(
    f"{'Median latency':<25}"
    f"{FP32_REFERENCE['median_latency_ms']:>17.3f} ms"
    f"{median_latency_ms:>17.3f} ms"
)


print(
    f"{'P95 latency':<25}"
    f"{FP32_REFERENCE['p95_latency_ms']:>17.3f} ms"
    f"{p95_latency_ms:>17.3f} ms"
)


print(
    f"{'Minimum latency':<25}"
    f"{FP32_REFERENCE['min_latency_ms']:>17.3f} ms"
    f"{min_latency_ms:>17.3f} ms"
)


print(
    f"{'Maximum latency':<25}"
    f"{FP32_REFERENCE['max_latency_ms']:>17.3f} ms"
    f"{max_latency_ms:>17.3f} ms"
)


print(
    f"{'FPS':<25}"
    f"{FP32_REFERENCE['fps']:>20.3f}"
    f"{fps:>20.3f}"
)


print(
    f"{'Model size':<25}"
    f"{FP32_REFERENCE['model_size_mb']:>17.2f} MB"
    f"{fp16_size_mb:>17.2f} MB"
)


print("\nSpeedup:")

print(
    f"    {latency_speedup:.3f}x"
)


print("\nLatency reduction:")

print(
    f"    {latency_reduction:+.2f}%"
)


print("\nFPS improvement:")

print(
    f"    {fps_improvement:+.2f}%"
)


# =============================================================================
# OCR ACCURACY COMPARISON
# =============================================================================

print("\n" + "=" * 90)
print("OCR ACCURACY COMPARISON")
print("=" * 90)


print("\nCharacter Accuracy:")


print(
    f"    FP32 : "
    f"{FP32_REFERENCE['character_accuracy'] * 100:.4f}%"
)


print(
    f"    FP16 : "
    f"{fp16_metrics['character_accuracy'] * 100:.4f}%"
)


print("\nCER:")


print(
    f"    FP32 : "
    f"{FP32_REFERENCE['cer']:.6f}"
)


print(
    f"    FP16 : "
    f"{fp16_metrics['cer']:.6f}"
)


print("\nExact Match:")


print(
    f"    FP32 : "
    f"{FP32_REFERENCE['exact_match'] * 100:.4f}%"
)


print(
    f"    FP16 : "
    f"{fp16_metrics['exact_match'] * 100:.4f}%"
)


# =============================================================================
# ACCURACY DIFFERENCES
# =============================================================================

print(
    "\nChanges FP32 → FP16:"
)


print(
    f"  Character Accuracy : "
    f"{character_accuracy_change:+.4f} "
    f"percentage points"
)


print(
    f"  CER                : "
    f"{cer_change:+.6f}"
)


print(
    f"  Exact Match        : "
    f"{exact_match_change:+.4f} "
    f"percentage points"
)


# =============================================================================
# FINAL SUMMARY
# =============================================================================

print("\n" + "=" * 90)
print("STEP 3-OCR — COMPLETED")
print("=" * 90)


print("\nTEST SET")
print("--------")


print(
    f"Images                 : "
    f"{len(test_images)}"
)


print(
    f"Input                  : "
    f"{IMG_SIZE}x{IMG_SIZE}"
)


print(
    f"Parameters             : "
    f"{total_params:,}"
)


print(
    f"GPU                    : "
    f"{gpu_name}"
)


print("\nONNX FP32 CUDA")
print("--------------")


print(
    f"Character Accuracy     : "
    f"{FP32_REFERENCE['character_accuracy'] * 100:.4f}%"
)


print(
    f"CER                    : "
    f"{FP32_REFERENCE['cer']:.6f}"
)


print(
    f"Exact Match            : "
    f"{FP32_REFERENCE['exact_match'] * 100:.4f}%"
)


print(
    f"Average latency        : "
    f"{FP32_REFERENCE['average_latency_ms']:.3f} ms"
)


print(
    f"Median latency         : "
    f"{FP32_REFERENCE['median_latency_ms']:.3f} ms"
)


print(
    f"P95 latency            : "
    f"{FP32_REFERENCE['p95_latency_ms']:.3f} ms"
)


print(
    f"Min latency            : "
    f"{FP32_REFERENCE['min_latency_ms']:.3f} ms"
)


print(
    f"Max latency            : "
    f"{FP32_REFERENCE['max_latency_ms']:.3f} ms"
)


print(
    f"FPS                    : "
    f"{FP32_REFERENCE['fps']:.3f}"
)


print(
    f"Model size             : "
    f"{FP32_REFERENCE['model_size_mb']:.2f} MB"
)


print("\nONNX FP16 CUDA")
print("--------------")


print(
    f"Character Accuracy     : "
    f"{fp16_metrics['character_accuracy'] * 100:.4f}%"
)


print(
    f"CER                    : "
    f"{fp16_metrics['cer']:.6f}"
)


print(
    f"Exact Match            : "
    f"{fp16_metrics['exact_match'] * 100:.4f}%"
)


print(
    f"Average latency        : "
    f"{average_latency_ms:.3f} ms"
)


print(
    f"Median latency         : "
    f"{median_latency_ms:.3f} ms"
)


print(
    f"P95 latency            : "
    f"{p95_latency_ms:.3f} ms"
)


print(
    f"Min latency            : "
    f"{min_latency_ms:.3f} ms"
)


print(
    f"Max latency            : "
    f"{max_latency_ms:.3f} ms"
)


print(
    f"FPS                    : "
    f"{fps:.3f}"
)


print(
    f"Mean preprocessing     : "
    f"{mean_preprocessing_ms:.3f} ms"
)


print(
    f"Model size             : "
    f"{fp16_size_mb:.2f} MB"
)


print("\nFP32 → FP16")
print("-----------")


print(
    f"Latency speedup        : "
    f"{latency_speedup:.3f}x"
)


print(
    f"Latency reduction      : "
    f"{latency_reduction:+.2f}%"
)


print(
    f"FPS improvement        : "
    f"{fps_improvement:+.2f}%"
)


print(
    f"Model size reduction   : "
    f"{size_reduction:.2f}%"
)


print(
    f"Prediction mismatches  : "
    f"{len(mismatches)} / "
    f"{len(test_images)}"
)


print(
    f"Mismatch rate          : "
    f"{mismatch_rate_percent:.4f}%"
)


# =============================================================================
# SAVE JSON SUMMARY
# =============================================================================

summary = {

    "step":
        "Step 3 - ONNX FP16 CUDA",

    "environment": {

        "python":
            sys.version.split()[0],

        "pytorch":
            torch.__version__,

        "onnx":
            onnx.__version__,

        "onnxruntime":
            ort.__version__,

        "ultralytics":
            "8.4.137",

        "gpu":
            gpu_name,

        "gpu_memory_gb":
            gpu_memory,

        "pytorch_cuda":
            torch.version.cuda,

        "execution_provider":
            "CUDAExecutionProvider",

    },

    "dataset": {

        "test_images":
            len(test_images),

        "image_size":
            f"{IMG_SIZE}x{IMG_SIZE}",

    },

    "model": {

        "parameters":
            int(total_params),

        "pytorch_model":
            PT_MODEL,

        "fp32_model":
            FP32_ONNX,

        "fp16_model":
            FP16_ONNX,

        "fp32_size_mb":
            fp32_size_mb,

        "fp16_size_mb":
            fp16_size_mb,

        "size_reduction_percent":
            size_reduction,

    },

    "fp32_onnx_cuda":
        FP32_REFERENCE,

    "fp16_onnx_cuda": {

        "character_accuracy":
            fp16_metrics[
                "character_accuracy"
            ],

        "cer":
            fp16_metrics[
                "cer"
            ],

        "exact_match":
            fp16_metrics[
                "exact_match"
            ],

        "normalized_similarity":
            fp16_metrics[
                "normalized_similarity"
            ],

        "total_inference_time_s":
            total_inference_time_s,

        "average_latency_ms":
            average_latency_ms,

        "median_latency_ms":
            median_latency_ms,

        "p95_latency_ms":
            p95_latency_ms,

        "min_latency_ms":
            min_latency_ms,

        "max_latency_ms":
            max_latency_ms,

        "fps":
            fps,

        "mean_preprocessing_ms":
            mean_preprocessing_ms,

    },

    "comparison": {

        "latency_speedup":
            latency_speedup,

        "latency_reduction_percent":
            latency_reduction,

        "fps_improvement_percent":
            fps_improvement,

        "character_accuracy_change_percentage_points":
            character_accuracy_change,

        "cer_change":
            cer_change,

        "exact_match_change_percentage_points":
            exact_match_change,

        "prediction_mismatches":
            len(mismatches),

        "prediction_mismatch_rate_percent":
            mismatch_rate_percent,

    },

    "mismatches":
        mismatches,

}


# =============================================================================
# WRITE JSON
# =============================================================================

with open(

    SUMMARY_JSON,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        summary,

        f,

        indent=4,

        ensure_ascii=False

    )


# =============================================================================
# FINAL STATUS
# =============================================================================

print("\n" + "=" * 90)


print(
    "✓ JSON summary saved:"
)


print(
    f"  {SUMMARY_JSON}"
)


print("=" * 90)


print(
    "\n✓ Original best.pt remains untouched."
)

print(
    "✓ Test dataset remains untouched."
)

print(
    "✓ Step 2 ONNX FP32 remains untouched."
)

print(
    "✓ FP16 ONNX generated separately."
)

print(
    "✓ FP16 ONNX passed ONNX checker."
)

print(
    "✓ CUDAExecutionProvider explicitly verified."
)

print(
    "✓ Warm-up runs excluded from benchmark."
)

print(
    "✓ Latency statistics calculated."
)

print(
    "✓ FPS calculated."
)

print(
    "✓ OCR accuracy calculated."
)

print(
    "✓ FP32 ↔ FP16 prediction comparison completed."
)

print(
    "✓ JSON summary generated."
)

print(
    "✓ Step 3 is complete."
)

STEP 3-OCR — ONNX FP16 CUDA OPTIMIZATION
Python       : 3.13.15
PyTorch      : 2.11.0+cu128
Ultralytics  : 8.4.137
ONNX         : 1.22.0
ONNX Runtime : 1.26.0

VERIFYING CUDA
✓ CUDA available
✓ GPU: Tesla T4
✓ GPU memory: 14.56 GB
✓ PyTorch CUDA: 12.8

VERIFYING ONNX RUNTIME CUDA
Available providers:
  - TensorrtExecutionProvider
  - CUDAExecutionProvider
  - CPUExecutionProvider

✓ CUDAExecutionProvider is available.

VERIFYING STEP 1C / STEP 2 REGISTRY
✓ PyTorch model:
  /content/drive/MyDrive/Stage_PFA/experiments/YOLOV11_OCR/weights/best.pt
✓ ONNX FP32 model:
  /content/drive/MyDrive/Stage_PFA/OCR_OPTIMIZATION/02_onnx_fp32/yolo11_ocr_fp32.onnx
✓ Data YAML:
  /content/drive/MyDrive/Stage_PFA/datasets/data.yaml
✓ Test images:
  /content/drive/MyDrive/Stage_PFA/datasets/test/images
✓ Test labels:
  /content/drive/MyDrive/Stage_PFA/datasets/test/labels

VALIDATING STEP 2 FP32 ONNX
✓ Step 2 FP32 ONNX checker passed.

FP32 ONNX input:
  Name : images
  Shape: [1, 3, 608, 608]
  Type : 1


In [1]:
# =============================================================================
# STEP 4 — YOLO11 OCR ONNX INT8 STATIC QUANTIZATION
# =============================================================================
#
# PURPOSE
# -------
# Convert the existing YOLO11 OCR FP32 ONNX model to a conservative,
# accuracy-preserving INT8 QDQ ONNX model.
#
# PIPELINE
# --------
#   FP32 ONNX
#       |
#       v
#   ONNX quantization preprocessing
#       |
#       v
#   Representative calibration dataset
#       |
#       v
#   Conv-only INT8 QDQ quantization
#       |
#       v
#   Protect output-side sensitive graph nodes
#       |
#       v
#   ONNX validation
#       |
#       v
#   Single-image sanity check
#       |
#       v
#   FP32 vs INT8 raw-output comparison
#       |
#       v
#   FP32 vs INT8 decoded OCR comparison
#       |
#       v
#   ONLY IF SANITY CHECKS PASS:
#       full 508-image evaluation
#
# IMPORTANT
# ---------
# - Original .pt is untouched
# - FP32 ONNX is untouched
# - FP16 ONNX is untouched
# - Test dataset is untouched
# - Previous INT8 model is replaced
# - Full validation is NOT performed if INT8 is obviously broken
#
# TARGET
# ------
# YOLO11 OCR
# Input  : [1, 3, 608, 608]
# Output : [1, 300, 6]
#
# =============================================================================


import os
import sys
import json
import time
import random
import shutil
import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import torch
import onnx
import onnxruntime as ort

warnings.filterwarnings("ignore")


# =============================================================================
# SECTION HELPERS
# =============================================================================

def section(title):
    print("\n")
    print("=" * 100)
    print(title)
    print("=" * 100)


def subsection(title):
    print("\n")
    print("-" * 100)
    print(title)
    print("-" * 100)


def file_size_mb(path):
    return os.path.getsize(path) / (1024 ** 2)


def safe_float(value):
    try:
        return float(value)
    except Exception:
        return np.nan


# =============================================================================
# STEP 4A — CONFIGURATION
# =============================================================================

section("STEP 4A — YOLO11 OCR INT8 CONFIGURATION")


# -----------------------------------------------------------------------------
# IMPORTANT: YOUR CURRENT PATHS
# -----------------------------------------------------------------------------

STAGE_ROOT = Path(
    "/content/drive/MyDrive/Stage_PFA"
)

FP32_PATH = STAGE_ROOT / (
    "OCR_OPTIMIZATION/02_onnx_fp32/"
    "yolo11_ocr_fp32.onnx"
)

FP16_PATH = STAGE_ROOT / (
    "OCR_OPTIMIZATION/03_onnx_fp16/"
    "yolo11_ocr_fp16.onnx"
)

INT8_DIR = STAGE_ROOT / (
    "OCR_OPTIMIZATION/04_onnx_int8"
)

CALIB_DIR = STAGE_ROOT / (
    "OCR_OPTIMIZATION/05_calibration"
)

VALIDATION_DIR = STAGE_ROOT / (
    "OCR_OPTIMIZATION/06_validation"
)

LOG_DIR = STAGE_ROOT / (
    "OCR_OPTIMIZATION/09_logs"
)

DATASET_ROOT = STAGE_ROOT / "datasets"

TEST_IMAGES_DIR = DATASET_ROOT / "test" / "images"
TEST_LABELS_DIR = DATASET_ROOT / "test" / "labels"

DATA_YAML = DATASET_ROOT / "data.yaml"


# -----------------------------------------------------------------------------
# MODEL SETTINGS
# -----------------------------------------------------------------------------

IMG_SIZE = 608

N_CALIBRATION_IMAGES = 200

RANDOM_SEED = 42

# Number of graph hops protected around output tensors.
#
# We start conservatively. The script will report exactly how many
# nodes are protected.
#
TAIL_HOPS = 8


# -----------------------------------------------------------------------------
# VALIDATION SETTINGS
# -----------------------------------------------------------------------------

CONF_THRESHOLD = 0.001

IOU_THRESHOLD = 0.7

WARMUP_RUNS = 5

SANITY_IMAGES = 5


# -----------------------------------------------------------------------------
# CREATE DIRECTORIES
# -----------------------------------------------------------------------------

for directory in [
    INT8_DIR,
    CALIB_DIR,
    VALIDATION_DIR,
    LOG_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


print(f"""
FP32 ONNX:
    {FP32_PATH}

FP16 ONNX:
    {FP16_PATH}

INT8 output:
    {INT8_DIR}

Calibration:
    {CALIB_DIR}

Validation:
    {VALIDATION_DIR}

Test images:
    {TEST_IMAGES_DIR}

Test labels:
    {TEST_LABELS_DIR}

Input resolution:
    {IMG_SIZE} × {IMG_SIZE}

Calibration images:
    {N_CALIBRATION_IMAGES}

Random seed:
    {RANDOM_SEED}

Quantization:
    ONNX Runtime static QDQ

Weights:
    INT8

Activations:
    UINT8

Quantization scope:
    Conv only

Tail protection:
    {TAIL_HOPS} graph hops
""")


# =============================================================================
# STEP 4B — ENVIRONMENT VERIFICATION
# =============================================================================

section("STEP 4B — ENVIRONMENT VERIFICATION")


print(f"Python       : {sys.version.split()[0]}")
print(f"PyTorch      : {torch.__version__}")
print(f"ONNX         : {onnx.__version__}")
print(f"ONNX Runtime : {ort.__version__}")


# -----------------------------------------------------------------------------
# CUDA
# -----------------------------------------------------------------------------

print("\nCUDA verification")

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available. "
        "Please reconnect the Colab runtime using GPU."
    )

print("✓ CUDA available")
print(f"✓ GPU: {torch.cuda.get_device_name(0)}")

gpu_memory = (
    torch.cuda.get_device_properties(0).total_memory
    / (1024 ** 3)
)

print(f"✓ GPU memory: {gpu_memory:.2f} GB")
print(f"✓ CUDA version: {torch.version.cuda}")


# -----------------------------------------------------------------------------
# ONNX RUNTIME PROVIDERS
# -----------------------------------------------------------------------------

print("\nONNX Runtime providers:")

providers = ort.get_available_providers()

for provider in providers:
    print(f"  - {provider}")

if "CUDAExecutionProvider" not in providers:
    raise RuntimeError(
        "CUDAExecutionProvider is unavailable."
    )

print("\n✓ CUDAExecutionProvider available")


# -----------------------------------------------------------------------------
# QUANTIZATION IMPORTS
# -----------------------------------------------------------------------------

try:

    from onnxruntime.quantization import (
        quantize_static,
        CalibrationDataReader,
        QuantType,
        QuantFormat,
        CalibrationMethod,
    )

    from onnxruntime.quantization.shape_inference import (
        quant_pre_process
    )

except Exception as e:

    raise RuntimeError(
        f"Failed to import ONNX Runtime quantization tools:\n{e}"
    )

print("✓ Static quantization API available")
print("✓ Quantization preprocessing API available")


# =============================================================================
# STEP 4C — VERIFY FILES
# =============================================================================

section("STEP 4C — VERIFYING EXISTING ARTIFACTS")


required_files = {
    "FP32 ONNX": FP32_PATH,
    "FP16 ONNX": FP16_PATH,
    "Data YAML": DATA_YAML,
}


for name, path in required_files.items():

    if not path.exists():

        raise FileNotFoundError(
            f"{name} not found:\n{path}"
        )

    print(f"✓ {name}")
    print(f"  {path}")


if not TEST_IMAGES_DIR.exists():

    raise FileNotFoundError(
        f"Test image directory not found:\n"
        f"{TEST_IMAGES_DIR}"
    )


print("\n✓ Required files verified")


# =============================================================================
# STEP 4D — INSPECT FP32 ONNX
# =============================================================================

section("STEP 4D — INSPECTING FP32 ONNX")


fp32_model = onnx.load(str(FP32_PATH))

onnx.checker.check_model(fp32_model)

print("✓ FP32 ONNX checker passed")


print("\nFP32 inputs:")

for inp in fp32_model.graph.input:

    shape = []

    for dim in inp.type.tensor_type.shape.dim:

        if dim.dim_value:
            shape.append(dim.dim_value)

        elif dim.dim_param:
            shape.append(dim.dim_param)

        else:
            shape.append("?")

    print(
        f"  {inp.name}: "
        f"{shape}"
    )


print("\nFP32 outputs:")

for out in fp32_model.graph.output:

    shape = []

    for dim in out.type.tensor_type.shape.dim:

        if dim.dim_value:
            shape.append(dim.dim_value)

        elif dim.dim_param:
            shape.append(dim.dim_param)

        else:
            shape.append("?")

    print(
        f"  {out.name}: "
        f"{shape}"
    )


# =============================================================================
# STEP 4E — COLLECT TEST IMAGES
# =============================================================================

section("STEP 4E — PREPARING OCR DATASET")


image_extensions = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
}


all_images = sorted(
    [
        p
        for p in TEST_IMAGES_DIR.iterdir()
        if p.is_file()
        and p.suffix.lower() in image_extensions
    ]
)


print(f"Total test images: {len(all_images)}")


if len(all_images) == 0:

    raise RuntimeError(
        "No test images found."
    )


if len(all_images) < N_CALIBRATION_IMAGES:

    raise RuntimeError(
        f"Only {len(all_images)} test images are available, "
        f"but {N_CALIBRATION_IMAGES} calibration images were requested."
    )


# =============================================================================
# STEP 4F — REPRODUCIBLE CALIBRATION SAMPLING
# =============================================================================

section("STEP 4F — BUILDING REPRESENTATIVE CALIBRATION SET")


random.seed(RANDOM_SEED)

calibration_images = random.sample(
    all_images,
    N_CALIBRATION_IMAGES
)


calibration_list = CALIB_DIR / (
    "yolo11_ocr_calibration_images.json"
)


with open(calibration_list, "w") as f:

    json.dump(
        [str(p) for p in calibration_images],
        f,
        indent=2,
    )


print(
    f"✓ Selected {len(calibration_images)} calibration images"
)

print(
    f"✓ Calibration list saved:\n"
    f"  {calibration_list}"
)


# =============================================================================
# STEP 4G — ULTRALYTICS-STYLE PREPROCESSING
# =============================================================================

section("STEP 4G — INPUT PREPROCESSING")


def letterbox(
    image,
    new_shape=(608, 608),
    color=(114, 114, 114),
):
    """
    Ultralytics-style letterbox preprocessing.

    Maintains aspect ratio and pads to target resolution.
    """

    shape = image.shape[:2]

    if isinstance(new_shape, int):

        new_shape = (
            new_shape,
            new_shape
        )

    ratio = min(
        new_shape[0] / shape[0],
        new_shape[1] / shape[1],
    )

    new_unpad = (
        int(round(shape[1] * ratio)),
        int(round(shape[0] * ratio)),
    )

    dw = new_shape[1] - new_unpad[0]
    dh = new_shape[0] - new_unpad[1]

    dw /= 2
    dh /= 2

    if shape[::-1] != new_unpad:

        image = cv2.resize(
            image,
            new_unpad,
            interpolation=cv2.INTER_LINEAR,
        )

    top = int(round(dh - 0.1))
    bottom = int(round(dh + 0.1))

    left = int(round(dw - 0.1))
    right = int(round(dw + 0.1))

    image = cv2.copyMakeBorder(
        image,
        top,
        bottom,
        left,
        right,
        cv2.BORDER_CONSTANT,
        value=color,
    )

    return image


def preprocess_image(
    image_path,
    img_size=IMG_SIZE,
):
    """
    Convert image to the exact FP32 tensor convention:

        BGR
        ↓
        RGB
        ↓
        Letterbox
        ↓
        float32
        ↓
        /255
        ↓
        HWC → CHW
        ↓
        batch dimension
    """

    image = cv2.imread(
        str(image_path)
    )

    if image is None:

        raise RuntimeError(
            f"Could not read image:\n{image_path}"
        )

    image = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2RGB,
    )

    image = letterbox(
        image,
        (img_size, img_size),
    )

    image = image.astype(
        np.float32
    ) / 255.0

    image = image.transpose(
        2,
        0,
        1,
    )

    image = np.expand_dims(
        image,
        axis=0,
    )

    return np.ascontiguousarray(
        image,
        dtype=np.float32,
    )


test_tensor = preprocess_image(
    calibration_images[0]
)

print(
    f"✓ Example tensor shape: "
    f"{test_tensor.shape}"
)

print(
    f"✓ Tensor dtype: "
    f"{test_tensor.dtype}"
)

print(
    f"✓ Tensor range: "
    f"{test_tensor.min():.4f} → "
    f"{test_tensor.max():.4f}"
)


# =============================================================================
# STEP 4H — CALIBRATION DATA READER
# =============================================================================

class OCRCalibrationDataReader(
    CalibrationDataReader
):

    def __init__(
        self,
        image_paths,
        input_name,
    ):

        self.image_paths = image_paths
        self.input_name = input_name
        self.index = 0

    def get_next(self):

        if self.index >= len(
            self.image_paths
        ):

            return None

        image_path = self.image_paths[
            self.index
        ]

        self.index += 1

        tensor = preprocess_image(
            image_path,
            IMG_SIZE,
        )

        return {
            self.input_name: tensor
        }

    def rewind(self):

        self.index = 0


# =============================================================================
# STEP 4I — QUANTIZATION PREPROCESSING
# =============================================================================

section("STEP 4I — ONNX QUANTIZATION PREPROCESSING")


PREPROCESSED_FP32_PATH = (
    INT8_DIR /
    "yolo11_ocr_fp32_preprocessed.onnx"
)


if PREPROCESSED_FP32_PATH.exists():

    PREPROCESSED_FP32_PATH.unlink()


print(
    "Running ONNX Runtime quantization preprocessing..."
)

preprocess_start = time.time()


quant_pre_process(
    str(FP32_PATH),
    str(PREPROCESSED_FP32_PATH),
    skip_symbolic_shape=False,
)


preprocess_time = (
    time.time() -
    preprocess_start
)


print(
    f"✓ Preprocessing completed "
    f"in {preprocess_time:.2f} s"
)

print(
    f"✓ Preprocessed model:\n"
    f"  {PREPROCESSED_FP32_PATH}"
)


# =============================================================================
# STEP 4J — INSPECT PREPROCESSED GRAPH
# =============================================================================

section("STEP 4J — IDENTIFYING MODEL INPUT / OUTPUT")


preprocessed_model = onnx.load(
    str(PREPROCESSED_FP32_PATH)
)

onnx.checker.check_model(
    preprocessed_model
)


input_name = (
    preprocessed_model.graph
    .input[0]
    .name
)

output_names = [
    output.name
    for output
    in preprocessed_model.graph.output
]


print(f"Input name: {input_name}")

print(
    f"Output names: {output_names}"
)


# =============================================================================
# STEP 4K — DISCOVER OUTPUT-SIDE SENSITIVE NODES
# =============================================================================

section(
    "STEP 4K — PROTECTING OUTPUT-SIDE MODEL NODES"
)


def get_output_tail_nodes(
    model_path,
    hops=8,
):

    model = onnx.load(
        str(model_path)
    )

    graph = model.graph

    # ---------------------------------------------------------
    # Map tensor -> producing node
    # ---------------------------------------------------------

    producer_of = {}

    for node in graph.node:

        for output in node.output:

            producer_of[output] = node

    # ---------------------------------------------------------
    # Generate stable node identifier
    # ---------------------------------------------------------

    def node_id(node):

        if node.name:

            return node.name

        if node.output:

            return (
                f"{node.op_type}__"
                f"{node.output[0]}"
            )

        return (
            f"{node.op_type}__anonymous"
        )

    # ---------------------------------------------------------
    # Walk backwards from graph outputs
    # ---------------------------------------------------------

    frontier = set(
        output_names
    )

    excluded = set()

    visited_tensors = set()

    for _ in range(hops):

        next_frontier = set()

        for tensor_name in frontier:

            if tensor_name in visited_tensors:

                continue

            visited_tensors.add(
                tensor_name
            )

            node = producer_of.get(
                tensor_name
            )

            if node is None:

                continue

            excluded.add(
                node_id(node)
            )

            for input_tensor in node.input:

                if input_tensor:

                    next_frontier.add(
                        input_tensor
                    )

        frontier = next_frontier

    return sorted(excluded)


nodes_to_exclude = get_output_tail_nodes(
    PREPROCESSED_FP32_PATH,
    hops=TAIL_HOPS,
)


print(
    f"Output-tail nodes protected: "
    f"{len(nodes_to_exclude)}"
)


if len(nodes_to_exclude) == 0:

    raise RuntimeError(
        "No output-tail nodes were identified. "
        "Refusing to continue because the graph protection "
        "mechanism may not be working."
    )


# =============================================================================
# STEP 4L — CLEAN PREVIOUS INT8 ARTIFACT
# =============================================================================

section("STEP 4L — CLEANING PREVIOUS INT8 OUTPUT")


INT8_PATH = (
    INT8_DIR /
    "yolo11_ocr_int8.onnx"
)


if INT8_PATH.exists():

    print(
        "Removing previous INT8 model..."
    )

    INT8_PATH.unlink()


print(
    "✓ INT8 output location is clean"
)


# =============================================================================
# STEP 4M — STATIC INT8 QUANTIZATION
# =============================================================================

section(
    "STEP 4M — STATIC INT8 QDQ QUANTIZATION"
)


print("""
QUANTIZATION STRATEGY
---------------------

Format:
    QDQ

Weight:
    INT8

Activation:
    UINT8

Calibration:
    MinMax

Per-channel:
    Enabled

Quantized operations:
    Conv only

Protected:
    Output-side graph tail

Reason:
    Preserve sensitive OCR prediction/output operations
    while still quantizing the computationally expensive
    convolution layers.
""")


calibration_reader = OCRCalibrationDataReader(
    calibration_images,
    input_name,
)


quant_start = time.time()


quantize_static(

    model_input=str(
        PREPROCESSED_FP32_PATH
    ),

    model_output=str(
        INT8_PATH
    ),

    calibration_data_reader=(
        calibration_reader
    ),

    quant_format=(
        QuantFormat.QDQ
    ),

    activation_type=(
        QuantType.QUInt8
    ),

    weight_type=(
        QuantType.QInt8
    ),

    calibrate_method=(
        CalibrationMethod.MinMax
    ),

    per_channel=True,

    op_types_to_quantize=[
        "Conv"
    ],

    nodes_to_exclude=(
        nodes_to_exclude
    ),
)


quant_time = (
    time.time() -
    quant_start
)


print(
    f"✓ INT8 quantization completed "
    f"in {quant_time:.2f} s"
)

print(
    f"✓ INT8 model:\n"
    f"  {INT8_PATH}"
)


# =============================================================================
# STEP 4N — STRUCTURAL VALIDATION
# =============================================================================

section(
    "STEP 4N — VALIDATING INT8 ONNX STRUCTURE"
)


int8_model = onnx.load(
    str(INT8_PATH)
)

onnx.checker.check_model(
    int8_model
)

print(
    "✓ ONNX checker passed"
)


# Count Q/DQ nodes

quant_nodes = [
    node
    for node in int8_model.graph.node
    if node.op_type in (
        "QuantizeLinear",
        "DequantizeLinear",
    )
]


print(
    f"Quantization nodes: "
    f"{len(quant_nodes)}"
)


if len(quant_nodes) == 0:

    raise RuntimeError(
        "No QuantizeLinear/DequantizeLinear nodes found. "
        "INT8 quantization did not occur."
    )


print(
    "✓ QDQ quantization confirmed"
)


# =============================================================================
# STEP 4O — MODEL SIZE
# =============================================================================

section("STEP 4O — MODEL SIZE COMPARISON")


fp32_size = file_size_mb(
    FP32_PATH
)

fp16_size = file_size_mb(
    FP16_PATH
)

int8_size = file_size_mb(
    INT8_PATH
)


print(
    f"FP32 : {fp32_size:.2f} MB"
)

print(
    f"FP16 : {fp16_size:.2f} MB"
)

print(
    f"INT8 : {int8_size:.2f} MB"
)


fp32_reduction = (
    (fp32_size - int8_size)
    / fp32_size
    * 100
)


fp16_reduction = (
    (fp16_size - int8_size)
    / fp16_size
    * 100
)


print(
    f"\nFP32 → INT8 reduction : "
    f"{fp32_reduction:.2f}%"
)

print(
    f"FP16 → INT8 reduction : "
    f"{fp16_reduction:.2f}%"
)


# =============================================================================
# STEP 4P — CREATE FP32 / INT8 CUDA SESSIONS
# =============================================================================

section(
    "STEP 4P — INITIALIZING FP32 / INT8 CUDA SESSIONS"
)


provider_options = {
    "device_id": 0
}


FP32_SESSION = ort.InferenceSession(

    str(FP32_PATH),

    providers=[
        (
            "CUDAExecutionProvider",
            provider_options
        ),
        "CPUExecutionProvider",
    ],
)


INT8_SESSION = ort.InferenceSession(

    str(INT8_PATH),

    providers=[
        (
            "CUDAExecutionProvider",
            provider_options
        ),
        "CPUExecutionProvider",
    ],
)


print(
    "FP32 providers:"
)

for p in FP32_SESSION.get_providers():

    print(
        f"  - {p}"
    )


print(
    "\nINT8 providers:"
)

for p in INT8_SESSION.get_providers():

    print(
        f"  - {p}"
    )


if (
    "CUDAExecutionProvider"
    not in INT8_SESSION.get_providers()
):

    raise RuntimeError(
        "INT8 session did not initialize CUDA."
    )


print(
    "\n✓ CUDAExecutionProvider active"
)


# =============================================================================
# STEP 4Q — VERIFY MODEL INPUT
# =============================================================================

section(
    "STEP 4Q — VERIFYING INT8 INPUT"
)


fp32_input = FP32_SESSION.get_inputs()[0]

int8_input = INT8_SESSION.get_inputs()[0]


print(
    f"FP32 input:"
)

print(
    f"  Name : {fp32_input.name}"
)

print(
    f"  Shape: {fp32_input.shape}"
)

print(
    f"  Type : {fp32_input.type}"
)


print(
    f"\nINT8 input:"
)

print(
    f"  Name : {int8_input.name}"
)

print(
    f"  Shape: {int8_input.shape}"
)

print(
    f"  Type : {int8_input.type}"
)


if int8_input.type != "tensor(float)":

    raise RuntimeError(
        "Unexpected INT8 model input type. "
        "Expected tensor(float)."
    )


print(
    "\n✓ INT8 model accepts FP32 input"
)


# =============================================================================
# STEP 4R — WARM-UP
# =============================================================================

section(
    "STEP 4R — CUDA WARM-UP"
)


warmup_tensor = preprocess_image(
    calibration_images[0]
)


for _ in range(WARMUP_RUNS):

    FP32_SESSION.run(
        None,
        {
            fp32_input.name:
            warmup_tensor
        }
    )


for _ in range(WARMUP_RUNS):

    INT8_SESSION.run(
        None,
        {
            int8_input.name:
            warmup_tensor
        }
    )


print(
    f"✓ {WARMUP_RUNS} FP32 warm-up runs"
)

print(
    f"✓ {WARMUP_RUNS} INT8 warm-up runs"
)


# =============================================================================
# STEP 4S — RAW OUTPUT SANITY CHECK
# =============================================================================

section(
    "STEP 4S — FP32 vs INT8 RAW OUTPUT SANITY CHECK"
)


def output_statistics(outputs):

    values = []

    for output in outputs:

        arr = np.asarray(
            output
        )

        values.append({

            "shape":
                list(arr.shape),

            "dtype":
                str(arr.dtype),

            "min":
                float(arr.min()),

            "max":
                float(arr.max()),

            "mean":
                float(arr.mean()),

            "std":
                float(arr.std()),

            "abs_mean":
                float(
                    np.mean(
                        np.abs(arr)
                    )
                ),

            "max_abs":
                float(
                    np.max(
                        np.abs(arr)
                    )
                ),

        })

    return values


sanity_results = []


for i in range(
    min(
        SANITY_IMAGES,
        len(calibration_images)
    )
):

    image_path = calibration_images[i]

    tensor = preprocess_image(
        image_path
    )


    fp32_outputs = FP32_SESSION.run(
        None,
        {
            fp32_input.name:
            tensor
        }
    )


    int8_outputs = INT8_SESSION.run(
        None,
        {
            int8_input.name:
            tensor
        }
    )


    if len(fp32_outputs) != len(
        int8_outputs
    ):

        raise RuntimeError(
            "FP32 and INT8 output counts differ."
        )


    for output_index in range(
        len(fp32_outputs)
    ):

        fp32_arr = np.asarray(
            fp32_outputs[
                output_index
            ]
        )

        int8_arr = np.asarray(
            int8_outputs[
                output_index
            ]
        )


        if fp32_arr.shape != int8_arr.shape:

            raise RuntimeError(
                f"Output shape mismatch on "
                f"{image_path.name}"
            )


        mae = float(
            np.mean(
                np.abs(
                    fp32_arr -
                    int8_arr
                )
            )
        )


        max_error = float(
            np.max(
                np.abs(
                    fp32_arr -
                    int8_arr
                )
            )
        )


        fp32_norm = float(
            np.linalg.norm(
                fp32_arr.ravel()
            )
        )


        int8_norm = float(
            np.linalg.norm(
                int8_arr.ravel()
            )
        )


        relative_error = (
            np.linalg.norm(
                (
                    fp32_arr -
                    int8_arr
                ).ravel()
            )
            /
            max(
                fp32_norm,
                1e-12
            )
        )


        sanity_results.append({

            "image":
                image_path.name,

            "output_index":
                output_index,

            "mae":
                mae,

            "max_error":
                max_error,

            "relative_l2_error":
                float(
                    relative_error
                ),

            "fp32_max_abs":
                float(
                    np.max(
                        np.abs(
                            fp32_arr
                        )
                    )
                ),

            "int8_max_abs":
                float(
                    np.max(
                        np.abs(
                            int8_arr
                        )
                    )
                ),

        })


sanity_df = pd.DataFrame(
    sanity_results
)


print(
    sanity_df.to_string(
        index=False
    )
)


# -----------------------------------------------------------------------------
# IMPORTANT:
#
# We do NOT require raw INT8 outputs to be numerically identical.
#
# Quantization necessarily introduces numerical error.
#
# What matters is whether the prediction behavior is preserved.
# -----------------------------------------------------------------------------

max_relative_error = float(
    sanity_df[
        "relative_l2_error"
    ].max()
)


print(
    f"\nMaximum relative L2 output error: "
    f"{max_relative_error:.6f}"
)


# =============================================================================
# STEP 4T — DECODE YOLO OCR OUTPUT
# =============================================================================

section(
    "STEP 4T — FP32 vs INT8 OCR PREDICTION CHECK"
)


# -----------------------------------------------------------------------------
# CLASS MAP FROM YOUR CURRENT MODEL
# -----------------------------------------------------------------------------

CLASS_NAMES = {

    0: "0",
    1: "1",
    2: "2",
    3: "3",
    4: "4",
    5: "5",
    6: "6",
    7: "7",
    8: "8",
    9: "9",

    10: "a",
    11: "b",
    12: "d",
    13: "gr",
    14: "h",
    15: "t",
    16: "w",
    17: "waw",
    18: "y",
}


def decode_yolo_ocr_output(
    output,
    conf_threshold=0.001,
):
    """
    Decode YOLO detection output.

    Expected output:
        [1, 300, 6]

    Each detection:
        [x1, y1, x2, y2, confidence, class_id]

    The characters are sorted by the horizontal
    center coordinate of their bounding boxes.
    """

    arr = np.asarray(
        output
    )

    if arr.ndim == 3:

        arr = arr[0]


    if arr.ndim != 2:

        raise ValueError(
            f"Unexpected output shape: "
            f"{arr.shape}"
        )


    if arr.shape[1] < 6:

        raise ValueError(
            f"Expected at least 6 output values, "
            f"got {arr.shape[1]}"
        )


    detections = []


    for row in arr:

        x1, y1, x2, y2, conf, cls = (
            row[:6]
        )


        conf = float(conf)


        if conf < conf_threshold:

            continue


        cls = int(
            round(
                float(cls)
            )
        )


        if cls not in CLASS_NAMES:

            continue


        center_x = (
            float(x1) +
            float(x2)
        ) / 2.0


        detections.append({

            "x1":
                float(x1),

            "y1":
                float(y1),

            "x2":
                float(x2),

            "y2":
                float(y2),

            "confidence":
                conf,

            "class_id":
                cls,

            "center_x":
                center_x,

            "char":
                CLASS_NAMES[cls],

        })


    detections.sort(
        key=lambda d:
        d["center_x"]
    )


    text = "".join(
        d["char"]
        for d in detections
    )


    return text, detections


# =============================================================================
# PREDICTION COMPARISON
# =============================================================================

comparison_results = []


for image_path in calibration_images:

    tensor = preprocess_image(
        image_path
    )


    fp32_outputs = FP32_SESSION.run(
        None,
        {
            fp32_input.name:
            tensor
        }
    )


    int8_outputs = INT8_SESSION.run(
        None,
        {
            int8_input.name:
            tensor
        }
    )


    fp32_text, fp32_det = (
        decode_yolo_ocr_output(
            fp32_outputs[0],
            CONF_THRESHOLD,
        )
    )


    int8_text, int8_det = (
        decode_yolo_ocr_output(
            int8_outputs[0],
            CONF_THRESHOLD,
        )
    )


    same_text = (
        fp32_text ==
        int8_text
    )


    comparison_results.append({

        "image":
            image_path.name,

        "fp32_text":
            fp32_text,

        "int8_text":
            int8_text,

        "same_text":
            same_text,

        "fp32_detection_count":
            len(fp32_det),

        "int8_detection_count":
            len(int8_det),

    })


comparison_df = pd.DataFrame(
    comparison_results
)


same_prediction_count = int(
    comparison_df[
        "same_text"
    ].sum()
)


prediction_match_rate = (
    same_prediction_count /
    len(comparison_df)
)


print(
    f"\nCalibration prediction comparison:"
)

print(
    f"Samples: "
    f"{len(comparison_df)}"
)

print(
    f"Matching OCR predictions: "
    f"{same_prediction_count}"
)

print(
    f"Prediction agreement: "
    f"{prediction_match_rate * 100:.2f}%"
)


print(
    "\nFirst prediction comparisons:"
)

print(
    comparison_df.head(15).to_string(
        index=False
    )
)


# =============================================================================
# STEP 4U — DECISION GATE
# =============================================================================

section(
    "STEP 4U — INT8 QUALITY GATE"
)


# -----------------------------------------------------------------------------
# We use prediction agreement as the principal sanity signal.
#
# We deliberately do NOT demand 100% raw floating-point agreement.
# Quantization naturally changes the numerical values.
#
# However, if the INT8 model returns empty predictions while FP32
# returns valid OCR predictions, that is a clear failure.
# -----------------------------------------------------------------------------

fp32_empty = int(
    (
        comparison_df[
            "fp32_text"
        ].astype(str).str.len()
        == 0
    ).sum()
)


int8_empty = int(
    (
        comparison_df[
            "int8_text"
        ].astype(str).str.len()
        == 0
    ).sum()
)


print(
    f"FP32 empty predictions : "
    f"{fp32_empty}"
)

print(
    f"INT8 empty predictions : "
    f"{int8_empty}"
)


# -----------------------------------------------------------------------------
# Conservative acceptance criterion
# -----------------------------------------------------------------------------

MIN_PREDICTION_AGREEMENT = 0.80


quality_passed = (
    prediction_match_rate
    >= MIN_PREDICTION_AGREEMENT
)


if int8_empty > fp32_empty + (
    0.20 *
    len(comparison_df)
):

    quality_passed = False


if not quality_passed:

    print(
        "\n❌ INT8 QUALITY GATE FAILED"
    )

    print(
        f"""
The INT8 model does not sufficiently preserve
the FP32 OCR predictions.

Prediction agreement:
    {prediction_match_rate * 100:.2f}%

Required minimum:
    {MIN_PREDICTION_AGREEMENT * 100:.2f}%

INT8 empty predictions:
    {int8_empty}

The full 508-image validation will NOT be run.

This is intentional. It prevents wasting Colab
GPU time on a quantized model that is already
known to be unsuitable.
"""
    )

    # Save diagnostic results before stopping.

    diagnostic_path = (
        VALIDATION_DIR /
        "step4_int8_sanity_comparison.csv"
    )

    comparison_df.to_csv(
        diagnostic_path,
        index=False,
    )


    raw_output_path = (
        VALIDATION_DIR /
        "step4_int8_raw_output_sanity.csv"
    )

    sanity_df.to_csv(
        raw_output_path,
        index=False,
    )


    print(
        f"\n✓ Diagnostic prediction comparison saved:"
        f"\n  {diagnostic_path}"
    )

    print(
        f"\n✓ Raw output diagnostic saved:"
        f"\n  {raw_output_path}"
    )


    raise RuntimeError(
        "\nINT8 model rejected by quality gate."
        "\nFull validation was intentionally skipped."
    )


print(
    "\n✓ INT8 quality gate passed"
)

print(
    "✓ Proceeding to full 508-image validation"
)


# =============================================================================
# STEP 4V — FULL FP32 / INT8 DATASET BENCHMARK
# =============================================================================

section(
    "STEP 4V — FULL FP32 vs INT8 CUDA BENCHMARK"
)


benchmark_rows = []


def benchmark_session(
    session,
    input_name,
    images,
    precision_name,
):

    times = []

    start_total = time.perf_counter()


    for index, image_path in enumerate(
        images,
        start=1,
    ):

        tensor = preprocess_image(
            image_path
        )


        start = time.perf_counter()


        session.run(
            None,
            {
                input_name:
                tensor
            }
        )


        elapsed = (
            time.perf_counter() -
            start
        ) * 1000.0


        times.append(
            elapsed
        )


        if (
            index % 50 == 0
            or
            index == len(images)
        ):

            print(
                f"{precision_name}: "
                f"{index}/{len(images)}"
            )


    total_time = (
        time.perf_counter() -
        start_total
    )


    times = np.asarray(
        times,
        dtype=np.float64,
    )


    return {

        "precision":
            precision_name,

        "samples":
            len(images),

        "total_s":
            float(total_time),

        "mean_ms":
            float(np.mean(times)),

        "median_ms":
            float(np.median(times)),

        "p95_ms":
            float(
                np.percentile(
                    times,
                    95
                )
            ),

        "min_ms":
            float(np.min(times)),

        "max_ms":
            float(np.max(times)),

        "fps":
            float(
                1000.0 /
                np.mean(times)
            ),

    }


fp32_benchmark = benchmark_session(

    FP32_SESSION,

    fp32_input.name,

    all_images,

    "FP32",

)


int8_benchmark = benchmark_session(

    INT8_SESSION,

    int8_input.name,

    all_images,

    "INT8",

)


benchmark_rows.extend(
    [
        fp32_benchmark,
        int8_benchmark,
    ]
)


benchmark_df = pd.DataFrame(
    benchmark_rows
)


print(
    "\nBenchmark results:"
)

print(
    benchmark_df.to_string(
        index=False
    )
)


# =============================================================================
# STEP 4W — FINAL SPEED COMPARISON
# =============================================================================

section(
    "STEP 4W — FINAL PERFORMANCE COMPARISON"
)


fp32_mean = (
    fp32_benchmark[
        "mean_ms"
    ]
)

int8_mean = (
    int8_benchmark[
        "mean_ms"
    ]
)


speedup = (
    fp32_mean /
    int8_mean
)


latency_change = (
    (
        int8_mean -
        fp32_mean
    )
    /
    fp32_mean
    *
    100
)


fps_change = (
    (
        int8_benchmark["fps"] -
        fp32_benchmark["fps"]
    )
    /
    fp32_benchmark["fps"]
    *
    100
)


print(
    f"""
FP32 mean latency:
    {fp32_mean:.3f} ms

INT8 mean latency:
    {int8_mean:.3f} ms

INT8 speedup:
    {speedup:.3f}x

Latency change:
    {latency_change:+.2f}%

FP32 FPS:
    {fp32_benchmark["fps"]:.3f}

INT8 FPS:
    {int8_benchmark["fps"]:.3f}

FPS change:
    {fps_change:+.2f}%
"""
)


# =============================================================================
# STEP 4X — SAVE RESULTS
# =============================================================================

section(
    "STEP 4X — SAVING RESULTS"
)


sanity_csv = (
    VALIDATION_DIR /
    "step4_int8_sanity_prediction_comparison.csv"
)


comparison_df.to_csv(
    sanity_csv,
    index=False,
)


raw_output_csv = (
    VALIDATION_DIR /
    "step4_int8_raw_output_comparison.csv"
)


sanity_df.to_csv(
    raw_output_csv,
    index=False,
)


benchmark_csv = (
    VALIDATION_DIR /
    "step4_int8_fp32_benchmark.csv"
)


benchmark_df.to_csv(
    benchmark_csv,
    index=False,
)


# =============================================================================
# JSON SUMMARY
# =============================================================================

summary = {

    "step":
        "STEP 4 — YOLO11 OCR INT8",

    "status":
        "PASSED",

    "model":
        "YOLO11 OCR",

    "input_shape":
        [1, 3, IMG_SIZE, IMG_SIZE],

    "test_images":
        len(all_images),

    "calibration_images":
        len(calibration_images),

    "random_seed":
        RANDOM_SEED,

    "quantization": {

        "format":
            "QDQ",

        "activation_type":
            "QUInt8",

        "weight_type":
            "QInt8",

        "per_channel":
            True,

        "calibration_method":
            "MinMax",

        "op_types":
            ["Conv"],

        "tail_hops":
            TAIL_HOPS,

        "excluded_nodes":
            len(nodes_to_exclude),

    },

    "model_size_mb": {

        "fp32":
            fp32_size,

        "fp16":
            fp16_size,

        "int8":
            int8_size,

    },

    "size_reduction_percent": {

        "fp32_to_int8":
            fp32_reduction,

        "fp16_to_int8":
            fp16_reduction,

    },

    "sanity_check": {

        "samples":
            len(comparison_df),

        "prediction_agreement":
            prediction_match_rate,

        "prediction_agreement_percent":
            prediction_match_rate * 100,

        "fp32_empty_predictions":
            fp32_empty,

        "int8_empty_predictions":
            int8_empty,

        "maximum_relative_output_error":
            max_relative_error,

        "quality_gate":
            "PASSED",

    },

    "performance": {

        "fp32_mean_latency_ms":
            fp32_mean,

        "int8_mean_latency_ms":
            int8_mean,

        "fp32_median_latency_ms":
            fp32_benchmark[
                "median_ms"
            ],

        "int8_median_latency_ms":
            int8_benchmark[
                "median_ms"
            ],

        "fp32_p95_latency_ms":
            fp32_benchmark[
                "p95_ms"
            ],

        "int8_p95_latency_ms":
            int8_benchmark[
                "p95_ms"
            ],

        "fp32_fps":
            fp32_benchmark[
                "fps"
            ],

        "int8_fps":
            int8_benchmark[
                "fps"
            ],

        "speedup":
            speedup,

        "latency_change_percent":
            latency_change,

        "fps_change_percent":
            fps_change,

    },

    "files": {

        "fp32":
            str(FP32_PATH),

        "fp16":
            str(FP16_PATH),

        "int8":
            str(INT8_PATH),

        "calibration_list":
            str(calibration_list),

        "sanity_csv":
            str(sanity_csv),

        "raw_output_csv":
            str(raw_output_csv),

        "benchmark_csv":
            str(benchmark_csv),

    },

}


summary_path = (
    VALIDATION_DIR /
    "step4_ocr_int8_cuda_summary.json"
)


with open(
    summary_path,
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=2,
    )


# =============================================================================
# FINAL REPORT
# =============================================================================

section(
    "STEP 4 — FINAL INT8 RESULT"
)


print(
    f"""
================================================================================
YOLO11 OCR INT8 QUANTIZATION — FINAL RESULT
================================================================================

TEST SET
--------
Images                 : {len(all_images)}
Calibration images     : {len(calibration_images)}
Input                  : {IMG_SIZE} × {IMG_SIZE}
GPU                    : {torch.cuda.get_device_name(0)}

QUANTIZATION
------------
Format                : QDQ
Activation            : UINT8
Weights               : INT8
Per-channel           : True
Quantized operations  : Conv
Protected tail nodes  : {len(nodes_to_exclude)}

MODEL SIZE
----------
FP32                  : {fp32_size:.2f} MB
FP16                  : {fp16_size:.2f} MB
INT8                  : {int8_size:.2f} MB

FP32 → INT8 reduction : {fp32_reduction:.2f}%

QUALITY
-------
Prediction agreement : {prediction_match_rate * 100:.2f}%
FP32 empty predictions : {fp32_empty}
INT8 empty predictions : {int8_empty}

QUALITY GATE          : PASSED

PERFORMANCE
-----------
FP32 latency          : {fp32_mean:.3f} ms
INT8 latency          : {int8_mean:.3f} ms

FP32 FPS              : {fp32_benchmark["fps"]:.3f}
INT8 FPS              : {int8_benchmark["fps"]:.3f}

INT8 speedup          : {speedup:.3f}x
Latency change        : {latency_change:+.2f}%
FPS change            : {fps_change:+.2f}%

OUTPUT
------
INT8 model:
    {INT8_PATH}

Summary:
    {summary_path}

================================================================================
STEP 4-OCR — INT8 COMPLETED
================================================================================
"""
)


# =============================================================================
# CLEANUP
# =============================================================================

del fp32_model
del preprocessed_model
del int8_model

gc.collect()

torch.cuda.empty_cache()

print(
    "\n✓ GPU memory cleanup completed."
)



STEP 4A — YOLO11 OCR INT8 CONFIGURATION

FP32 ONNX:
    /content/drive/MyDrive/Stage_PFA/OCR_OPTIMIZATION/02_onnx_fp32/yolo11_ocr_fp32.onnx

FP16 ONNX:
    /content/drive/MyDrive/Stage_PFA/OCR_OPTIMIZATION/03_onnx_fp16/yolo11_ocr_fp16.onnx

INT8 output:
    /content/drive/MyDrive/Stage_PFA/OCR_OPTIMIZATION/04_onnx_int8

Calibration:
    /content/drive/MyDrive/Stage_PFA/OCR_OPTIMIZATION/05_calibration

Validation:
    /content/drive/MyDrive/Stage_PFA/OCR_OPTIMIZATION/06_validation

Test images:
    /content/drive/MyDrive/Stage_PFA/datasets/test/images

Test labels:
    /content/drive/MyDrive/Stage_PFA/datasets/test/labels

Input resolution:
    608 × 608

Calibration images:
    200

Random seed:
    42

Quantization:
    ONNX Runtime static QDQ

Weights:
    INT8

Activations:
    UINT8

Quantization scope:
    Conv only

Tail protection:
    8 graph hops



STEP 4B — ENVIRONMENT VERIFICATION
Python       : 3.13.15
PyTorch      : 2.11.0+cu128
ONNX         : 1.22.0
ONNX Runtime : 1.26

In [3]:
# =============================================================================
# STEP 4Y — YOLO11 OCR FP32 vs INT8 FINAL GROUND-TRUTH EVALUATION
# =============================================================================
#
# PURPOSE
# -------
# Evaluate the EXISTING FP32 and INT8 ONNX OCR models on the complete
# 508-image test set.
#
# IMPORTANT
# ---------
# This script DOES NOT:
#   - retrain the model
#   - quantize the model
#   - modify FP32 ONNX
#   - modify FP16 ONNX
#   - modify INT8 ONNX
#   - modify the test dataset
#
# It ONLY performs evaluation.
#
# METRICS
# -------
# Ground-truth OCR:
#   1. Character Accuracy
#   2. CER (Character Error Rate)
#   3. Exact Match
#
# FP32 vs INT8:
#   4. Prediction agreement
#   5. Prediction mismatch rate
#   6. Empty prediction rate
#
# Performance:
#   7. Mean latency
#   8. Median latency
#   9. P95 latency
#  10. Minimum latency
#  11. Maximum latency
#  12. FPS
#
# OUTPUTS
# -------
# 06_validation/
#   step4_ocr_fp32_vs_int8_ground_truth_results.csv
#   step4_ocr_fp32_vs_int8_summary.json
#
# =============================================================================

import os
import json
import time
import gc
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import onnx
import onnxruntime as ort


# =============================================================================
# CONFIGURATION
# =============================================================================

FP32_MODEL = Path(
    "/content/drive/MyDrive/Stage_PFA/"
    "OCR_OPTIMIZATION/02_onnx_fp32/"
    "yolo11_ocr_fp32.onnx"
)

INT8_MODEL = Path(
    "/content/drive/MyDrive/Stage_PFA/"
    "OCR_OPTIMIZATION/04_onnx_int8/"
    "yolo11_ocr_int8.onnx"
)

TEST_IMAGES_DIR = Path(
    "/content/drive/MyDrive/Stage_PFA/"
    "datasets/test/images"
)

TEST_LABELS_DIR = Path(
    "/content/drive/MyDrive/Stage_PFA/"
    "datasets/test/labels"
)

VALIDATION_DIR = Path(
    "/content/drive/MyDrive/Stage_PFA/"
    "OCR_OPTIMIZATION/06_validation"
)

VALIDATION_DIR.mkdir(parents=True, exist_ok=True)


# -----------------------------------------------------------------------------
# Model / dataset settings
# -----------------------------------------------------------------------------

IMG_SIZE = 608

CONF_THRESHOLD = 0.25

# YOLO OCR classes from your trained model.
# IMPORTANT:
# These IDs correspond to the class mapping printed by your YOLO11 model.
CLASS_NAMES = {
    0: "0",
    1: "1",
    2: "2",
    3: "3",
    4: "4",
    5: "5",
    6: "6",
    7: "7",
    8: "8",
    9: "9",
    10: "a",
    11: "b",
    12: "d",
    13: "gr",
    14: "h",
    15: "t",
    16: "w",
    17: "waw",
    18: "y",
}


# =============================================================================
# OUTPUT FILES
# =============================================================================

PER_IMAGE_CSV = (
    VALIDATION_DIR /
    "step4_ocr_fp32_vs_int8_ground_truth_results.csv"
)

SUMMARY_JSON = (
    VALIDATION_DIR /
    "step4_ocr_fp32_vs_int8_summary.json"
)


# =============================================================================
# DISPLAY HELPERS
# =============================================================================

def section(title):
    print("\n")
    print("=" * 100)
    print(title)
    print("=" * 100)


def subsection(title):
    print("\n")
    print("-" * 100)
    print(title)
    print("-" * 100)


def file_size_mb(path):
    return path.stat().st_size / (1024 ** 2)


# =============================================================================
# STEP 4Y-A — VERIFY FILES
# =============================================================================

section("STEP 4Y-A — VERIFYING EXISTING ARTIFACTS")

required_files = [
    FP32_MODEL,
    INT8_MODEL,
    TEST_IMAGES_DIR,
    TEST_LABELS_DIR,
]

for path in required_files:

    if not path.exists():
        raise FileNotFoundError(
            f"\nRequired path does not exist:\n{path}"
        )

    print(f"✓ {path}")


print("\n✓ All required artifacts found")

print("\nModels:")
print(f"  FP32 : {file_size_mb(FP32_MODEL):.2f} MB")
print(f"  INT8 : {file_size_mb(INT8_MODEL):.2f} MB")


# =============================================================================
# STEP 4Y-B — VERIFY ONNX STRUCTURE
# =============================================================================

section("STEP 4Y-B — VERIFYING ONNX MODELS")

for name, model_path in [
    ("FP32", FP32_MODEL),
    ("INT8", INT8_MODEL),
]:

    print(f"\n{name} model:")

    model = onnx.load(str(model_path))
    onnx.checker.check_model(model)

    print("  ✓ ONNX checker passed")

    for inp in model.graph.input:

        dims = []

        for dim in inp.type.tensor_type.shape.dim:

            if dim.dim_value:
                dims.append(dim.dim_value)

            elif dim.dim_param:
                dims.append(dim.dim_param)

            else:
                dims.append("?")

        print(
            f"  Input : {inp.name}"
            f"\n  Shape : {dims}"
        )

    for out in model.graph.output:

        dims = []

        for dim in out.type.tensor_type.shape.dim:

            if dim.dim_value:
                dims.append(dim.dim_value)

            elif dim.dim_param:
                dims.append(dim.dim_param)

            else:
                dims.append("?")

        print(
            f"  Output: {out.name}"
            f"\n  Shape : {dims}"
        )


# =============================================================================
# STEP 4Y-C — COLLECT TEST DATA
# =============================================================================

section("STEP 4Y-C — PREPARING TEST DATA")

image_extensions = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
}

image_paths = sorted(
    [
        p
        for p in TEST_IMAGES_DIR.iterdir()
        if p.suffix.lower() in image_extensions
    ]
)

if len(image_paths) == 0:
    raise RuntimeError(
        f"No test images found in:\n{TEST_IMAGES_DIR}"
    )

print(f"Test images found : {len(image_paths)}")

if len(image_paths) != 508:

    print(
        "\n⚠ WARNING:"
        f"\nExpected approximately 508 test images, "
        f"but found {len(image_paths)}."
    )

else:

    print("✓ Expected 508-image test set confirmed")


# =============================================================================
# STEP 4Y-D — YOLO LETTERBOX PREPROCESSING
# =============================================================================

section("STEP 4Y-D — CONFIGURING INPUT PREPROCESSING")


def letterbox(
    image,
    new_shape=(608, 608),
    color=(114, 114, 114)
):
    """
    Ultralytics-compatible letterbox preprocessing.
    """

    shape = image.shape[:2]  # height, width

    r = min(
        new_shape[0] / shape[0],
        new_shape[1] / shape[1]
    )

    new_unpad = (
        int(round(shape[1] * r)),
        int(round(shape[0] * r)),
    )

    dw = new_shape[1] - new_unpad[0]
    dh = new_shape[0] - new_unpad[1]

    dw /= 2
    dh /= 2

    if shape[::-1] != new_unpad:

        image = cv2.resize(
            image,
            new_unpad,
            interpolation=cv2.INTER_LINEAR,
        )

    top = int(round(dh - 0.1))
    bottom = int(round(dh + 0.1))

    left = int(round(dw - 0.1))
    right = int(round(dw + 0.1))

    image = cv2.copyMakeBorder(
        image,
        top,
        bottom,
        left,
        right,
        cv2.BORDER_CONSTANT,
        value=color,
    )

    return image


def preprocess_image(image_path):
    """
    Convert image to ONNX input tensor.

    BGR
      ↓
    RGB
      ↓
    Letterbox
      ↓
    float32
      ↓
    /255
      ↓
    HWC → CHW
      ↓
    batch dimension
    """

    image = cv2.imread(str(image_path))

    if image is None:
        raise RuntimeError(
            f"Could not read image:\n{image_path}"
        )

    image = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2RGB
    )

    image = letterbox(
        image,
        (IMG_SIZE, IMG_SIZE)
    )

    image = image.astype(
        np.float32
    ) / 255.0

    image = image.transpose(
        2,
        0,
        1
    )

    image = np.expand_dims(
        image,
        axis=0
    )

    return np.ascontiguousarray(image)


# =============================================================================
# STEP 4Y-E — GROUND TRUTH RECONSTRUCTION
# =============================================================================

section("STEP 4Y-E — CONFIGURING GROUND-TRUTH OCR RECONSTRUCTION")


def read_ground_truth(label_path):

    if not label_path.exists():

        return {
            "text": "",
            "objects": [],
            "missing": True,
        }

    objects = []

    with open(
        label_path,
        "r",
        encoding="utf-8"
    ) as f:

        lines = [
            line.strip()
            for line in f
            if line.strip()
        ]

    for line in lines:

        parts = line.split()

        if len(parts) < 5:
            continue

        try:

            class_id = int(float(parts[0]))
            x_center = float(parts[1])
            y_center = float(parts[2])
            width = float(parts[3])
            height = float(parts[4])

        except ValueError:
            continue

        if class_id not in CLASS_NAMES:
            continue

        objects.append({
            "class_id": class_id,
            "text": CLASS_NAMES[class_id],
            "x_center": x_center,
            "y_center": y_center,
            "width": width,
            "height": height,
        })

    # Sort characters from left to right.
    objects.sort(
        key=lambda x: x["x_center"]
    )

    text = "".join(
        obj["text"]
        for obj in objects
    )

    return {
        "text": text,
        "objects": objects,
        "missing": False,
    }


# =============================================================================
# STEP 4Y-F — ONNX RUNTIME SESSIONS
# =============================================================================

section("STEP 4Y-F — INITIALIZING CUDA SESSIONS")

available_providers = ort.get_available_providers()

print("Available providers:")

for provider in available_providers:
    print(f"  - {provider}")

if "CUDAExecutionProvider" not in available_providers:

    raise RuntimeError(
        "\nCUDAExecutionProvider is not available."
    )

providers = [
    (
        "CUDAExecutionProvider",
        {
            "device_id": 0
        }
    ),
    "CPUExecutionProvider",
]


print("\n→ Loading FP32 session...")

fp32_session = ort.InferenceSession(
    str(FP32_MODEL),
    providers=providers,
)

print("✓ FP32 session initialized")

print("\n→ Loading INT8 session...")

int8_session = ort.InferenceSession(
    str(INT8_MODEL),
    providers=providers,
)

print("✓ INT8 session initialized")


print("\nFP32 active providers:")

for p in fp32_session.get_providers():
    print(f"  - {p}")


print("\nINT8 active providers:")

for p in int8_session.get_providers():
    print(f"  - {p}")


fp32_input_name = fp32_session.get_inputs()[0].name
int8_input_name = int8_session.get_inputs()[0].name

print("\nFP32 input :", fp32_input_name)
print("INT8 input :", int8_input_name)


# =============================================================================
# STEP 4Y-G — OUTPUT DECODER
# =============================================================================

section("STEP 4Y-G — CONFIGURING YOLO OCR OUTPUT DECODER")


def decode_predictions(
    output,
    confidence_threshold=CONF_THRESHOLD
):
    """
    Decode YOLO exported detection output.

    Expected output:
        [1, 300, 6]

    Columns:
        x1
        y1
        x2
        y2
        confidence
        class_id

    Characters are sorted from left to right.
    """

    output = np.asarray(output)

    if output.ndim == 3:

        output = output[0]

    if output.ndim != 2:

        raise RuntimeError(
            f"Unexpected output shape: {output.shape}"
        )

    if output.shape[1] < 6:

        raise RuntimeError(
            f"Unexpected output dimensions: {output.shape}"
        )

    detections = []

    for row in output:

        x1 = float(row[0])
        y1 = float(row[1])
        x2 = float(row[2])
        y2 = float(row[3])

        confidence = float(row[4])
        class_id = int(round(float(row[5])))

        if confidence < confidence_threshold:
            continue

        if class_id not in CLASS_NAMES:
            continue

        x_center = (x1 + x2) / 2.0

        detections.append({
            "class_id": class_id,
            "text": CLASS_NAMES[class_id],
            "confidence": confidence,
            "x1": x1,
            "y1": y1,
            "x2": x2,
            "y2": y2,
            "x_center": x_center,
        })

    detections.sort(
        key=lambda x: x["x_center"]
    )

    text = "".join(
        det["text"]
        for det in detections
    )

    return text, detections


# =============================================================================
# STEP 4Y-H — EDIT DISTANCE / CER
# =============================================================================

section("STEP 4Y-H — CONFIGURING OCR METRICS")


def levenshtein_distance(a, b):

    if a == b:
        return 0

    if len(a) == 0:
        return len(b)

    if len(b) == 0:
        return len(a)

    previous = list(range(len(b) + 1))

    for i, char_a in enumerate(a, start=1):

        current = [i]

        for j, char_b in enumerate(b, start=1):

            insertion = current[j - 1] + 1
            deletion = previous[j] + 1
            substitution = previous[j - 1] + (
                char_a != char_b
            )

            current.append(
                min(
                    insertion,
                    deletion,
                    substitution,
                )
            )

        previous = current

    return previous[-1]


def calculate_cer(gt, prediction):

    if len(gt) == 0:

        if len(prediction) == 0:
            return 0.0

        return 1.0

    distance = levenshtein_distance(
        gt,
        prediction
    )

    return distance / len(gt)


def calculate_sequence_character_accuracy(
    gt,
    prediction
):
    """
    Character accuracy derived from sequence edit distance.

    This is:
        1 - CER

    It therefore directly corresponds to the CER metric.
    """

    if len(gt) == 0:

        return 1.0 if len(prediction) == 0 else 0.0

    cer = calculate_cer(
        gt,
        prediction
    )

    return max(
        0.0,
        1.0 - cer
    )


# =============================================================================
# STEP 4Y-I — WARM-UP
# =============================================================================

section("STEP 4Y-I — CUDA WARM-UP")

warmup_tensor = preprocess_image(
    image_paths[0]
)

for _ in range(5):

    fp32_session.run(
        None,
        {
            fp32_input_name:
            warmup_tensor
        }
    )

for _ in range(5):

    int8_session.run(
        None,
        {
            int8_input_name:
            warmup_tensor
        }
    )

print("✓ 5 FP32 warm-up runs completed")
print("✓ 5 INT8 warm-up runs completed")


# =============================================================================
# STEP 4Y-J — FULL TEST SET EVALUATION
# =============================================================================

section(
    "STEP 4Y-J — FULL 508-IMAGE FP32 vs INT8 "
    "GROUND-TRUTH EVALUATION"
)

results = []

fp32_latencies = []
int8_latencies = []


for index, image_path in enumerate(
    image_paths,
    start=1
):

    # -------------------------------------------------------------------------
    # Ground truth
    # -------------------------------------------------------------------------

    label_path = (
        TEST_LABELS_DIR /
        f"{image_path.stem}.txt"
    )

    gt_info = read_ground_truth(
        label_path
    )

    gt_text = gt_info["text"]

    # -------------------------------------------------------------------------
    # Preprocessing
    # -------------------------------------------------------------------------

    tensor = preprocess_image(
        image_path
    )

    # -------------------------------------------------------------------------
    # FP32 inference
    # -------------------------------------------------------------------------

    start = time.perf_counter()

    fp32_outputs = fp32_session.run(
        None,
        {
            fp32_input_name:
            tensor
        }
    )

    fp32_elapsed = (
        time.perf_counter() - start
    ) * 1000.0

    fp32_latencies.append(
        fp32_elapsed
    )

    fp32_text, fp32_detections = (
        decode_predictions(
            fp32_outputs[0]
        )
    )

    # -------------------------------------------------------------------------
    # INT8 inference
    # -------------------------------------------------------------------------

    start = time.perf_counter()

    int8_outputs = int8_session.run(
        None,
        {
            int8_input_name:
            tensor
        }
    )

    int8_elapsed = (
        time.perf_counter() - start
    ) * 1000.0

    int8_latencies.append(
        int8_elapsed
    )

    int8_text, int8_detections = (
        decode_predictions(
            int8_outputs[0]
        )
    )

    # -------------------------------------------------------------------------
    # Ground-truth metrics
    # -------------------------------------------------------------------------

    fp32_cer = calculate_cer(
        gt_text,
        fp32_text
    )

    int8_cer = calculate_cer(
        gt_text,
        int8_text
    )

    fp32_char_accuracy = (
        calculate_sequence_character_accuracy(
            gt_text,
            fp32_text
        )
    )

    int8_char_accuracy = (
        calculate_sequence_character_accuracy(
            gt_text,
            int8_text
        )
    )

    fp32_exact = (
        gt_text == fp32_text
    )

    int8_exact = (
        gt_text == int8_text
    )

    # -------------------------------------------------------------------------
    # FP32 vs INT8 agreement
    # -------------------------------------------------------------------------

    prediction_same = (
        fp32_text == int8_text
    )

    # -------------------------------------------------------------------------
    # Store result
    # -------------------------------------------------------------------------

    results.append({

        "image": image_path.name,

        "ground_truth": gt_text,

        "fp32_prediction": fp32_text,
        "int8_prediction": int8_text,

        "fp32_character_accuracy":
            fp32_char_accuracy,

        "int8_character_accuracy":
            int8_char_accuracy,

        "fp32_cer":
            fp32_cer,

        "int8_cer":
            int8_cer,

        "fp32_exact_match":
            int(fp32_exact),

        "int8_exact_match":
            int(int8_exact),

        "fp32_int8_same":
            int(prediction_same),

        "fp32_empty":
            int(len(fp32_text) == 0),

        "int8_empty":
            int(len(int8_text) == 0),

        "fp32_detection_count":
            len(fp32_detections),

        "int8_detection_count":
            len(int8_detections),

        "fp32_latency_ms":
            fp32_elapsed,

        "int8_latency_ms":
            int8_elapsed,

        "latency_difference_ms":
            int8_elapsed - fp32_elapsed,

        "gt_length":
            len(gt_text),

        "fp32_length":
            len(fp32_text),

        "int8_length":
            len(int8_text),
    })

    # -------------------------------------------------------------------------
    # Progress
    # -------------------------------------------------------------------------

    if (
        index % 50 == 0
        or index == len(image_paths)
    ):

        print(
            f"Processed "
            f"{index}/{len(image_paths)}"
        )


# =============================================================================
# STEP 4Y-K — DATAFRAME
# =============================================================================

section("STEP 4Y-K — BUILDING RESULTS TABLE")

results_df = pd.DataFrame(
    results
)

results_df.to_csv(
    PER_IMAGE_CSV,
    index=False
)

print(
    f"✓ Per-image results saved:\n"
    f"  {PER_IMAGE_CSV}"
)


# =============================================================================
# STEP 4Y-L — AGGREGATE OCR METRICS
# =============================================================================

section(
    "STEP 4Y-L — GROUND-TRUTH OCR METRICS"
)


def safe_mean(series):

    if len(series) == 0:
        return np.nan

    return float(
        np.mean(series)
    )


# -----------------------------------------------------------------------------
# FP32
# -----------------------------------------------------------------------------

fp32_character_accuracy = (
    results_df[
        "fp32_character_accuracy"
    ].mean()
)

int8_character_accuracy = (
    results_df[
        "int8_character_accuracy"
    ].mean()
)

fp32_cer = (
    results_df[
        "fp32_cer"
    ].mean()
)

int8_cer = (
    results_df[
        "int8_cer"
    ].mean()
)

fp32_exact_match = (
    results_df[
        "fp32_exact_match"
    ].mean()
)

int8_exact_match = (
    results_df[
        "int8_exact_match"
    ].mean()
)


print("\nFP32 OCR:")

print(
    f"  Character Accuracy : "
    f"{fp32_character_accuracy * 100:.4f}%"
)

print(
    f"  CER                : "
    f"{fp32_cer:.6f}"
)

print(
    f"  Exact Match        : "
    f"{fp32_exact_match * 100:.4f}%"
)


print("\nINT8 OCR:")

print(
    f"  Character Accuracy : "
    f"{int8_character_accuracy * 100:.4f}%"
)

print(
    f"  CER                : "
    f"{int8_cer:.6f}"
)

print(
    f"  Exact Match        : "
    f"{int8_exact_match * 100:.4f}%"
)


# =============================================================================
# STEP 4Y-M — FP32 / INT8 AGREEMENT
# =============================================================================

section(
    "STEP 4Y-M — FP32 vs INT8 PREDICTION AGREEMENT"
)

agreement = (
    results_df[
        "fp32_int8_same"
    ].mean()
)

mismatch_rate = 1.0 - agreement

fp32_empty_rate = (
    results_df[
        "fp32_empty"
    ].mean()
)

int8_empty_rate = (
    results_df[
        "int8_empty"
    ].mean()
)


print(
    f"Samples evaluated        : "
    f"{len(results_df)}"
)

print(
    f"Matching predictions     : "
    f"{int(results_df['fp32_int8_same'].sum())}"
)

print(
    f"Prediction agreement     : "
    f"{agreement * 100:.4f}%"
)

print(
    f"Prediction mismatch rate : "
    f"{mismatch_rate * 100:.4f}%"
)

print(
    f"\nFP32 empty predictions   : "
    f"{int(results_df['fp32_empty'].sum())}"
)

print(
    f"INT8 empty predictions   : "
    f"{int(results_df['int8_empty'].sum())}"
)


# =============================================================================
# STEP 4Y-N — OCR ACCURACY CHANGE
# =============================================================================

section(
    "STEP 4Y-N — FP32 → INT8 OCR QUALITY CHANGE"
)

character_accuracy_delta = (
    int8_character_accuracy
    - fp32_character_accuracy
)

cer_delta = (
    int8_cer
    - fp32_cer
)

exact_match_delta = (
    int8_exact_match
    - fp32_exact_match
)


print(
    f"Character Accuracy change : "
    f"{character_accuracy_delta * 100:+.4f} percentage points"
)

print(
    f"CER change                : "
    f"{cer_delta:+.6f}"
)

print(
    f"Exact Match change        : "
    f"{exact_match_delta * 100:+.4f} percentage points"
)


# =============================================================================
# STEP 4Y-O — LATENCY STATISTICS
# =============================================================================

section(
    "STEP 4Y-O — PERFORMANCE STATISTICS"
)


def latency_statistics(values):

    values = np.asarray(
        values,
        dtype=np.float64
    )

    return {
        "mean_ms":
            float(np.mean(values)),

        "median_ms":
            float(np.median(values)),

        "p95_ms":
            float(np.percentile(values, 95)),

        "min_ms":
            float(np.min(values)),

        "max_ms":
            float(np.max(values)),

        "fps":
            float(
                1000.0 /
                np.mean(values)
            ),
    }


fp32_perf = latency_statistics(
    fp32_latencies
)

int8_perf = latency_statistics(
    int8_latencies
)


print("\nFP32:")

for key, value in fp32_perf.items():

    if key == "fps":

        print(
            f"  FPS            : "
            f"{value:.3f}"
        )

    else:

        print(
            f"  {key:<14}: "
            f"{value:.3f} ms"
        )


print("\nINT8:")

for key, value in int8_perf.items():

    if key == "fps":

        print(
            f"  FPS            : "
            f"{value:.3f}"
        )

    else:

        print(
            f"  {key:<14}: "
            f"{value:.3f} ms"
        )


# =============================================================================
# STEP 4Y-P — PERFORMANCE DIFFERENCE
# =============================================================================

section(
    "STEP 4Y-P — FP32 → INT8 PERFORMANCE CHANGE"
)

latency_speedup = (
    fp32_perf["mean_ms"]
    /
    int8_perf["mean_ms"]
)

latency_change_percent = (
    (
        int8_perf["mean_ms"]
        -
        fp32_perf["mean_ms"]
    )
    /
    fp32_perf["mean_ms"]
) * 100.0

fps_change_percent = (
    (
        int8_perf["fps"]
        -
        fp32_perf["fps"]
    )
    /
    fp32_perf["fps"]
) * 100.0


print(
    f"FP32 mean latency : "
    f"{fp32_perf['mean_ms']:.3f} ms"
)

print(
    f"INT8 mean latency : "
    f"{int8_perf['mean_ms']:.3f} ms"
)

print(
    f"\nINT8 speedup      : "
    f"{latency_speedup:.3f}x"
)

print(
    f"Latency change    : "
    f"{latency_change_percent:+.2f}%"
)

print(
    f"\nFP32 FPS          : "
    f"{fp32_perf['fps']:.3f}"
)

print(
    f"INT8 FPS          : "
    f"{int8_perf['fps']:.3f}"
)

print(
    f"FPS change        : "
    f"{fps_change_percent:+.2f}%"
)


# =============================================================================
# STEP 4Y-Q — MODEL SIZE
# =============================================================================

section(
    "STEP 4Y-Q — MODEL SIZE COMPARISON"
)

fp32_size = file_size_mb(
    FP32_MODEL
)

int8_size = file_size_mb(
    INT8_MODEL
)

size_reduction_mb = (
    fp32_size -
    int8_size
)

size_reduction_percent = (
    size_reduction_mb /
    fp32_size
) * 100.0


print(
    f"FP32 : {fp32_size:.2f} MB"
)

print(
    f"INT8 : {int8_size:.2f} MB"
)

print(
    f"\nFP32 → INT8 reduction : "
    f"{size_reduction_mb:.2f} MB"
)

print(
    f"FP32 → INT8 reduction : "
    f"{size_reduction_percent:.2f}%"
)


# =============================================================================
# STEP 4Y-R — FINAL COMPARISON TABLE
# =============================================================================

section(
    "STEP 4Y-R — FINAL FP32 vs INT8 COMPARISON"
)

comparison_df = pd.DataFrame({

    "Metric": [

        "Character Accuracy",
        "CER",
        "Exact Match",

        "Mean latency (ms)",
        "Median latency (ms)",
        "P95 latency (ms)",
        "Minimum latency (ms)",
        "Maximum latency (ms)",
        "FPS",

        "Model size (MB)",
    ],

    "FP32": [

        f"{fp32_character_accuracy * 100:.4f}%",
        f"{fp32_cer:.6f}",
        f"{fp32_exact_match * 100:.4f}%",

        f"{fp32_perf['mean_ms']:.3f}",
        f"{fp32_perf['median_ms']:.3f}",
        f"{fp32_perf['p95_ms']:.3f}",
        f"{fp32_perf['min_ms']:.3f}",
        f"{fp32_perf['max_ms']:.3f}",
        f"{fp32_perf['fps']:.3f}",

        f"{fp32_size:.2f}",
    ],

    "INT8": [

        f"{int8_character_accuracy * 100:.4f}%",
        f"{int8_cer:.6f}",
        f"{int8_exact_match * 100:.4f}%",

        f"{int8_perf['mean_ms']:.3f}",
        f"{int8_perf['median_ms']:.3f}",
        f"{int8_perf['p95_ms']:.3f}",
        f"{int8_perf['min_ms']:.3f}",
        f"{int8_perf['max_ms']:.3f}",
        f"{int8_perf['fps']:.3f}",

        f"{int8_size:.2f}",
    ],
})


print(
    comparison_df.to_string(
        index=False
    )
)


# =============================================================================
# STEP 4Y-S — FINAL JSON SUMMARY
# =============================================================================

section(
    "STEP 4Y-S — SAVING FINAL JSON SUMMARY"
)

summary = {

    "step":
        "STEP 4 — YOLO11 OCR FP32 vs INT8 final evaluation",

    "dataset": {

        "test_images":
            int(len(results_df)),

        "test_images_directory":
            str(TEST_IMAGES_DIR),

        "test_labels_directory":
            str(TEST_LABELS_DIR),
    },

    "input": {

        "resolution":
            f"{IMG_SIZE}x{IMG_SIZE}",

        "confidence_threshold":
            CONF_THRESHOLD,
    },

    "models": {

        "fp32": {

            "path":
                str(FP32_MODEL),

            "size_mb":
                fp32_size,
        },

        "int8": {

            "path":
                str(INT8_MODEL),

            "size_mb":
                int8_size,

            "size_reduction_percent":
                size_reduction_percent,
        },
    },

    "ocr_accuracy": {

        "fp32": {

            "character_accuracy":
                float(
                    fp32_character_accuracy
                ),

            "cer":
                float(fp32_cer),

            "exact_match":
                float(fp32_exact_match),
        },

        "int8": {

            "character_accuracy":
                float(
                    int8_character_accuracy
                ),

            "cer":
                float(int8_cer),

            "exact_match":
                float(int8_exact_match),
        },

        "change": {

            "character_accuracy_percentage_points":
                float(
                    character_accuracy_delta
                    * 100
                ),

            "cer":
                float(cer_delta),

            "exact_match_percentage_points":
                float(
                    exact_match_delta
                    * 100
                ),
        },
    },

    "fp32_int8_agreement": {

        "matching_predictions":
            int(
                results_df[
                    "fp32_int8_same"
                ].sum()
            ),

        "total_predictions":
            int(len(results_df)),

        "agreement":
            float(agreement),

        "mismatch_rate":
            float(mismatch_rate),

        "fp32_empty_predictions":
            int(
                results_df[
                    "fp32_empty"
                ].sum()
            ),

        "int8_empty_predictions":
            int(
                results_df[
                    "int8_empty"
                ].sum()
            ),
    },

    "performance": {

        "fp32": fp32_perf,

        "int8": int8_perf,

        "comparison": {

            "int8_speedup":
                float(latency_speedup),

            "latency_change_percent":
                float(
                    latency_change_percent
                ),

            "fps_change_percent":
                float(
                    fps_change_percent
                ),
        },
    },

    "quantization": {

        "format":
            "QDQ",

        "weights":
            "INT8",

        "activations":
            "UINT8",

        "per_channel":
            True,

        "quantized_operations":
            "Conv only",

        "calibration_images":
            200,

        "protected_tail_hops":
            8,
    },

    "files": {

        "per_image_results":
            str(PER_IMAGE_CSV),

        "summary":
            str(SUMMARY_JSON),
    },
}


with open(
    SUMMARY_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )


print(
    f"✓ JSON summary saved:\n"
    f"  {SUMMARY_JSON}"
)


# =============================================================================
# STEP 4Y-T — DISPLAY FINAL RESULT
# =============================================================================

section(
    "STEP 4Y-T — FINAL RESULT"
)

print("""
================================================================================
YOLO11 OCR — FP32 vs INT8 FINAL GROUND-TRUTH EVALUATION
================================================================================
""")

print("TEST SET")
print("--------")
print(
    f"Images                 : "
    f"{len(results_df)}"
)

print(
    f"Input                  : "
    f"{IMG_SIZE} × {IMG_SIZE}"
)


print("\nOCR QUALITY")
print("----------")

print(
    f"FP32 Character Accuracy : "
    f"{fp32_character_accuracy * 100:.4f}%"
)

print(
    f"INT8 Character Accuracy : "
    f"{int8_character_accuracy * 100:.4f}%"
)

print(
    f"FP32 CER                : "
    f"{fp32_cer:.6f}"
)

print(
    f"INT8 CER                : "
    f"{int8_cer:.6f}"
)

print(
    f"FP32 Exact Match        : "
    f"{fp32_exact_match * 100:.4f}%"
)

print(
    f"INT8 Exact Match        : "
    f"{int8_exact_match * 100:.4f}%"
)


print("\nFP32 → INT8 QUALITY CHANGE")
print("--------------------------")

print(
    f"Character Accuracy     : "
    f"{character_accuracy_delta * 100:+.4f} pp"
)

print(
    f"CER                    : "
    f"{cer_delta:+.6f}"
)

print(
    f"Exact Match             : "
    f"{exact_match_delta * 100:+.4f} pp"
)


print("\nFP32 ↔ INT8 AGREEMENT")
print("---------------------")

print(
    f"Agreement               : "
    f"{agreement * 100:.4f}%"
)

print(
    f"Mismatch rate           : "
    f"{mismatch_rate * 100:.4f}%"
)

print(
    f"FP32 empty predictions  : "
    f"{int(results_df['fp32_empty'].sum())}"
)

print(
    f"INT8 empty predictions  : "
    f"{int(results_df['int8_empty'].sum())}"
)


print("\nPERFORMANCE")
print("-----------")

print(
    f"FP32 latency            : "
    f"{fp32_perf['mean_ms']:.3f} ms"
)

print(
    f"INT8 latency            : "
    f"{int8_perf['mean_ms']:.3f} ms"
)

print(
    f"FP32 FPS                : "
    f"{fp32_perf['fps']:.3f}"
)

print(
    f"INT8 FPS                : "
    f"{int8_perf['fps']:.3f}"
)

print(
    f"INT8 speedup            : "
    f"{latency_speedup:.3f}x"
)

print(
    f"Latency change          : "
    f"{latency_change_percent:+.2f}%"
)

print(
    f"FPS change              : "
    f"{fps_change_percent:+.2f}%"
)


print("\nMODEL SIZE")
print("----------")

print(
    f"FP32                    : "
    f"{fp32_size:.2f} MB"
)

print(
    f"INT8                    : "
    f"{int8_size:.2f} MB"
)

print(
    f"FP32 → INT8 reduction   : "
    f"{size_reduction_percent:.2f}%"
)


print("""
================================================================================
FILES
================================================================================
""")

print(
    f"Per-image CSV:\n"
    f"  {PER_IMAGE_CSV}"
)

print(
    f"\nSummary JSON:\n"
    f"  {SUMMARY_JSON}"
)


print("""
================================================================================
STEP 4Y — COMPLETE
================================================================================
""")

gc.collect()

print("✓ Evaluation completed.")
print("✓ Existing FP32 model untouched.")
print("✓ Existing INT8 model untouched.")
print("✓ Test dataset untouched.")
print("✓ No quantization performed.")
print("✓ Ground-truth OCR metrics calculated.")
print("✓ FP32/INT8 agreement calculated.")
print("✓ Performance metrics calculated.")
print("✓ Results saved.")



STEP 4Y-A — VERIFYING EXISTING ARTIFACTS
✓ /content/drive/MyDrive/Stage_PFA/OCR_OPTIMIZATION/02_onnx_fp32/yolo11_ocr_fp32.onnx
✓ /content/drive/MyDrive/Stage_PFA/OCR_OPTIMIZATION/04_onnx_int8/yolo11_ocr_int8.onnx
✓ /content/drive/MyDrive/Stage_PFA/datasets/test/images
✓ /content/drive/MyDrive/Stage_PFA/datasets/test/labels

✓ All required artifacts found

Models:
  FP32 : 10.14 MB
  INT8 : 3.03 MB


STEP 4Y-B — VERIFYING ONNX MODELS

FP32 model:
  ✓ ONNX checker passed
  Input : images
  Shape : [1, 3, 608, 608]
  Output: output0
  Shape : [1, 300, 6]

INT8 model:
  ✓ ONNX checker passed
  Input : images
  Shape : [1, 3, 608, 608]
  Output: output0
  Shape : [1, 300, 6]


STEP 4Y-C — PREPARING TEST DATA
Test images found : 508
✓ Expected 508-image test set confirmed


STEP 4Y-D — CONFIGURING INPUT PREPROCESSING


STEP 4Y-E — CONFIGURING GROUND-TRUTH OCR RECONSTRUCTION


STEP 4Y-F — INITIALIZING CUDA SESSIONS
Available providers:
  - TensorrtExecutionProvider
  - CUDAExecutionProvider

In [2]:
# =============================================================================
# STEP 4Z — YOLO11 OCR FP32 vs FP16 vs INT8
#             FINAL GROUND-TRUTH EVALUATION
# =============================================================================
#
# PURPOSE
# -------
# Evaluate the EXISTING FP32, FP16 and INT8 ONNX models on the exact same
# 508-image test set.
#
# IMPORTANT INPUT-TYPE HANDLING
# -----------------------------
# FP32 ONNX -> float32 input
# FP16 ONNX -> float16 input
# INT8 ONNX -> float32 input
#
# The models themselves are NOT modified.
#
# THIS SCRIPT DOES NOT:
#   - retrain any model
#   - quantize any model
#   - modify FP32 ONNX
#   - modify FP16 ONNX
#   - modify INT8 ONNX
#   - modify the dataset
#
# MODELS
# ------
# FP32:
#   02_onnx_fp32/yolo11_ocr_fp32.onnx
#
# FP16:
#   03_onnx_fp16/yolo11_ocr_fp16.onnx
#
# INT8:
#   04_onnx_int8/yolo11_ocr_int8.onnx
#
# DATASET
# -------
# Test images : 508
# Test labels : YOLO-format character annotations
#
# METRICS
# -------
# OCR QUALITY:
#   1. Character Accuracy
#   2. CER
#   3. Exact Match
#
# MODEL AGREEMENT:
#   4. FP32 ↔ FP16 agreement
#   5. FP32 ↔ INT8 agreement
#   6. FP16 ↔ INT8 agreement
#
# PERFORMANCE:
#   7. Mean latency
#   8. Median latency
#   9. P95 latency
#  10. Minimum latency
#  11. Maximum latency
#  12. FPS
#
# MODEL SIZE:
#  13. Model size
#
# OUTPUTS
# -------
# 06_validation/
#
#   step4z_ocr_fp32_fp16_int8_ground_truth_results.csv
#   step4z_ocr_fp32_fp16_int8_summary.json
#
# =============================================================================


import os
import json
import time
import gc
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import onnx
import onnxruntime as ort


# =============================================================================
# CONFIGURATION
# =============================================================================

FP32_MODEL = Path(
    "/content/drive/MyDrive/Stage_PFA/"
    "OCR_OPTIMIZATION/02_onnx_fp32/"
    "yolo11_ocr_fp32.onnx"
)

FP16_MODEL = Path(
    "/content/drive/MyDrive/Stage_PFA/"
    "OCR_OPTIMIZATION/03_onnx_fp16/"
    "yolo11_ocr_fp16.onnx"
)

INT8_MODEL = Path(
    "/content/drive/MyDrive/Stage_PFA/"
    "OCR_OPTIMIZATION/04_onnx_int8/"
    "yolo11_ocr_int8.onnx"
)

TEST_IMAGES_DIR = Path(
    "/content/drive/MyDrive/Stage_PFA/"
    "datasets/test/images"
)

TEST_LABELS_DIR = Path(
    "/content/drive/MyDrive/Stage_PFA/"
    "datasets/test/labels"
)

VALIDATION_DIR = Path(
    "/content/drive/MyDrive/Stage_PFA/"
    "OCR_OPTIMIZATION/06_validation"
)

VALIDATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# MODEL / DATASET SETTINGS
# =============================================================================

IMG_SIZE = 608

CONF_THRESHOLD = 0.25

EXPECTED_TEST_IMAGES = 508

WARMUP_RUNS = 10


# =============================================================================
# CLASS MAPPING
# =============================================================================

CLASS_NAMES = {

    0: "0",
    1: "1",
    2: "2",
    3: "3",
    4: "4",
    5: "5",
    6: "6",
    7: "7",
    8: "8",
    9: "9",

    10: "a",
    11: "b",
    12: "d",
    13: "gr",
    14: "h",
    15: "t",
    16: "w",
    17: "waw",
    18: "y",
}


# =============================================================================
# OUTPUT FILES
# =============================================================================

PER_IMAGE_CSV = (
    VALIDATION_DIR /
    "step4z_ocr_fp32_fp16_int8_ground_truth_results.csv"
)

SUMMARY_JSON = (
    VALIDATION_DIR /
    "step4z_ocr_fp32_fp16_int8_summary.json"
)


# =============================================================================
# DISPLAY HELPERS
# =============================================================================

def section(title):

    print("\n")
    print("=" * 100)
    print(title)
    print("=" * 100)


def file_size_mb(path):

    return (
        path.stat().st_size /
        (1024 ** 2)
    )


# =============================================================================
# STEP 4Z-A — VERIFY ARTIFACTS
# =============================================================================

section(
    "STEP 4Z-A — VERIFYING EXISTING ARTIFACTS"
)


required_files = [

    FP32_MODEL,
    FP16_MODEL,
    INT8_MODEL,

    TEST_IMAGES_DIR,
    TEST_LABELS_DIR,
]


for path in required_files:

    if not path.exists():

        raise FileNotFoundError(
            f"\nRequired path does not exist:\n{path}"
        )

    print(f"✓ {path}")


print(
    "\n✓ All required artifacts found"
)


print("\nModel sizes:")

print(
    f"  FP32 : "
    f"{file_size_mb(FP32_MODEL):.2f} MB"
)

print(
    f"  FP16 : "
    f"{file_size_mb(FP16_MODEL):.2f} MB"
)

print(
    f"  INT8 : "
    f"{file_size_mb(INT8_MODEL):.2f} MB"
)


# =============================================================================
# STEP 4Z-B — VERIFY ONNX STRUCTURE
# =============================================================================

section(
    "STEP 4Z-B — VERIFYING ONNX MODELS"
)


model_paths = {

    "FP32": FP32_MODEL,
    "FP16": FP16_MODEL,
    "INT8": INT8_MODEL,
}


model_metadata = {}


for model_name, model_path in model_paths.items():

    print(f"\n{model_name} model:")

    model = onnx.load(
        str(model_path)
    )

    onnx.checker.check_model(
        model
    )

    print(
        "  ✓ ONNX checker passed"
    )

    inputs = model.graph.input
    outputs = model.graph.output

    input_name = inputs[0].name
    output_name = outputs[0].name

    input_shape = []

    for dim in (
        inputs[0]
        .type
        .tensor_type
        .shape
        .dim
    ):

        if dim.dim_value:

            input_shape.append(
                dim.dim_value
            )

        elif dim.dim_param:

            input_shape.append(
                dim.dim_param
            )

        else:

            input_shape.append("?")

    output_shape = []

    for dim in (
        outputs[0]
        .type
        .tensor_type
        .shape
        .dim
    ):

        if dim.dim_value:

            output_shape.append(
                dim.dim_value
            )

        elif dim.dim_param:

            output_shape.append(
                dim.dim_param
            )

        else:

            output_shape.append("?")

    # Determine ONNX input datatype
    tensor_type = (
        inputs[0]
        .type
        .tensor_type
        .elem_type
    )

    onnx_type_map = {

        1: "float32",
        10: "float16",
    }

    input_dtype = onnx_type_map.get(
        tensor_type,
        f"ONNX type {tensor_type}"
    )

    print(
        f"  Input : {input_name}"
    )

    print(
        f"  Shape : {input_shape}"
    )

    print(
        f"  Type  : {input_dtype}"
    )

    print(
        f"  Output: {output_name}"
    )

    print(
        f"  Shape : {output_shape}"
    )

    model_metadata[model_name] = {

        "input_name":
            input_name,

        "output_name":
            output_name,

        "input_shape":
            input_shape,

        "output_shape":
            output_shape,

        "input_dtype":
            input_dtype,
    }


# =============================================================================
# STEP 4Z-C — COLLECT TEST DATA
# =============================================================================

section(
    "STEP 4Z-C — PREPARING TEST DATA"
)


image_extensions = {

    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
}


image_paths = sorted(

    [
        p

        for p in TEST_IMAGES_DIR.iterdir()

        if p.suffix.lower()
        in image_extensions
    ]
)


if len(image_paths) == 0:

    raise RuntimeError(
        f"No test images found:\n"
        f"{TEST_IMAGES_DIR}"
    )


print(
    f"Test images found : "
    f"{len(image_paths)}"
)


if len(image_paths) != EXPECTED_TEST_IMAGES:

    raise RuntimeError(

        f"\nExpected exactly "
        f"{EXPECTED_TEST_IMAGES} "
        f"test images, but found "
        f"{len(image_paths)}.\n"
        f"Stopping to prevent an invalid comparison."
    )


print(
    f"✓ Exact {EXPECTED_TEST_IMAGES}-image "
    f"test set confirmed"
)


# =============================================================================
# STEP 4Z-D — LETTERBOX PREPROCESSING
# =============================================================================

section(
    "STEP 4Z-D — CONFIGURING INPUT PREPROCESSING"
)


def letterbox(
    image,
    new_shape=(608, 608),
    color=(114, 114, 114)
):

    shape = image.shape[:2]

    r = min(

        new_shape[0] / shape[0],
        new_shape[1] / shape[1]
    )

    new_unpad = (

        int(
            round(shape[1] * r)
        ),

        int(
            round(shape[0] * r)
        ),
    )

    dw = (
        new_shape[1]
        - new_unpad[0]
    )

    dh = (
        new_shape[0]
        - new_unpad[1]
    )

    dw /= 2
    dh /= 2

    if shape[::-1] != new_unpad:

        image = cv2.resize(

            image,

            new_unpad,

            interpolation=cv2.INTER_LINEAR
        )

    top = int(
        round(dh - 0.1)
    )

    bottom = int(
        round(dh + 0.1)
    )

    left = int(
        round(dw - 0.1)
    )

    right = int(
        round(dw + 0.1)
    )

    image = cv2.copyMakeBorder(

        image,

        top,
        bottom,
        left,
        right,

        cv2.BORDER_CONSTANT,

        value=color
    )

    return image


def preprocess_image(
    image_path,
    dtype=np.float32
):

    image = cv2.imread(
        str(image_path)
    )

    if image is None:

        raise RuntimeError(
            f"Could not read image:\n"
            f"{image_path}"
        )

    image = cv2.cvtColor(

        image,

        cv2.COLOR_BGR2RGB
    )

    image = letterbox(

        image,

        (
            IMG_SIZE,
            IMG_SIZE
        )
    )

    # Always normalize in FP32 first.
    #
    # This guarantees that FP16 receives the exact same
    # normalized values, only represented as float16.
    #
    image = (
        image.astype(
            np.float32
        )
        / 255.0
    )

    image = image.transpose(
        2,
        0,
        1
    )

    image = np.expand_dims(
        image,
        axis=0
    )

    # Convert ONLY at the final stage according to
    # the ONNX model input requirement.
    image = np.ascontiguousarray(
        image,
        dtype=dtype
    )

    return image


# -------------------------------------------------------------------------
# Example tensors
# -------------------------------------------------------------------------

example_tensor_fp32 = preprocess_image(
    image_paths[0],
    dtype=np.float32
)

example_tensor_fp16 = preprocess_image(
    image_paths[0],
    dtype=np.float16
)


print(
    "Example FP32 tensor:"
)

print(
    f"  Shape : "
    f"{example_tensor_fp32.shape}"
)

print(
    f"  Dtype : "
    f"{example_tensor_fp32.dtype}"
)

print(
    f"  Range : "
    f"{example_tensor_fp32.min():.4f}"
    f" → "
    f"{example_tensor_fp32.max():.4f}"
)


print(
    "\nExample FP16 tensor:"
)

print(
    f"  Shape : "
    f"{example_tensor_fp16.shape}"
)

print(
    f"  Dtype : "
    f"{example_tensor_fp16.dtype}"
)

print(
    f"  Range : "
    f"{example_tensor_fp16.min():.4f}"
    f" → "
    f"{example_tensor_fp16.max():.4f}"
)


# =============================================================================
# STEP 4Z-E — GROUND TRUTH RECONSTRUCTION
# =============================================================================

section(
    "STEP 4Z-E — CONFIGURING GROUND-TRUTH OCR RECONSTRUCTION"
)


def read_ground_truth(
    label_path
):

    if not label_path.exists():

        return {

            "text": "",

            "objects": [],

            "missing": True,
        }


    objects = []


    with open(

        label_path,

        "r",

        encoding="utf-8"

    ) as f:

        lines = [

            line.strip()

            for line in f

            if line.strip()
        ]


    for line in lines:

        parts = line.split()

        if len(parts) < 5:

            continue


        try:

            class_id = int(
                float(parts[0])
            )

            x_center = float(
                parts[1]
            )

            y_center = float(
                parts[2]
            )

            width = float(
                parts[3]
            )

            height = float(
                parts[4]
            )

        except ValueError:

            continue


        if class_id not in CLASS_NAMES:

            continue


        objects.append({

            "class_id":
                class_id,

            "text":
                CLASS_NAMES[class_id],

            "x_center":
                x_center,

            "y_center":
                y_center,

            "width":
                width,

            "height":
                height,
        })


    objects.sort(

        key=lambda x:
        x["x_center"]
    )


    text = "".join(

        obj["text"]

        for obj in objects
    )


    return {

        "text":
            text,

        "objects":
            objects,

        "missing":
            False,
    }


# =============================================================================
# STEP 4Z-F — INITIALIZE CUDA SESSIONS
# =============================================================================

section(
    "STEP 4Z-F — INITIALIZING CUDA SESSIONS"
)


available_providers = (
    ort.get_available_providers()
)


print(
    "Available providers:"
)

for provider in available_providers:

    print(
        f"  - {provider}"
    )


if (
    "CUDAExecutionProvider"
    not in available_providers
):

    raise RuntimeError(
        "CUDAExecutionProvider "
        "is not available."
    )


providers = [

    (
        "CUDAExecutionProvider",

        {
            "device_id": 0
        }
    ),

    "CPUExecutionProvider",
]


sessions = {}


for model_name, model_path in model_paths.items():

    print(
        f"\n→ Loading {model_name} session..."
    )

    session = ort.InferenceSession(

        str(model_path),

        providers=providers
    )

    sessions[model_name] = session

    print(
        f"✓ {model_name} session initialized"
    )

    print(
        "  Active providers:"
    )

    for provider in session.get_providers():

        print(
            f"    - {provider}"
        )


# =============================================================================
# STEP 4Z-G — INPUT / OUTPUT NAMES AND TYPES
# =============================================================================

section(
    "STEP 4Z-G — IDENTIFYING MODEL INPUT / OUTPUT"
)


input_names = {}

output_names = {}

model_input_dtypes = {}


for model_name, session in sessions.items():

    input_meta = (
        session
        .get_inputs()[0]
    )

    output_meta = (
        session
        .get_outputs()[0]
    )

    input_names[model_name] = (
        input_meta.name
    )

    output_names[model_name] = (
        output_meta.name
    )

    input_type = input_meta.type

    if input_type == "tensor(float16)":

        model_input_dtypes[
            model_name
        ] = np.float16

    elif input_type == "tensor(float)":

        model_input_dtypes[
            model_name
        ] = np.float32

    else:

        raise RuntimeError(

            f"Unsupported ONNX input type "
            f"for {model_name}: "
            f"{input_type}"
        )


    print(
        f"{model_name}:"
    )

    print(
        f"  Input : "
        f"{input_names[model_name]}"
    )

    print(
        f"  Type  : "
        f"{input_type}"
    )

    print(
        f"  NumPy : "
        f"{model_input_dtypes[model_name]}"
    )

    print(
        f"  Output: "
        f"{output_names[model_name]}"
    )


print(
    "\n✓ Model-specific input dtypes configured."
)


# =============================================================================
# STEP 4Z-H — OUTPUT DECODER
# =============================================================================

section(
    "STEP 4Z-H — CONFIGURING YOLO OCR OUTPUT DECODER"
)


def decode_predictions(
    output,
    confidence_threshold=CONF_THRESHOLD
):

    output = np.asarray(
        output
    )


    if output.ndim == 3:

        output = output[0]


    if output.ndim != 2:

        raise RuntimeError(

            f"Unexpected output shape: "
            f"{output.shape}"
        )


    if output.shape[1] < 6:

        raise RuntimeError(

            f"Unexpected output dimensions: "
            f"{output.shape}"
        )


    detections = []


    for row in output:

        x1 = float(row[0])
        y1 = float(row[1])
        x2 = float(row[2])
        y2 = float(row[3])

        confidence = float(row[4])

        class_id = int(
            round(
                float(row[5])
            )
        )


        if confidence < confidence_threshold:

            continue


        if class_id not in CLASS_NAMES:

            continue


        x_center = (
            x1 + x2
        ) / 2.0


        detections.append({

            "class_id":
                class_id,

            "text":
                CLASS_NAMES[class_id],

            "confidence":
                confidence,

            "x1":
                x1,

            "y1":
                y1,

            "x2":
                x2,

            "y2":
                y2,

            "x_center":
                x_center,
        })


    detections.sort(

        key=lambda x:
        x["x_center"]
    )


    text = "".join(

        det["text"]

        for det in detections
    )


    return text, detections


# =============================================================================
# STEP 4Z-I — OCR METRICS
# =============================================================================

section(
    "STEP 4Z-I — CONFIGURING OCR METRICS"
)


def levenshtein_distance(
    a,
    b
):

    if a == b:

        return 0


    if len(a) == 0:

        return len(b)


    if len(b) == 0:

        return len(a)


    previous = list(
        range(
            len(b) + 1
        )
    )


    for i, char_a in enumerate(

        a,

        start=1

    ):

        current = [i]


        for j, char_b in enumerate(

            b,

            start=1

        ):

            insertion = (
                current[j - 1]
                + 1
            )

            deletion = (
                previous[j]
                + 1
            )

            substitution = (
                previous[j - 1]
                + (
                    char_a != char_b
                )
            )


            current.append(

                min(
                    insertion,
                    deletion,
                    substitution
                )
            )


        previous = current


    return previous[-1]


def calculate_cer(
    gt,
    prediction
):

    if len(gt) == 0:

        if len(prediction) == 0:

            return 0.0

        return 1.0


    distance = (
        levenshtein_distance(
            gt,
            prediction
        )
    )


    return (
        distance /
        len(gt)
    )


def sequence_character_accuracy(
    gt,
    prediction
):

    if len(gt) == 0:

        return (
            1.0
            if len(prediction) == 0
            else 0.0
        )


    distance = (
        levenshtein_distance(
            gt,
            prediction
        )
    )


    return max(

        0.0,

        1.0 -
        (
            distance /
            len(gt)
        )
    )


# =============================================================================
# STEP 4Z-J — CUDA WARM-UP
# =============================================================================

section(
    "STEP 4Z-J — CUDA WARM-UP"
)


for model_name, session in sessions.items():

    dtype = model_input_dtypes[
        model_name
    ]

    tensor = preprocess_image(

        image_paths[0],

        dtype=dtype
    )


    print(
        f"\n{model_name} warm-up:"
    )

    print(
        f"  Input dtype : "
        f"{tensor.dtype}"
    )

    print(
        f"  Input shape : "
        f"{tensor.shape}"
    )


    for _ in range(
        WARMUP_RUNS
    ):

        session.run(

            None,

            {
                input_names[
                    model_name
                ]:
                    tensor
            }
        )


    print(
        f"✓ {WARMUP_RUNS} "
        f"{model_name} warm-up runs completed"
    )


# =============================================================================
# STEP 4Z-K — FULL 508 IMAGE EVALUATION
# =============================================================================

section(
    "STEP 4Z-K — FULL 508-IMAGE "
    "FP32 vs FP16 vs INT8 EVALUATION"
)


results = []


latencies = {

    "FP32": [],
    "FP16": [],
    "INT8": [],
}


for index, image_path in enumerate(

    image_paths,

    start=1
):


    # -------------------------------------------------------------------------
    # Ground truth
    # -------------------------------------------------------------------------

    label_path = (

        TEST_LABELS_DIR /
        f"{image_path.stem}.txt"
    )


    gt_info = read_ground_truth(
        label_path
    )


    if gt_info["missing"]:

        raise RuntimeError(

            f"Missing label file:\n"
            f"{label_path}"
        )


    gt_text = gt_info["text"]


    # -------------------------------------------------------------------------
    # Prepare model-specific tensors
    # -------------------------------------------------------------------------

    tensors = {}


    for model_name in [

        "FP32",
        "FP16",
        "INT8",
    ]:

        tensors[
            model_name
        ] = preprocess_image(

            image_path,

            dtype=model_input_dtypes[
                model_name
            ]
        )


    # -------------------------------------------------------------------------
    # Store predictions
    # -------------------------------------------------------------------------

    predictions = {}

    detections = {}


    # -------------------------------------------------------------------------
    # Run all three models
    # -------------------------------------------------------------------------

    for model_name in [

        "FP32",
        "FP16",
        "INT8",
    ]:

        session = sessions[
            model_name
        ]

        tensor = tensors[
            model_name
        ]


        # -------------------------------------------------------------
        # Sanity check before inference
        # -------------------------------------------------------------

        expected_dtype = (
            model_input_dtypes[
                model_name
            ]
        )

        if tensor.dtype != np.dtype(
            expected_dtype
        ):

            raise RuntimeError(

                f"Input dtype mismatch for "
                f"{model_name}: "
                f"got {tensor.dtype}, "
                f"expected {expected_dtype}"
            )


        start = time.perf_counter()


        outputs = session.run(

            None,

            {
                input_names[
                    model_name
                ]:
                    tensor
            }
        )


        elapsed = (

            time.perf_counter()
            - start

        ) * 1000.0


        latencies[
            model_name
        ].append(
            elapsed
        )


        text, dets = (
            decode_predictions(
                outputs[0]
            )
        )


        predictions[
            model_name
        ] = text


        detections[
            model_name
        ] = dets


    # -------------------------------------------------------------------------
    # Individual OCR metrics
    # -------------------------------------------------------------------------

    fp32_cer = calculate_cer(

        gt_text,

        predictions[
            "FP32"
        ]
    )


    fp16_cer = calculate_cer(

        gt_text,

        predictions[
            "FP16"
        ]
    )


    int8_cer = calculate_cer(

        gt_text,

        predictions[
            "INT8"
        ]
    )


    fp32_char_acc = (
        sequence_character_accuracy(

            gt_text,

            predictions[
                "FP32"
            ]
        )
    )


    fp16_char_acc = (
        sequence_character_accuracy(

            gt_text,

            predictions[
                "FP16"
            ]
        )
    )


    int8_char_acc = (
        sequence_character_accuracy(

            gt_text,

            predictions[
                "INT8"
            ]
        )
    )


    fp32_exact = (
        gt_text ==
        predictions[
            "FP32"
        ]
    )


    fp16_exact = (
        gt_text ==
        predictions[
            "FP16"
        ]
    )


    int8_exact = (
        gt_text ==
        predictions[
            "INT8"
        ]
    )


    # -------------------------------------------------------------------------
    # Model-to-model agreement
    # -------------------------------------------------------------------------

    fp32_fp16_same = (

        predictions[
            "FP32"
        ]

        ==

        predictions[
            "FP16"
        ]
    )


    fp32_int8_same = (

        predictions[
            "FP32"
        ]

        ==

        predictions[
            "INT8"
        ]
    )


    fp16_int8_same = (

        predictions[
            "FP16"
        ]

        ==

        predictions[
            "INT8"
        ]
    )


    # -------------------------------------------------------------------------
    # Store per-image result
    # -------------------------------------------------------------------------

    results.append({

        "image":
            image_path.name,

        "ground_truth":
            gt_text,


        # Predictions
        "fp32_prediction":
            predictions["FP32"],

        "fp16_prediction":
            predictions["FP16"],

        "int8_prediction":
            predictions["INT8"],


        # Character accuracy
        "fp32_character_accuracy":
            fp32_char_acc,

        "fp16_character_accuracy":
            fp16_char_acc,

        "int8_character_accuracy":
            int8_char_acc,


        # CER
        "fp32_cer":
            fp32_cer,

        "fp16_cer":
            fp16_cer,

        "int8_cer":
            int8_cer,


        # Exact match
        "fp32_exact_match":
            int(fp32_exact),

        "fp16_exact_match":
            int(fp16_exact),

        "int8_exact_match":
            int(int8_exact),


        # Agreement
        "fp32_fp16_same":
            int(fp32_fp16_same),

        "fp32_int8_same":
            int(fp32_int8_same),

        "fp16_int8_same":
            int(fp16_int8_same),


        # Empty predictions
        "fp32_empty":
            int(
                len(
                    predictions[
                        "FP32"
                    ]
                ) == 0
            ),

        "fp16_empty":
            int(
                len(
                    predictions[
                        "FP16"
                    ]
                ) == 0
            ),

        "int8_empty":
            int(
                len(
                    predictions[
                        "INT8"
                    ]
                ) == 0
            ),


        # Detection counts
        "fp32_detection_count":
            len(
                detections[
                    "FP32"
                ]
            ),

        "fp16_detection_count":
            len(
                detections[
                    "FP16"
                ]
            ),

        "int8_detection_count":
            len(
                detections[
                    "INT8"
                ]
            ),


        # Latencies
        "fp32_latency_ms":
            latencies[
                "FP32"
            ][-1],

        "fp16_latency_ms":
            latencies[
                "FP16"
            ][-1],

        "int8_latency_ms":
            latencies[
                "INT8"
            ][-1],


        # Ground-truth / prediction lengths
        "gt_length":
            len(gt_text),

        "fp32_length":
            len(
                predictions[
                    "FP32"
                ]
            ),

        "fp16_length":
            len(
                predictions[
                    "FP16"
                ]
            ),

        "int8_length":
            len(
                predictions[
                    "INT8"
                ]
            ),
    })


    if (

        index % 50 == 0

        or

        index == len(
            image_paths
        )

    ):

        print(
            f"Processed "
            f"{index}/{len(image_paths)}"
        )


# =============================================================================
# STEP 4Z-L — BUILD DATAFRAME
# =============================================================================

section(
    "STEP 4Z-L — BUILDING RESULTS TABLE"
)


results_df = pd.DataFrame(
    results
)


results_df.to_csv(

    PER_IMAGE_CSV,

    index=False
)


print(
    "✓ Per-image results saved:"
)

print(
    f"  {PER_IMAGE_CSV}"
)


# =============================================================================
# STEP 4Z-M — AGGREGATE OCR QUALITY
# =============================================================================

section(
    "STEP 4Z-M — OCR QUALITY METRICS"
)


def aggregate_metrics(
    model_name
):

    prefix = (
        model_name.lower()
    )

    return {

        "character_accuracy":
            float(
                results_df[
                    f"{prefix}_character_accuracy"
                ].mean()
            ),

        "cer":
            float(
                results_df[
                    f"{prefix}_cer"
                ].mean()
            ),

        "exact_match":
            float(
                results_df[
                    f"{prefix}_exact_match"
                ].mean()
            ),

        "empty_predictions":
            int(
                results_df[
                    f"{prefix}_empty"
                ].sum()
            ),
    }


ocr_metrics = {

    "FP32":
        aggregate_metrics(
            "FP32"
        ),

    "FP16":
        aggregate_metrics(
            "FP16"
        ),

    "INT8":
        aggregate_metrics(
            "INT8"
        ),
}


for model_name in [

    "FP32",
    "FP16",
    "INT8",
]:

    metrics = ocr_metrics[
        model_name
    ]

    print(
        f"\n{model_name} OCR:"
    )

    print(
        f"  Character Accuracy : "
        f"{metrics['character_accuracy'] * 100:.4f}%"
    )

    print(
        f"  CER                : "
        f"{metrics['cer']:.6f}"
    )

    print(
        f"  Exact Match        : "
        f"{metrics['exact_match'] * 100:.4f}%"
    )

    print(
        f"  Empty predictions  : "
        f"{metrics['empty_predictions']}"
    )


# =============================================================================
# STEP 4Z-N — MODEL AGREEMENT
# =============================================================================

section(
    "STEP 4Z-N — MODEL PREDICTION AGREEMENT"
)


agreement_pairs = {

    "FP32_vs_FP16":
        "fp32_fp16_same",

    "FP32_vs_INT8":
        "fp32_int8_same",

    "FP16_vs_INT8":
        "fp16_int8_same",
}


agreement_results = {}


for pair_name, column in agreement_pairs.items():

    matching = int(
        results_df[column].sum()
    )

    total = len(
        results_df
    )

    agreement = (
        matching /
        total
    )

    mismatch = (
        1.0 -
        agreement
    )

    agreement_results[pair_name] = {

        "matching_predictions":
            matching,

        "total_predictions":
            total,

        "agreement":
            float(
                agreement
            ),

        "mismatch_rate":
            float(
                mismatch
            ),
    }

    print(
        f"\n{pair_name}:"
    )

    print(
        f"  Matching : "
        f"{matching}/{total}"
    )

    print(
        f"  Agreement: "
        f"{agreement * 100:.4f}%"
    )

    print(
        f"  Mismatch : "
        f"{mismatch * 100:.4f}%"
    )


# =============================================================================
# STEP 4Z-O — QUALITY CHANGE RELATIVE TO FP32
# =============================================================================

section(
    "STEP 4Z-O — QUALITY CHANGE RELATIVE TO FP32"
)


quality_changes = {}


for model_name in [

    "FP16",
    "INT8",
]:

    char_delta = (

        ocr_metrics[
            model_name
        ][
            "character_accuracy"
        ]

        -

        ocr_metrics[
            "FP32"
        ][
            "character_accuracy"
        ]
    )


    cer_delta = (

        ocr_metrics[
            model_name
        ][
            "cer"
        ]

        -

        ocr_metrics[
            "FP32"
        ][
            "cer"
        ]
    )


    exact_delta = (

        ocr_metrics[
            model_name
        ][
            "exact_match"
        ]

        -

        ocr_metrics[
            "FP32"
        ][
            "exact_match"
        ]
    )


    quality_changes[
        model_name
    ] = {

        "character_accuracy_percentage_points":
            float(
                char_delta * 100
            ),

        "cer_change":
            float(
                cer_delta
            ),

        "exact_match_percentage_points":
            float(
                exact_delta * 100
            ),
    }


    print(
        f"\nFP32 → {model_name}:"
    )

    print(
        f"  Character Accuracy : "
        f"{char_delta * 100:+.4f} pp"
    )

    print(
        f"  CER                : "
        f"{cer_delta:+.6f}"
    )

    print(
        f"  Exact Match        : "
        f"{exact_delta * 100:+.4f} pp"
    )


# =============================================================================
# STEP 4Z-P — PERFORMANCE STATISTICS
# =============================================================================

section(
    "STEP 4Z-P — PERFORMANCE STATISTICS"
)


def latency_statistics(
    values
):

    values = np.asarray(
        values,
        dtype=np.float64
    )


    mean_ms = float(
        np.mean(values)
    )


    return {

        "mean_ms":
            mean_ms,

        "median_ms":
            float(
                np.median(values)
            ),

        "p95_ms":
            float(
                np.percentile(
                    values,
                    95
                )
            ),

        "min_ms":
            float(
                np.min(values)
            ),

        "max_ms":
            float(
                np.max(values)
            ),

        "fps":
            float(
                1000.0 /
                mean_ms
            ),
    }


performance = {}


for model_name in [

    "FP32",
    "FP16",
    "INT8",
]:

    performance[
        model_name
    ] = latency_statistics(

        latencies[
            model_name
        ]
    )


    print(
        f"\n{model_name}:"
    )

    for key, value in (
        performance[
            model_name
        ].items()
    ):

        if key == "fps":

            print(
                f"  FPS            : "
                f"{value:.3f}"
            )

        else:

            print(
                f"  {key:<14}: "
                f"{value:.3f} ms"
            )


# =============================================================================
# STEP 4Z-Q — PERFORMANCE RELATIVE TO FP32
# =============================================================================

section(
    "STEP 4Z-Q — PERFORMANCE CHANGE RELATIVE TO FP32"
)


performance_changes = {}


for model_name in [

    "FP16",
    "INT8",
]:

    speedup = (

        performance[
            "FP32"
        ][
            "mean_ms"
        ]

        /

        performance[
            model_name
        ][
            "mean_ms"
        ]
    )


    latency_change = (

        (

            performance[
                model_name
            ][
                "mean_ms"
            ]

            -

            performance[
                "FP32"
            ][
                "mean_ms"
            ]

        )

        /

        performance[
            "FP32"
        ][
            "mean_ms"
        ]

    ) * 100.0


    fps_change = (

        (

            performance[
                model_name
            ][
                "fps"
            ]

            -

            performance[
                "FP32"
            ][
                "fps"
            ]

        )

        /

        performance[
            "FP32"
        ][
            "fps"
        ]

    ) * 100.0


    performance_changes[
        model_name
    ] = {

        "speedup":
            float(
                speedup
            ),

        "latency_change_percent":
            float(
                latency_change
            ),

        "fps_change_percent":
            float(
                fps_change
            ),
    }


    print(
        f"\nFP32 → {model_name}:"
    )

    print(
        f"  Speedup       : "
        f"{speedup:.3f}x"
    )

    print(
        f"  Latency change: "
        f"{latency_change:+.2f}%"
    )

    print(
        f"  FPS change    : "
        f"{fps_change:+.2f}%"
    )


# =============================================================================
# STEP 4Z-R — MODEL SIZE
# =============================================================================

section(
    "STEP 4Z-R — MODEL SIZE COMPARISON"
)


model_sizes = {

    "FP32":
        file_size_mb(
            FP32_MODEL
        ),

    "FP16":
        file_size_mb(
            FP16_MODEL
        ),

    "INT8":
        file_size_mb(
            INT8_MODEL
        ),
}


fp32_size = model_sizes[
    "FP32"
]


size_changes = {}


for model_name in [

    "FP16",
    "INT8",
]:

    reduction_mb = (

        fp32_size

        -

        model_sizes[
            model_name
        ]
    )


    reduction_percent = (

        reduction_mb /

        fp32_size

    ) * 100.0


    size_changes[
        model_name
    ] = {

        "reduction_mb":
            float(
                reduction_mb
            ),

        "reduction_percent":
            float(
                reduction_percent
            ),
    }


for model_name in [

    "FP32",
    "FP16",
    "INT8",
]:

    print(
        f"{model_name} : "
        f"{model_sizes[model_name]:.2f} MB"
    )


print(
    f"\nFP32 → FP16 reduction : "
    f"{size_changes['FP16']['reduction_mb']:.2f} MB "
    f"({size_changes['FP16']['reduction_percent']:.2f}%)"
)


print(
    f"FP32 → INT8 reduction : "
    f"{size_changes['INT8']['reduction_mb']:.2f} MB "
    f"({size_changes['INT8']['reduction_percent']:.2f}%)"
)


# =============================================================================
# STEP 4Z-S — FINAL COMPARISON TABLE
# =============================================================================

section(
    "STEP 4Z-S — FINAL FP32 vs FP16 vs INT8 COMPARISON"
)


comparison_df = pd.DataFrame({

    "Metric": [

        "Character Accuracy",
        "CER",
        "Exact Match",

        "Mean latency (ms)",
        "Median latency (ms)",
        "P95 latency (ms)",
        "Minimum latency (ms)",
        "Maximum latency (ms)",

        "FPS",

        "Model size (MB)",
    ],


    "FP32": [

        f"{ocr_metrics['FP32']['character_accuracy'] * 100:.4f}%",
        f"{ocr_metrics['FP32']['cer']:.6f}",
        f"{ocr_metrics['FP32']['exact_match'] * 100:.4f}%",

        f"{performance['FP32']['mean_ms']:.3f}",
        f"{performance['FP32']['median_ms']:.3f}",
        f"{performance['FP32']['p95_ms']:.3f}",
        f"{performance['FP32']['min_ms']:.3f}",
        f"{performance['FP32']['max_ms']:.3f}",

        f"{performance['FP32']['fps']:.3f}",

        f"{model_sizes['FP32']:.2f}",
    ],


    "FP16": [

        f"{ocr_metrics['FP16']['character_accuracy'] * 100:.4f}%",
        f"{ocr_metrics['FP16']['cer']:.6f}",
        f"{ocr_metrics['FP16']['exact_match'] * 100:.4f}%",

        f"{performance['FP16']['mean_ms']:.3f}",
        f"{performance['FP16']['median_ms']:.3f}",
        f"{performance['FP16']['p95_ms']:.3f}",
        f"{performance['FP16']['min_ms']:.3f}",
        f"{performance['FP16']['max_ms']:.3f}",

        f"{performance['FP16']['fps']:.3f}",

        f"{model_sizes['FP16']:.2f}",
    ],


    "INT8": [

        f"{ocr_metrics['INT8']['character_accuracy'] * 100:.4f}%",
        f"{ocr_metrics['INT8']['cer']:.6f}",
        f"{ocr_metrics['INT8']['exact_match'] * 100:.4f}%",

        f"{performance['INT8']['mean_ms']:.3f}",
        f"{performance['INT8']['median_ms']:.3f}",
        f"{performance['INT8']['p95_ms']:.3f}",
        f"{performance['INT8']['min_ms']:.3f}",
        f"{performance['INT8']['max_ms']:.3f}",

        f"{performance['INT8']['fps']:.3f}",

        f"{model_sizes['INT8']:.2f}",
    ],
})


print(
    comparison_df.to_string(
        index=False
    )
)


# =============================================================================
# STEP 4Z-T — SAVE JSON SUMMARY
# =============================================================================

section(
    "STEP 4Z-T — SAVING FINAL JSON SUMMARY"
)


summary = {

    "step":
        "STEP 4Z — YOLO11 OCR FP32 vs FP16 vs INT8 evaluation",


    "dataset": {

        "test_images":
            len(results_df),

        "test_images_directory":
            str(TEST_IMAGES_DIR),

        "test_labels_directory":
            str(TEST_LABELS_DIR),
    },


    "input": {

        "resolution":
            f"{IMG_SIZE}x{IMG_SIZE}",

        "confidence_threshold":
            CONF_THRESHOLD,

        "model_input_dtypes":
            {

                model_name:
                    str(
                        np.dtype(
                            model_input_dtypes[
                                model_name
                            ]
                        )
                    )

                for model_name in [

                    "FP32",
                    "FP16",
                    "INT8",
                ]
            },
    },


    "models": {

        model_name: {

            "path":
                str(
                    model_paths[
                        model_name
                    ]
                ),

            "size_mb":
                model_sizes[
                    model_name
                ],

            "input_name":
                model_metadata[
                    model_name
                ][
                    "input_name"
                ],

            "output_name":
                model_metadata[
                    model_name
                ][
                    "output_name"
                ],

            "input_shape":
                model_metadata[
                    model_name
                ][
                    "input_shape"
                ],

            "output_shape":
                model_metadata[
                    model_name
                ][
                    "output_shape"
                ],

            "input_dtype":
                model_metadata[
                    model_name
                ][
                    "input_dtype"
                ],
        }

        for model_name in [

            "FP32",
            "FP16",
            "INT8",
        ]
    },


    "ocr_quality":
        ocr_metrics,


    "quality_change_relative_to_fp32":
        quality_changes,


    "model_agreement":
        agreement_results,


    "performance":
        performance,


    "performance_change_relative_to_fp32":
        performance_changes,


    "model_size_change":
        size_changes,


    "quantization": {

        "format":
            "QDQ",

        "weights":
            "INT8",

        "activations":
            "UINT8",

        "per_channel":
            True,

        "quantized_operations":
            "Conv only",

        "calibration_images":
            200,

        "protected_tail_hops":
            8,
    },


    "files": {

        "per_image_results":
            str(
                PER_IMAGE_CSV
            ),

        "summary":
            str(
                SUMMARY_JSON
            ),
    },
}


with open(

    SUMMARY_JSON,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        summary,

        f,

        indent=2
    )


print(
    "✓ JSON summary saved:"
)

print(
    f"  {SUMMARY_JSON}"
)


# =============================================================================
# STEP 4Z-U — FINAL RESULT
# =============================================================================

section(
    "STEP 4Z-U — FINAL RESULT"
)


print("""
================================================================================
YOLO11 OCR — FP32 vs FP16 vs INT8
FINAL GROUND-TRUTH EVALUATION
================================================================================
""")


print("TEST SET")
print("--------")

print(
    f"Images                 : "
    f"{len(results_df)}"
)

print(
    f"Input                  : "
    f"{IMG_SIZE} × {IMG_SIZE}"
)


print("\nINPUT DATA TYPES")
print("----------------")


for model_name in [

    "FP32",
    "FP16",
    "INT8",
]:

    print(
        f"{model_name:<6}: "
        f"{model_input_dtypes[model_name]}"
    )


print("\nOCR QUALITY")
print("----------")


for model_name in [

    "FP32",
    "FP16",
    "INT8",
]:

    metrics = ocr_metrics[
        model_name
    ]

    print(
        f"\n{model_name}:"
    )

    print(
        f"  Character Accuracy : "
        f"{metrics['character_accuracy'] * 100:.4f}%"
    )

    print(
        f"  CER                : "
        f"{metrics['cer']:.6f}"
    )

    print(
        f"  Exact Match        : "
        f"{metrics['exact_match'] * 100:.4f}%"
    )


print("\nQUALITY CHANGE RELATIVE TO FP32")
print("--------------------------------")


for model_name in [

    "FP16",
    "INT8",
]:

    change = quality_changes[
        model_name
    ]

    print(
        f"\nFP32 → {model_name}:"
    )

    print(
        f"  Character Accuracy : "
        f"{change['character_accuracy_percentage_points']:+.4f} pp"
    )

    print(
        f"  CER change         : "
        f"{change['cer_change']:+.6f}"
    )

    print(
        f"  Exact Match        : "
        f"{change['exact_match_percentage_points']:+.4f} pp"
    )


print("\nMODEL AGREEMENT")
print("---------------")


for pair_name, values in (

    agreement_results.items()

):

    print(
        f"{pair_name:<20}: "
        f"{values['agreement'] * 100:.4f}%"
    )


print("\nPERFORMANCE")
print("-----------")


for model_name in [

    "FP32",
    "FP16",
    "INT8",
]:

    perf = performance[
        model_name
    ]

    print(
        f"\n{model_name}:"
    )

    print(
        f"  Mean latency : "
        f"{perf['mean_ms']:.3f} ms"
    )

    print(
        f"  Median      : "
        f"{perf['median_ms']:.3f} ms"
    )

    print(
        f"  P95         : "
        f"{perf['p95_ms']:.3f} ms"
    )

    print(
        f"  Min         : "
        f"{perf['min_ms']:.3f} ms"
    )

    print(
        f"  Max         : "
        f"{perf['max_ms']:.3f} ms"
    )

    print(
        f"  FPS         : "
        f"{perf['fps']:.3f}"
    )


print("\nMODEL SIZE")
print("----------")


for model_name in [

    "FP32",
    "FP16",
    "INT8",
]:

    print(
        f"{model_name:<6}: "
        f"{model_sizes[model_name]:.2f} MB"
    )


print("\nFILES")
print("-----")


print(
    f"Per-image CSV:\n"
    f"  {PER_IMAGE_CSV}"
)


print(
    f"\nSummary JSON:\n"
    f"  {SUMMARY_JSON}"
)


print("""
================================================================================
STEP 4Z — COMPLETE
================================================================================
""")


gc.collect()


print(
    "✓ Evaluation completed."
)

print(
    "✓ FP32 model untouched."
)

print(
    "✓ FP16 model untouched."
)

print(
    "✓ INT8 model untouched."
)

print(
    "✓ Test dataset untouched."
)

print(
    "✓ No quantization performed."
)

print(
    "✓ Ground-truth OCR metrics calculated."
)

print(
    "✓ FP32/FP16/INT8 agreement calculated."
)

print(
    "✓ Performance metrics calculated."
)

print(
    "✓ Model-size comparison calculated."
)

print(
    "✓ Model-specific input dtypes verified."
)

print(
    "✓ Results saved."
)



STEP 4Z-A — VERIFYING EXISTING ARTIFACTS
✓ /content/drive/MyDrive/Stage_PFA/OCR_OPTIMIZATION/02_onnx_fp32/yolo11_ocr_fp32.onnx
✓ /content/drive/MyDrive/Stage_PFA/OCR_OPTIMIZATION/03_onnx_fp16/yolo11_ocr_fp16.onnx
✓ /content/drive/MyDrive/Stage_PFA/OCR_OPTIMIZATION/04_onnx_int8/yolo11_ocr_int8.onnx
✓ /content/drive/MyDrive/Stage_PFA/datasets/test/images
✓ /content/drive/MyDrive/Stage_PFA/datasets/test/labels

✓ All required artifacts found

Model sizes:
  FP32 : 10.14 MB
  FP16 : 5.13 MB
  INT8 : 3.03 MB


STEP 4Z-B — VERIFYING ONNX MODELS

FP32 model:
  ✓ ONNX checker passed
  Input : images
  Shape : [1, 3, 608, 608]
  Type  : float32
  Output: output0
  Shape : [1, 300, 6]

FP16 model:
  ✓ ONNX checker passed
  Input : images
  Shape : [1, 3, 608, 608]
  Type  : float16
  Output: output0
  Shape : [1, 300, 6]

INT8 model:
  ✓ ONNX checker passed
  Input : images
  Shape : [1, 3, 608, 608]
  Type  : float32
  Output: output0
  Shape : [1, 300, 6]


STEP 4Z-C — PREPARING TEST DATA
Te

In [3]:
# =============================================================================
# STEP 5A — PREPARE RASPBERRY PI DEPLOYMENT PACKAGE
# =============================================================================
#
# PURPOSE
# -------
# Prepare a clean, self-contained deployment package for Raspberry Pi
# evaluation.
#
# THIS SCRIPT DOES NOT:
#   - retrain models
#   - quantize models
#   - modify ONNX models
#   - modify the dataset
#
# MODELS
# -------
#   FP32 : yolo11_ocr_fp32.onnx
#   FP16 : yolo11_ocr_fp16.onnx
#   INT8 : yolo11_ocr_int8.onnx
#
# DATASET
# -------
#   508 test images
#   508 YOLO-format label files
#
# OUTPUT
# -------
#   /content/drive/MyDrive/Stage_PFA/
#       OCR_OPTIMIZATION/
#           07_pi_deployment/
#
# =============================================================================


import os
import json
import shutil
import hashlib
from pathlib import Path

import onnx


# =============================================================================
# CONFIGURATION
# =============================================================================

BASE_DIR = Path(
    "/content/drive/MyDrive/Stage_PFA/"
)

OCR_OPT_DIR = (
    BASE_DIR /
    "OCR_OPTIMIZATION"
)

DEPLOYMENT_DIR = (
    OCR_OPT_DIR /
    "07_pi_deployment"
)

MODELS_DIR = (
    DEPLOYMENT_DIR /
    "models"
)

TEST_DIR = (
    DEPLOYMENT_DIR /
    "test"
)

TEST_IMAGES_DIR = (
    TEST_DIR /
    "images"
)

TEST_LABELS_DIR = (
    TEST_DIR /
    "labels"
)


# Original model locations

FP32_SOURCE = (
    OCR_OPT_DIR /
    "02_onnx_fp32" /
    "yolo11_ocr_fp32.onnx"
)

FP16_SOURCE = (
    OCR_OPT_DIR /
    "03_onnx_fp16" /
    "yolo11_ocr_fp16.onnx"
)

INT8_SOURCE = (
    OCR_OPT_DIR /
    "04_onnx_int8" /
    "yolo11_ocr_int8.onnx"
)


# Original dataset locations

SOURCE_IMAGES_DIR = (
    BASE_DIR /
    "datasets" /
    "test" /
    "images"
)

SOURCE_LABELS_DIR = (
    BASE_DIR /
    "datasets" /
    "test" /
    "labels"
)


# Output manifest

MANIFEST_PATH = (
    DEPLOYMENT_DIR /
    "deployment_manifest.json"
)


EXPECTED_TEST_IMAGES = 508

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
}


# =============================================================================
# HELPERS
# =============================================================================

def section(title):

    print("\n")
    print("=" * 100)
    print(title)
    print("=" * 100)


def file_size_mb(path):

    return (
        path.stat().st_size /
        (1024 ** 2)
    )


def sha256_file(path):

    sha256 = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            chunk = f.read(1024 * 1024)

            if not chunk:
                break

            sha256.update(chunk)

    return sha256.hexdigest()


def get_onnx_metadata(path):

    model = onnx.load(
        str(path)
    )

    onnx.checker.check_model(
        model
    )

    input_info = model.graph.input[0]
    output_info = model.graph.output[0]

    input_name = input_info.name
    output_name = output_info.name

    input_shape = []

    for dim in (
        input_info
        .type
        .tensor_type
        .shape
        .dim
    ):

        if dim.dim_value:

            input_shape.append(
                int(dim.dim_value)
            )

        elif dim.dim_param:

            input_shape.append(
                dim.dim_param
            )

        else:

            input_shape.append("?")

    output_shape = []

    for dim in (
        output_info
        .type
        .tensor_type
        .shape
        .dim
    ):

        if dim.dim_value:

            output_shape.append(
                int(dim.dim_value)
            )

        elif dim.dim_param:

            output_shape.append(
                dim.dim_param
            )

        else:

            output_shape.append("?")

    tensor_type = (
        input_info
        .type
        .tensor_type
        .elem_type
    )

    # ONNX TensorProto:
    # 1 = FLOAT
    # 10 = FLOAT16

    if tensor_type == 1:

        input_dtype = "float32"

    elif tensor_type == 10:

        input_dtype = "float16"

    else:

        input_dtype = str(
            tensor_type
        )

    return {

        "input_name":
            input_name,

        "input_shape":
            input_shape,

        "input_dtype":
            input_dtype,

        "output_name":
            output_name,

        "output_shape":
            output_shape,
    }


# =============================================================================
# STEP 5A-1 — VERIFY SOURCE FILES
# =============================================================================

section(
    "STEP 5A-1 — VERIFYING SOURCE ARTIFACTS"
)


required_sources = [

    FP32_SOURCE,
    FP16_SOURCE,
    INT8_SOURCE,

    SOURCE_IMAGES_DIR,
    SOURCE_LABELS_DIR,
]


for path in required_sources:

    if not path.exists():

        raise FileNotFoundError(
            f"\nRequired source does not exist:\n{path}"
        )

    print(
        f"✓ {path}"
    )


# =============================================================================
# STEP 5A-2 — VERIFY TEST DATASET
# =============================================================================

section(
    "STEP 5A-2 — VERIFYING TEST DATASET"
)


source_images = sorted(

    [
        p

        for p in SOURCE_IMAGES_DIR.iterdir()

        if p.is_file()
        and p.suffix.lower()
        in IMAGE_EXTENSIONS
    ]
)


source_labels = sorted(

    [
        p

        for p in SOURCE_LABELS_DIR.iterdir()

        if p.is_file()
        and p.suffix.lower() == ".txt"
    ]
)


print(
    f"Images : {len(source_images)}"
)

print(
    f"Labels : {len(source_labels)}"
)


if len(source_images) != EXPECTED_TEST_IMAGES:

    raise RuntimeError(
        f"Expected {EXPECTED_TEST_IMAGES} images, "
        f"found {len(source_images)}."
    )


if len(source_labels) != EXPECTED_TEST_IMAGES:

    raise RuntimeError(
        f"Expected {EXPECTED_TEST_IMAGES} labels, "
        f"found {len(source_labels)}."
    )


image_stems = {
    p.stem
    for p in source_images
}

label_stems = {
    p.stem
    for p in source_labels
}


missing_labels = (
    image_stems -
    label_stems
)

extra_labels = (
    label_stems -
    image_stems
)


if missing_labels:

    raise RuntimeError(
        "Missing labels for:\n"
        + "\n".join(
            sorted(missing_labels)
        )
    )


if extra_labels:

    raise RuntimeError(
        "Labels without corresponding images:\n"
        + "\n".join(
            sorted(extra_labels)
        )
    )


print(
    "✓ Exact 508-image / 508-label dataset confirmed"
)


# =============================================================================
# STEP 5A-3 — CREATE DEPLOYMENT DIRECTORIES
# =============================================================================

section(
    "STEP 5A-3 — CREATING DEPLOYMENT DIRECTORY"
)


if DEPLOYMENT_DIR.exists():

    print(
        "Existing deployment directory found."
    )

    print(
        "It will be replaced."
    )

    shutil.rmtree(
        DEPLOYMENT_DIR
    )


MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TEST_IMAGES_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TEST_LABELS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print(
    f"✓ Created:\n{DEPLOYMENT_DIR}"
)


# =============================================================================
# STEP 5A-4 — COPY MODELS
# =============================================================================

section(
    "STEP 5A-4 — COPYING ONNX MODELS"
)


model_sources = {

    "FP32":
        FP32_SOURCE,

    "FP16":
        FP16_SOURCE,

    "INT8":
        INT8_SOURCE,
}


model_destinations = {

    "FP32":
        MODELS_DIR /
        "yolo11_ocr_fp32.onnx",

    "FP16":
        MODELS_DIR /
        "yolo11_ocr_fp16.onnx",

    "INT8":
        MODELS_DIR /
        "yolo11_ocr_int8.onnx",
}


for model_name in [

    "FP32",
    "FP16",
    "INT8",
]:

    source = model_sources[
        model_name
    ]

    destination = model_destinations[
        model_name
    ]

    print(
        f"\n{model_name}:"
    )

    print(
        f"  Source      : {source}"
    )

    print(
        f"  Destination : {destination}"
    )

    shutil.copy2(
        source,
        destination
    )

    print(
        f"  Size        : "
        f"{file_size_mb(destination):.2f} MB"
    )

    print(
        "  ✓ Copied"
    )


# =============================================================================
# STEP 5A-5 — COPY TEST DATASET
# =============================================================================

section(
    "STEP 5A-5 — COPYING 508-IMAGE TEST DATASET"
)


print(
    "Copying test images..."
)


for index, source in enumerate(
    source_images,
    start=1
):

    destination = (
        TEST_IMAGES_DIR /
        source.name
    )

    shutil.copy2(
        source,
        destination
    )

    if (
        index % 100 == 0
        or index == len(source_images)
    ):

        print(
            f"  Images: "
            f"{index}/{len(source_images)}"
        )


print(
    "Copying test labels..."
)


for index, source in enumerate(
    source_labels,
    start=1
):

    destination = (
        TEST_LABELS_DIR /
        source.name
    )

    shutil.copy2(
        source,
        destination
    )

    if (
        index % 100 == 0
        or index == len(source_labels)
    ):

        print(
            f"  Labels: "
            f"{index}/{len(source_labels)}"
        )


print(
    "✓ Test dataset copied"
)


# =============================================================================
# STEP 5A-6 — VERIFY COPIED DATASET
# =============================================================================

section(
    "STEP 5A-6 — VERIFYING DEPLOYMENT DATASET"
)


deployment_images = sorted(

    [
        p

        for p in TEST_IMAGES_DIR.iterdir()

        if p.is_file()
        and p.suffix.lower()
        in IMAGE_EXTENSIONS
    ]
)


deployment_labels = sorted(

    [
        p

        for p in TEST_LABELS_DIR.iterdir()

        if p.is_file()
        and p.suffix.lower() == ".txt"
    ]
)


if len(deployment_images) != EXPECTED_TEST_IMAGES:

    raise RuntimeError(
        "Deployment image count incorrect."
    )


if len(deployment_labels) != EXPECTED_TEST_IMAGES:

    raise RuntimeError(
        "Deployment label count incorrect."
    )


print(
    f"✓ Deployment images : "
    f"{len(deployment_images)}"
)

print(
    f"✓ Deployment labels : "
    f"{len(deployment_labels)}"
)


# =============================================================================
# STEP 5A-7 — VERIFY ONNX MODELS
# =============================================================================

section(
    "STEP 5A-7 — VERIFYING COPIED ONNX MODELS"
)


models_manifest = {}


for model_name in [

    "FP32",
    "FP16",
    "INT8",
]:

    model_path = (
        model_destinations[
            model_name
        ]
    )

    metadata = get_onnx_metadata(
        model_path
    )

    checksum = sha256_file(
        model_path
    )

    models_manifest[
        model_name
    ] = {

        "filename":
            model_path.name,

        "relative_path":
            str(
                model_path.relative_to(
                    DEPLOYMENT_DIR
                )
            ),

        "size_mb":
            file_size_mb(
                model_path
            ),

        "sha256":
            checksum,

        **metadata,
    }

    print(
        f"\n{model_name}:"
    )

    print(
        f"  Input dtype : "
        f"{metadata['input_dtype']}"
    )

    print(
        f"  Input shape : "
        f"{metadata['input_shape']}"
    )

    print(
        f"  Output shape: "
        f"{metadata['output_shape']}"
    )

    print(
        f"  Size        : "
        f"{file_size_mb(model_path):.2f} MB"
    )

    print(
        f"  SHA256      : "
        f"{checksum}"
    )

    print(
        "  ✓ ONNX verified"
    )


# =============================================================================
# STEP 5A-8 — CREATE MANIFEST
# =============================================================================

section(
    "STEP 5A-8 — CREATING DEPLOYMENT MANIFEST"
)


manifest = {

    "deployment":

        "STEP 5 — Raspberry Pi YOLO11 OCR deployment",

    "created_for":

        "Raspberry Pi CPU evaluation",

    "dataset": {

        "test_images":
            EXPECTED_TEST_IMAGES,

        "test_labels":
            EXPECTED_TEST_IMAGES,

        "image_directory":
            "test/images",

        "label_directory":
            "test/labels",
    },

    "input": {

        "resolution":
            "608x608",

        "preprocessing":
            "RGB + letterbox + normalization /255.0",

        "confidence_threshold":
            0.25,
    },

    "models":
        models_manifest,

    "source": {

        "fp32":
            str(FP32_SOURCE),

        "fp16":
            str(FP16_SOURCE),

        "int8":
            str(INT8_SOURCE),
    },

    "evaluation_reference": {

        "step":
            "STEP 4Z",

        "test_images":
            508,

        "fp32_character_accuracy":
            93.5977,

        "fp16_character_accuracy":
            93.6687,

        "int8_character_accuracy":
            93.0037,

        "fp32_cer":
            0.110775,

        "fp16_cer":
            0.110065,

        "int8_cer":
            0.120160,

        "fp32_exact_match":
            84.4488,

        "fp16_exact_match":
            84.8425,

        "int8_exact_match":
            83.0709,
    },

    "integrity":

        "Models copied without modification. "
        "Dataset copied without modification.",
}


with open(
    MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2
    )


print(
    f"✓ Manifest saved:\n"
    f"{MANIFEST_PATH}"
)


# =============================================================================
# STEP 5A-9 — FINAL VERIFICATION
# =============================================================================

section(
    "STEP 5A-9 — FINAL DEPLOYMENT PACKAGE VERIFICATION"
)


print(
    "\nDeployment package:"
)

print(
    f"  {DEPLOYMENT_DIR}"
)

print(
    "\nModels:"
)

for model_name in [

    "FP32",
    "FP16",
    "INT8",
]:

    path = model_destinations[
        model_name
    ]

    print(
        f"  ✓ {path.name} "
        f"({file_size_mb(path):.2f} MB)"
    )


print(
    "\nDataset:"
)

print(
    f"  ✓ Images : "
    f"{len(deployment_images)}"
)

print(
    f"  ✓ Labels : "
    f"{len(deployment_labels)}"
)

print(
    "\nManifest:"
)

print(
    f"  ✓ {MANIFEST_PATH.name}"
)


print("""
================================================================================
STEP 5A — DEPLOYMENT PACKAGE READY
================================================================================

The package contains:

  ✓ FP32 ONNX
  ✓ FP16 ONNX
  ✓ INT8 ONNX
  ✓ Exact 508-image test set
  ✓ Exact 508 YOLO label files
  ✓ ONNX metadata
  ✓ SHA256 checksums
  ✓ Step 4Z reference metrics

No model was retrained.
No model was quantized.
No model was modified.
No dataset was modified.

The deployment package is ready to transfer to the Raspberry Pi.
================================================================================
""")



STEP 5A-1 — VERIFYING SOURCE ARTIFACTS
✓ /content/drive/MyDrive/Stage_PFA/OCR_OPTIMIZATION/02_onnx_fp32/yolo11_ocr_fp32.onnx
✓ /content/drive/MyDrive/Stage_PFA/OCR_OPTIMIZATION/03_onnx_fp16/yolo11_ocr_fp16.onnx
✓ /content/drive/MyDrive/Stage_PFA/OCR_OPTIMIZATION/04_onnx_int8/yolo11_ocr_int8.onnx
✓ /content/drive/MyDrive/Stage_PFA/datasets/test/images
✓ /content/drive/MyDrive/Stage_PFA/datasets/test/labels


STEP 5A-2 — VERIFYING TEST DATASET
Images : 508
Labels : 508
✓ Exact 508-image / 508-label dataset confirmed


STEP 5A-3 — CREATING DEPLOYMENT DIRECTORY
✓ Created:
/content/drive/MyDrive/Stage_PFA/OCR_OPTIMIZATION/07_pi_deployment


STEP 5A-4 — COPYING ONNX MODELS

FP32:
  Source      : /content/drive/MyDrive/Stage_PFA/OCR_OPTIMIZATION/02_onnx_fp32/yolo11_ocr_fp32.onnx
  Destination : /content/drive/MyDrive/Stage_PFA/OCR_OPTIMIZATION/07_pi_deployment/models/yolo11_ocr_fp32.onnx
  Size        : 10.14 MB
  ✓ Copied

FP16:
  Source      : /content/drive/MyDrive/Stage_PFA/OCR_OPTIMIZA